# 01 · Training, step by step (FSDR V4.0)

**Training** prepares the full customer base of one run date *D*. It validates the data, builds the features,
assigns the segments, and decides the two actions of every line. The scoring notebook (`02_scoring_step_by_step.ipynb`) then reads those results.

| Step | What it does | Writes |
|---|---|---|
| 0 | (optional) dummy data for tests | `data/training/<D>/`, `data/scoring/<D>/` |
| 1 | look at the two input folders (5 sub-folders each) | — |
| 2 | **T0** validation, completeness, Parquet cache | `data/runs/<D>/train/T0_load/`, `data/cache/<D>/` |
| 3 | look at the cache (16 partitions per table) | — |
| 4 | **T2** features B-01 … B-12 for the whole base | `data/runs/<D>/train/T2_features/` |
| 5 | **T3** S-rule segments, bands, micro-segments, week-0 history, `action_original` + `action_agent` | `data/runs/<D>/train/T3_segments/` |
| 6 | summary of the training steps | — |

**Folders of a run date (V4.0):**

```
data/training/<D>/   usage/ models/ subscription/ catalog/ others/     the base, history, model scores
data/scoring/<D>/    usage/ models/ subscription/ catalog/ others/     this week's scoring tables
                     target2/a_constraint/ target2/b_free/                the target lists (Target 2 only)
```

T0 reads the two folders **as one run**. A table filed on the wrong side is still loaded, with a warning.

**How to use:** set the run date in the next cell, then run the cells one by one (Shift+Enter). Read the explanation above each step.
Every step is **resumable**: a step already done with the same inputs is skipped, and `FORCE = True` re-runs it.

## Your machine: Python, GPU and the local LLM

Run this cell first. It only **reads** (nothing is installed, nothing leaves the machine) and prints:
- the Python of the kernel: it must be the project's `.venv` (`…\.venv\Scripts\python.exe`);
- the NVIDIA GPU of the laptop (`nvidia-smi`): name, driver, memory, how busy it is now, and which programs use it;
- the local LLM server of `.env` (`LLM_BACKEND`): is it listening?

**Who uses the GPU.** The steps of this notebook (pandas, Parquet) run on the **CPU**. The GPU is used only by the
local LLM server (vLLM, Ollama, llama.cpp or LM Studio), when an agent calls it. In V4.0 the decisions use the
agents' rule fallbacks (see the agent map below), so the GPU stays idle during these steps; that is expected.

**To see the GPU working:** start your LLM server (for example `ollama serve`), run this cell again, and look at
`programs on the GPU`. Or, in a terminal, run `nvidia-smi -l 1` (it refreshes every second) while the LLM answers:
`GPU-Util` goes up and the server process (`ollama.exe`, `python`, `llama-server`) is listed.

In [ ]:
# --- your machine: Python, GPU, local LLM (read only) -------------------------------------------------------
import os, platform, shutil, socket, subprocess, sys
from pathlib import Path
from urllib.parse import urlparse

print("Python  :", sys.version.split()[0], "|", sys.executable)
_venv = Path(sys.executable).parent.parent
print("kernel  :", "OK, the project's .venv" if _venv.name == ".venv" else
      f"NOT the project's .venv ({_venv}): choose the .venv kernel (top right)")
print("machine :", platform.platform(), "|", os.cpu_count(), "CPU threads")

_smi = shutil.which("nvidia-smi")
if not _smi:
    print("GPU     : no NVIDIA GPU found (nvidia-smi is not on the PATH: no NVIDIA driver, or another GPU brand)")
else:
    def _q(args):
        try:
            r = subprocess.run([_smi, *args, "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
            return [l.strip() for l in r.stdout.splitlines() if l.strip()]
        except (OSError, subprocess.SubprocessError) as e:
            return [f"nvidia-smi failed: {e}"]
    for i, row in enumerate(_q(["--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu"])):
        print(f"GPU {i}   : {row}   (name, driver, memory total, memory used, busy now)")
    apps = _q(["--query-compute-apps=process_name,used_memory"])
    print("programs on the GPU:", "; ".join(apps) if apps else "none right now (the LLM server is not running or idle)")

try:                                              # optional: only if PyTorch is installed in this venv
    import importlib.util
    if importlib.util.find_spec("torch"):
        import torch
        print("PyTorch : CUDA available =", torch.cuda.is_available(),
              f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")
except Exception as e:                            # noqa: BLE001 - a check, never a failure
    print("PyTorch : check failed:", e)

# the local LLM server of .env (the key is never printed)
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
try:
    from dotenv import dotenv_values
    _env = {**dotenv_values(_root / ".env"), **os.environ}
except ImportError:
    _env = dict(os.environ)
_backend = (_env.get("LLM_BACKEND") or "offline").lower()
_urls = {"vllm": ("VLLM_URL", "http://127.0.0.1:8001"), "ollama": ("OLLAMA_URL", "http://127.0.0.1:11434"),
         "llamacpp": ("LLAMACPP_URL", "http://127.0.0.1:8080"), "lmstudio": ("LMSTUDIO_URL", "http://127.0.0.1:1234")}
if _backend not in _urls:
    print(f"LLM     : LLM_BACKEND = {_backend}: no LLM server, the agents use their rule fallbacks (GPU not needed)")
else:
    _u = urlparse(_env.get(_urls[_backend][0]) or _urls[_backend][1])
    try:
        socket.create_connection((_u.hostname, _u.port or 80), timeout=2).close()
        _up = "listening"
    except OSError:
        _up = "NOT running (start it, or set LLM_BACKEND=offline)"
    print(f"LLM     : {_backend} at {_u.scheme}://{_u.hostname}:{_u.port} -> {_up}")


## The agents: Agent 1 … Agent 9

The engine is organised as numbered agents (the numbers of the plan, `docs/fsdr_plan.md` §5). The step titles below
say which agent works in each step.

| Agent | Name | Step (notebook) | What it does | How it decides in V4.0 |
|---|---|---|---|---|
| **Agent 1** ① | Data agent | 01 · Step 2 (T0) | validates the 47 tables, completeness, cache; **blocks the run** on bad data | rules (contract checks); its summary is fixed text |
| **Agent 2** ② | Segmentation | 01 · Step 5 (T3) | S-rules → segment, bands, micro-segment | rules (first rule that matches wins) |
| **Agent 2b** ②b | Agent dictionary | 01 · Step 5 (T3) | segment → `action_agent` (used by Target 2) | week 0: a copy of the production dictionary (v0) |
| **Agent 3** ③ | NBA, next best action | 02 · Step 4 | the action of each T2C line | fallback: the rule action `action_agent` (the LLM decides in notebooks 03 / 04) |
| **Agent 4** ④ | NBO, next best offer | 02 · Step 4 | scores the allowed offers of each T2C / T2F line | fallback: the order E-O3 + E-O1 (`score_source` says it) |
| **Agent 6A** ⑥A | Production rules | — | Target 1 (not scored in V4.0); its code 3 / code 4 passes remain the fallback of 6B | rules (YAML in `nbx/fsdr/rules/`) |
| **Agent 6B** ⑥B | Challenger rules | 02 · Step 4 | Target 2: CC-v1 (group a) and CF-v1 (group b), relative ARPU band | rules |
| **Agent 7** ⑦ | Code 5 | 02 · Step 5 | the 38-column `input_flytxt` file, F-01 … F-19 | rules |
| **Agent 9** ⑨ | Auditor | 02 · Steps 4 and 6 | A-12, A-12b, A-17, A-30, A-31; **blocks the final file** on any FAIL | rules |

Not in these notebooks: Agent 5 ⑤ (market intelligence, removed in V4.0), Agent 8 ⑧ (strategy report),
Agents 10 / 11 (⑩ bandit, ⑪ ranker: evidence helpers), Agent 12 ⑫ (ledger), Agent 13 ⑬ (revenue guard) and the
orchestrator ★. They belong to later phases or to the web interface.

**Does an agent create the actions by itself? Not in V4.0.** Every action and every offer of these notebooks comes
from fixed, versioned rules, and each row says so: `DECIDED_BY = fallback_rule_action`. When the LLM agents ③ / ④
are switched on (plan Phases 5–6), the local LLM will only **score** the actions and offers that the rules allow;
the code then picks the action (with ε exploration and a logged propensity), rejects any action or offer id the LLM
invents, and ⑨ can still block the final file. The LLM proposes; the rules decide.

## How to add new data (real data, or the next week)

Everything is driven by the **run date `D`** (`YYYY-MM-DD`, the extraction date). Each week has its own folders;
the old weeks stay. The full guide, with every table, is `docs/DATA_UPDATE.md`.

1. **Create the folders of `D`** (5 sub-folders on each side, plus the target folders):
   ```
   data/training/<D>/  usage/ models/ subscription/ catalog/ others/
   data/scoring/<D>/   usage/ models/ subscription/ catalog/ others/
                       target2/a_constraint/ target2/b_free/
   ```
2. **Copy the exports**: file name = the exact table name + `.csv.gz` (the only format accepted for real data), e.g. `activities_w1_auto.csv.gz`.
   - **Training side**: `base_prepaid`, `consumer_aact_account`, `subs_btl_auto`, the 11 model tables, `btl_catalog`.
     **Week 0 only**: the history `activities_*_auto`, `subs_amt_auto`, `party_auto`, `access_method_id_auto`,
     `targeting_dataiku_auto`. **Leave them out from week 1 on**, or T0 runs week 0 again on the old history.
   - **Scoring side** (every week): `subscriptions_tg`, `usage_data_voice_payg_tg`, `sub_btl_3months`,
     `activities_w1…w4_score`, `activities_m1…m3_score`, `subs_amt_score`, `party_scoring`, `targeting_auto`,
     `btl_catalog`, `BTL_script_english_arabic_ok`, `conflicting_offers_h`, `consent_contact`.
3. **Copy the 3 target lists** (column `access_method_val`, 8 digits): `target2/a_constraint/target2_a.csv.gz`
   (group a, 50,000 lines) and `target2/b_free/target2_b.csv.gz` (group b, 50,000 lines). No control list by default.
4. **Settings**: `RUN_DATE = "<D>"` in both notebooks, `GENERATE = False` (training) and `GENERATE_TARGETS = False`
   (scoring). Real exports and real lists are never overwritten by dummy data, even if you forget.
5. **Run** training (T0 → T2 → T3), then scoring. Check: T0 **DONE** (`train_week0` the first week,
   `train_weekly` after), both target audits **PASS**, and `data/output/<D>/final/input_flytxt.csv.gz` exists.
6. **Hand over** `data/output/<D>/final/input_flytxt.csv.gz` once it is checked. The engine only saves files, it sends nothing. Keep `data/runs/<D>/` and `data/output/<D>/`.

In [ ]:
# --- settings -------------------------------------------------------------------
RUN_DATE = "2026-11-19"     # the run date D of your real exports (a Thursday in production)
GENERATE = False            # False: your real exports. True: dummy data for a test (real exports are never overwritten)
N_LINES = 3000              # dummy lines (2000 = 30 s; 1,000,000 = the production size, ~16 min to generate)
WORKERS = 1                 # standalone notebook: T2 / T3 run in this process (the engine code lives here, not in files)
FORCE = False               # True: re-run every step even if its inputs did not change
PREVIEW_ROWS = 8


## Part A · The engine code (29 modules, 6,026 lines)

This notebook is **standalone**: every function the steps call is defined in the cells of this part. Nothing is
imported from the project folder (`nbx/`), and the only imports are pip packages (pandas, numpy, pyarrow, pyyaml,
python-dotenv, matplotlib). **vowpalwabbit is never loaded**: it is the V1.0 bandit only, and loaded after pandas it
crashes the Jupyter kernel on Windows (two copies of `msvcp140.dll`).

Each cell starts with `%%engine <module>`: the cell runs its own code and registers it under that name, so the
cells can call each other (an `import` of `nbx.…` inside these cells resolves to the cell above, never to a file).
The code is copied from the project by `notebooks/build_step_notebooks.py`, so it is **exactly** the code of the
terminal commands. Run this part once (it takes a few seconds), then go to Part B.

T2 / T3 run in this process (`WORKERS = 1`): worker processes would have to import the code from files.

| Module | Copied from | Lines | What it does |
|---|---|---|---|
| `nbx` | `nbx/__init__.py` | 24 | NBX: agentic Next-Best-Action / Next-Best-Offer engine. |
| `nbx.core` | `nbx/core/__init__.py` | 0 |  |
| `nbx.core.config` | `nbx/core/config.py` | 107 | App-level settings (read from .env): operator, LLM backend, ports, paths, API security. |
| `nbx.fsdr` | `nbx/fsdr/__init__.py` | 4 | FSDR: production-shaped agentic NBA / NBO on top of NBX. |
| `nbx.fsdr.config` | `nbx/fsdr/config.py` | 734 | FSDR configuration: every business rule constant (BR-xx), the model-score |
| `nbx.fsdr.calendar` | `nbx/fsdr/calendar.py` | 131 | Anchor date and every aggregation window of the business rules (BR §4). |
| `nbx.fsdr.schemas` | `nbx/fsdr/schemas.py` | 447 | Data contract: the exact raw schema of every input file. |
| `nbx.fsdr.datasource` | `nbx/fsdr/datasource.py` | 280 | Where the FSDR pipelines get their tables. |
| `nbx.fsdr.features` | `nbx/fsdr/features.py` | 361 | NBA feature engineering -- port of docs/fsdr_reference/nba_lib.py, without Dataiku. |
| `nbx.fsdr.base_features` | `nbx/fsdr/base_features.py` | 237 | T2_features (master prompt §6, B-01 … B-12): the features of the full base, partition by partition. |
| `nbx.fsdr.campaign_log` | `nbx/fsdr/campaign_log.py` | 269 | Campaign log and rewards: what was sent to whom, and what came back (prompt §6, audit M1..M4). |
| `nbx.fsdr.contract` | `nbx/fsdr/contract/__init__.py` | 9 | V2.0 input contract (plan §4, §13-§15). |
| `nbx.fsdr.steps` | `nbx/fsdr/steps.py` | 194 | Step folders (plan §14): every pipeline step saves its output before the next one starts. |
| `nbx.fsdr.contract.cache` | `nbx/fsdr/contract/cache.py` | 160 | Parquet cache of a run (plan §4, §15): each input table converted once, typed, split into partitions. |
| `nbx.fsdr.contract.completeness` | `nbx/fsdr/contract/completeness.py` | 48 | Weekly completeness (master prompt §5, §17.9): every table the pipeline about to run needs is present, has |
| `nbx.fsdr.validation` | `nbx/fsdr/validation.py` | 455 | Validation of a run folder (or any DataSource) against the data contract. |
| `nbx.fsdr.contract.data_agent` | `nbx/fsdr/contract/data_agent.py` | 117 | ① Data agent (master prompt §5): step T0_load of the training pipeline. |
| `nbx.fsdr.csvstd` | `nbx/fsdr/csvstd.py` | 181 | The FSDR CSV standard (master prompt §2, §20): the one format of every input AND output table file. |
| `nbx.fsdr.offer_rules` | `nbx/fsdr/offer_rules.py` | 270 | Offer business rules -- port of production Codes 3 (constraint arm) and 4 (free arm). |
| `nbx.fsdr.pipelines` | `nbx/fsdr/pipelines/__init__.py` | 71 | Deterministic FSDR pipelines (train, score, recommend). Each step returns a JSON report, |
| `nbx.fsdr.segmentation` | `nbx/fsdr/segmentation/__init__.py` | 2 | ② Segmentation (master prompt §7, §8) and ②b segment actions (§7.1): S-rules, bands, micro-segments, |
| `nbx.fsdr.segmentation.actions` | `nbx/fsdr/segmentation/actions.py` | 202 | ②b The two actions of every line (master prompt §7.1), plus the week-0 history of code 1 (B-02, B-03, B-09, B-10). |
| `nbx.fsdr.segmentation.bands` | `nbx/fsdr/segmentation/bands.py` | 56 | Bands and micro-segments (master prompt §8). Micro-segment = nationality group x age band x tenure band |
| `nbx.fsdr.segmentation.rules` | `nbx/fsdr/segmentation/rules.py` | 83 | S-rules (master prompt §7): the current segment of every line, first match wins, thresholds in config. |
| `nbx.fsdr.pipelines.training` | `nbx/fsdr/pipelines/training.py` | 367 | V2.0 training pipeline, stages T2 and T3 (plan §14, §15; master prompt §6 - §8, §18). |
| `nbx.fsdr.synth` | `nbx/fsdr/synth/__init__.py` | 1 | Dummy-data world, scenarios and generator (same JSON files as the real data). |
| `nbx.fsdr.synth.scenarios` | `nbx/fsdr/synth/scenarios.py` | 124 | Use-case scenarios for the dummy data (combinable, e.g. "churn_wave,data_growth"). |
| `nbx.fsdr.synth.world` | `nbx/fsdr/synth/world.py` | 370 | Hidden customer world for the dummy data and the closed-loop simulation. |
| `nbx.fsdr.synth.generator` | `nbx/fsdr/synth/generator.py` | 722 | Dummy-data generator: writes exactly the files of the real data contract. |

In [ ]:
# --- Part A loader: the cells below hold the engine code; %%engine registers each one as a module -------------
# Nothing is imported from the project folder: a cell "%%engine nbx.fsdr.csvstd" runs ITS OWN code and makes it
# available to the other cells under that name. Only pip packages are imported (pandas, numpy, pyarrow, pyyaml,
# python-dotenv, matplotlib). vowpalwabbit is never loaded.
import linecache, os, sys, types
from pathlib import Path

from IPython.core.magic import register_cell_magic

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:      # the project folder (for data/ and .env)
    ROOT = ROOT.parent
os.chdir(ROOT)
for _name in [m for m in sys.modules if m == "nbx" or m.startswith("nbx.")]:
    del sys.modules[_name]                                                  # a re-run starts clean


def _module(name, package=False):
    mod = types.ModuleType(name)
    mod.__fsdr_notebook__ = True                                            # proof: defined in this notebook
    if package:
        mod.__path__ = []          # its sub-modules are cells of this notebook, never files on disk
    mod.__file__ = str(ROOT / (name.replace(".", "/") + ("/__init__.py" if package else ".py")))  # paths only
    mod.__package__ = name if package else name.rpartition(".")[0]
    sys.modules[name] = mod
    parent, _, child = name.rpartition(".")
    if parent:
        setattr(_package(parent), child, mod)
    return mod


def _package(name):
    m = sys.modules.get(name)
    return m if getattr(m, "__fsdr_notebook__", False) else _module(name, package=True)


@register_cell_magic
def engine(line, cell):
    """%%engine <module> [package]: run this cell's code as that module."""
    name, *flags = line.split()
    mod = _package(name) if "package" in flags else _module(name)
    filename = f"<notebook: {name}>"
    linecache.cache[filename] = (len(cell), None, cell.splitlines(True), filename)   # readable tracebacks
    exec(compile(cell, filename, "exec"), mod.__dict__)


def use(module, *names):
    """The objects of an engine module (defined by a cell below), e.g. config, csvstd = use("nbx.fsdr", "config", "csvstd")."""
    mod = sys.modules[module]
    out = [getattr(mod, n) if hasattr(mod, n) else sys.modules[f"{module}.{n}"] for n in names]
    return out[0] if len(out) == 1 else tuple(out)


print("engine loader ready: the next cells define the engine modules")

In [ ]:
%%engine nbx package
"""NBX engine, standalone in this notebook (V4.0 training / scoring). vowpalwabbit is never loaded."""
import os

for _k, _v in {"GRADIO_ANALYTICS_ENABLED": "False", "HF_HUB_DISABLE_TELEMETRY": "1", "DO_NOT_TRACK": "1",
               "VLLM_NO_USAGE_STATS": "1", "ANONYMIZED_TELEMETRY": "False"}.items():
    os.environ.setdefault(_k, _v)

__version__ = "4.0.0"


In [ ]:
%%engine nbx.core package



In [ ]:
%%engine nbx.core.config
"""App-level settings (read from .env): operator, LLM backend, ports, paths, API security.

The FSDR engine's own business constants (BR-01 ... BR-86) are in nbx/fsdr/config.py.
"""
import os
from pathlib import Path

from dotenv import load_dotenv

# Project root: this file lives in nbx/core/.
BASE_DIR = Path(__file__).resolve().parents[2]

load_dotenv(BASE_DIR / ".env")

# Our operator's name in every prompt, SMS template and UI text.
OPERATOR_NAME = os.getenv("OPERATOR_NAME", "FSDR")

# ---------------------------------------------------------------------------
# LLM backend (llms/llm_client.py). Every backend runs ON THIS MACHINE (see LOCAL_ONLY below):
#   "offline"  -- pure Python, no server (default): every agent takes its rule-based path.
#   "vllm"     -- a vLLM server (GPU, WSL2 / Linux): `python run.py vllm-serve`            -> VLLM_URL
#   "ollama"   -- Ollama (Windows / macOS / Linux, CPU or GPU): `ollama serve`             -> OLLAMA_URL
#   "llamacpp" -- llama.cpp's llama-server with a GGUF file (CPU or GPU)                  -> LLAMACPP_URL
#   "lmstudio" -- LM Studio's local server (Developer tab -> Start server)                 -> LMSTUDIO_URL
# The four servers speak the same OpenAI-compatible API (/v1/models, /v1/chat/completions): one client
# (llms/vllm_client.py) serves them all. No cloud LLM API (OpenAI, Anthropic / Claude, …) is used anywhere.
# ---------------------------------------------------------------------------
LLM_BACKEND = os.getenv("LLM_BACKEND", "offline").lower()
LLM_SERVER_BACKENDS = ["vllm", "ollama", "llamacpp", "lmstudio"]
LLM_BACKENDS = ["offline", *LLM_SERVER_BACKENDS]
DEFAULT_LLM = os.getenv("DEFAULT_LLM", "qwen2.5:3b")
# 127.0.0.1, not "localhost": vLLM binds IPv4 only, and "localhost" tries ::1 first (slow on Windows).
VLLM_URL = os.getenv("VLLM_URL", "http://127.0.0.1:8001")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
LLAMACPP_URL = os.getenv("LLAMACPP_URL", "http://127.0.0.1:8080")
LMSTUDIO_URL = os.getenv("LMSTUDIO_URL", "http://127.0.0.1:1234")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")            # ollama / llamacpp / lmstudio, only if you set one there
# Grammar-constrained JSON (response_format json_object) per server; LM Studio only takes json_schema, so the
# agents parse its free text instead (llm_client.extract_json).
LLM_JSON_MODE = {"vllm": True, "ollama": True, "llamacpp": True, "lmstudio": False}


def llm_url(backend: str | None = None) -> str:
    """The URL of the server of a backend (the current one by default); "" for offline."""
    b = (backend or LLM_BACKEND).lower()
    return {"vllm": VLLM_URL, "ollama": OLLAMA_URL, "llamacpp": LLAMACPP_URL, "lmstudio": LMSTUDIO_URL}.get(b, "")


def llm_api_key(backend: str | None = None) -> str:
    return VLLM_API_KEY if (backend or LLM_BACKEND).lower() == "vllm" else LLM_API_KEY


# ---------------------------------------------------------------------------
# V4.0 local only. True (default): the app talks only to servers on this machine (127.0.0.1 / localhost / ::1,
# plus the hosts you list in FSDR_LOCAL_HOSTS, e.g. your own GPU box on the LAN). An LLM URL or a tracing
# endpoint elsewhere is refused (the agents fall back to their rules, tracing stays off), and the usage
# analytics of Gradio / Hugging Face / vLLM are switched off (nbx/__init__.py).
# ---------------------------------------------------------------------------
LOCAL_ONLY = os.getenv("FSDR_LOCAL_ONLY", "true").lower() == "true"
LOCAL_HOSTS = {"127.0.0.1", "localhost", "::1"} | {h.strip().lower() for h in
                                                   os.getenv("FSDR_LOCAL_HOSTS", "").split(",") if h.strip()}


def is_local_url(url: str) -> bool:
    from urllib.parse import urlparse

    host = (urlparse(url).hostname or "").lower()
    return host in LOCAL_HOSTS or host.startswith("127.")


def local_violations() -> list[str]:
    """What would leave the machine with the current settings (empty = everything local)."""
    out = []
    if LLM_BACKEND in LLM_SERVER_BACKENDS and not is_local_url(llm_url()):
        out.append(f"LLM_BACKEND={LLM_BACKEND} points to {llm_url()} (not this machine)")
    tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "")).lower() in ("true", "1")
    endpoint = os.getenv("LANGSMITH_ENDPOINT") or os.getenv("LANGCHAIN_ENDPOINT") or "https://api.smith.langchain.com"
    if tracing and not is_local_url(endpoint):
        out.append(f"LANGSMITH_TRACING=true sends traces to {endpoint} (not this machine)")
    return out
# Cap per completion; otherwise one runaway generation holds a batch slot for its whole length.
VLLM_MAX_TOKENS = int(os.getenv("VLLM_MAX_TOKENS", "512"))
LLM_TIMEOUT_S = float(os.getenv("LLM_TIMEOUT_S", "300"))

# ---------------------------------------------------------------------------
# Ports, paths, logging
# ---------------------------------------------------------------------------
FASTAPI_PORT = int(os.getenv("FASTAPI_PORT", "8000"))
GRADIO_PORT = int(os.getenv("GRADIO_PORT", "7860"))
UI_HOST = os.getenv("FSDR_UI_HOST", "127.0.0.1")            # 0.0.0.0 in the container (Dockerfile)
API_URL = os.getenv("FSDR_API_URL", "")                      # UI -> API base URL (default http://localhost:<port>)
DATA_DIR = Path(os.getenv("DATA_DIR", str(BASE_DIR / "data")))
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# API security (review finding #5). With FSDR_API_KEY set, every /fsdr/* route needs the
# header `X-API-Key: <key>` (the UI sends it). Empty = open, for local development only: the API then binds
# 127.0.0.1 by default. Model promotion that bypasses the IPS rule needs FSDR_ADMIN_KEY as well.
# ---------------------------------------------------------------------------
API_KEY = os.getenv("FSDR_API_KEY", "")
ADMIN_KEY = os.getenv("FSDR_ADMIN_KEY", "")
CORS_ORIGINS = [o.strip() for o in os.getenv("FSDR_CORS_ORIGINS", f"http://localhost:{GRADIO_PORT},"
                                             f"http://127.0.0.1:{GRADIO_PORT}").split(",") if o.strip()]

# ---------------------------------------------------------------------------


In [ ]:
%%engine nbx.fsdr package
"""FSDR: production-shaped agentic NBA / NBO on top of NBX.

Plan and phase status: docs/fsdr_plan.md.
"""


In [ ]:
%%engine nbx.fsdr.config
"""FSDR configuration: every business rule constant (BR-xx), the model-score
registry, arms, paths and generator defaults.

Business rules come from docs/fsdr_reference/FSDR_NBA_business_rules.md;
each constant names its BR number. Logic modules import from here and never
hard-code these values.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from datetime import date
from pathlib import Path

from nbx.core import config as nbx_config

OPERATOR_NAME = os.getenv("OPERATOR_NAME", "FSDR")
CURRENCY = os.getenv("FSDR_CURRENCY", "OMR")
SCHEMA_VERSION = "2.0"

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
FSDR_DATA_DIR = Path(os.getenv("FSDR_DATA_DIR", str(nbx_config.DATA_DIR / "fsdr")))
FSDR_STATE_DIR = FSDR_DATA_DIR / "state"          # registry, campaign log, rewards
CAMPAIGN_DIR = "campaign"                         # <data root>/state/campaign: nba_campaign_log.jsonl, nba_rewards.jsonl
JSON_SCHEMA_DIR = nbx_config.BASE_DIR / "docs" / "fsdr_json_schemas"
MANIFEST_FILE = "_manifest.json"
VALIDATION_JSON = "_validation.json"
VALIDATION_MD = "_validation.md"

# V2.0 data layout (plan §13-§15)
FSDR_INPUT_DIR = Path(os.getenv("FSDR_INPUT_DIR", str(nbx_config.DATA_DIR / "input")))    # <D>/<folder>/<table>.*
FSDR_CACHE_DIR = Path(os.getenv("FSDR_CACHE_DIR", str(nbx_config.DATA_DIR / "cache")))    # <D>/<table>/part-NN.parquet
FSDR_RUNS_DIR = Path(os.getenv("FSDR_RUNS_DIR", str(nbx_config.DATA_DIR / "runs")))       # <D>/{train,score,final}/<step>/
# V4.0 data layout: the inputs of a run date are split in two twin folders with the same 5 sub-folders, and the
# results of each target go to data/output/<D>/<target>/ (docs/SETUP_NEW_LAPTOP.md §5)
#   data/training/<D>/{usage,models,subscription,catalog,others}/     the base, history and model scores (T0 - T3)
#   data/scoring/<D>/{usage,models,subscription,catalog,others}/      this week's scoring tables (P1 - P7)
#   data/scoring/<D>/target1/{a_constraint,b_free}/  target2/{a_constraint,b_free}/  control/   the target lists
#   data/output/<D>/target1/  target2/  control/  final/                one folder per target, scored separately
TRAINING_FOLDER, SCORING_FOLDER = "training", "scoring"
FSDR_TRAINING_DIR = Path(os.getenv("FSDR_TRAINING_DIR", str(nbx_config.DATA_DIR / TRAINING_FOLDER)))
FSDR_SCORING_DIR = Path(os.getenv("FSDR_SCORING_DIR", str(nbx_config.DATA_DIR / SCORING_FOLDER)))
FSDR_WEEKLY_DIR = Path(os.getenv("FSDR_WEEKLY_DIR", str(nbx_config.DATA_DIR / "weekly")))   # notebooks 03 / 04: one folder
FSDR_OUTPUT_DIR = Path(os.getenv("FSDR_OUTPUT_DIR", str(nbx_config.DATA_DIR / "output")))
INPUT_FOLDERS = ("usage", "models", "subscription", "catalog", "others")
INPUT_FORMATS = (".csv.gz", ".json", ".jsonl", ".csv", ".parquet")   # V3.0: .csv.gz = production (master §2)
PRODUCTION_FORMAT = ".csv.gz"
# master §2: plain .csv / .json / .jsonl / .parquet only for tests and the dummy generator (its manifest says so)
ALLOW_TEST_FORMATS = os.getenv("FSDR_ALLOW_TEST_FORMATS", "false").lower() == "true"
CSV_GZIP_LEVEL = 6                                # FSDR CSV standard
ID_TEXT_COLUMNS = {"ACCESS_METHOD_VAL", "MSISDN", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID", "SUBSCRIPTION_ID",
                   "UBSCRIPTION_ID"}              # always read and written as text
OUTPUT_COMPRESSION = os.getenv("FSDR_OUTPUT_COMPRESSION", os.getenv("FSDR_PUSH_COMPRESSION", "gzip"))    # "gzip" | "none" (also writes input_flytxt.csv)
CACHE_PARTITIONS = int(os.getenv("FSDR_CACHE_PARTITIONS", "16"))     # plan §15: hash of ACCESS_METHOD_ID
TABLE_FOLDERS = {                                 # plan §13; the 11 model-score tables go to "models"
    "usage": ["subs_amt_auto", "subs_amt_score", "activities_w1_auto", "activities_w2_auto", "activities_w3_auto",
              "activities_w4_auto", "activities_m1_auto", "activities_auto_m2", "activities_m3_auto",
              "activities_w1_score", "activities_w2_score", "activities_w3_score", "activities_w4_score",
              "activities_m1_score", "activities_m2_score", "activities_m3_score", "paygo_activity_auto",
              "usage_data_voice_payg_tg"],
    "subscription": ["base_prepaid", "consumer_aact_account", "access_method_id_auto", "subscriptions_tg",
                     "subs_btl_auto", "sub_btl_3months"],
    "catalog": ["btl_catalog", "BTL_script_english_arabic_ok", "conflicting_offers_h"],
    "others": ["party_auto", "party_scoring", "customer_profile", "location_auto", "consent_contact",
               "DEVICEBEHAVIORSEGMENTATION.dv_segment", "targeting_dataiku_auto", "targeting_auto"],
}                                                 # any other table (e.g. the test sample) -> "others"
# V4.0: the tables filed under data/scoring/<D>/ (everything else under data/training/<D>/). A table found in both
# twins is read from its own side (btl_catalog is written to both: training history and this week's offers).
SCORING_TABLES = {"subscriptions_tg", "usage_data_voice_payg_tg", "sub_btl_3months", "targeting_auto",
                  "btl_catalog", "BTL_script_english_arabic_ok", "conflicting_offers_h", "consent_contact",
                  "subs_amt_score", "activities_w1_score", "activities_w2_score", "activities_w3_score",
                  "activities_w4_score", "activities_m1_score", "activities_m2_score", "activities_m3_score",
                  "party_scoring"}
BOTH_TABLES = {"btl_catalog"}                     # the generator writes these to both twins


def table_side(table: str) -> str:
    return SCORING_FOLDER if table in SCORING_TABLES or table == TARGET_SAMPLE_TABLE else TRAINING_FOLDER


def run_twins(folder: Path) -> dict[str, Path]:
    """{"training": ..., "scoring": ...} of a run folder in the V4.0 layout (data/training/<D> <-> data/scoring/<D>),
    only the twins that exist; {} for a single folder (V3.0 layout: every table in data/input/<D>/)."""
    folder = Path(folder)
    side = folder.parent.name
    if side not in (TRAINING_FOLDER, SCORING_FOLDER):
        return {}
    other = SCORING_FOLDER if side == TRAINING_FOLDER else TRAINING_FOLDER
    twins = {side: folder, other: folder.parent.parent / other / folder.name}
    return {k: v for k, v in twins.items() if v.is_dir()}


def training_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_TRAINING_DIR) / str(run_date)


def scoring_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_SCORING_DIR) / str(run_date)


def output_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_OUTPUT_DIR) / str(run_date)


STEP_FILE = "_step.json"
STEP_PREVIEW_ROWS = 1000                          # training outputs (1M rows): Parquet + the first rows as preview.csv
STEP_FULL_CSV_MAX_ROWS = 100_000                  # scoring outputs (<= 50k rows): full CSV next to the Parquet

# customer_profile (master prompt §2.1, §5): age and tenure of every line, provided weekly
PROFILE_TABLE = "customer_profile"
PROFILE_TENURE_UNIT = os.getenv("FSDR_PROFILE_TENURE_UNIT", "days")          # open item 5
PROFILE_AGE_RANGE = (15, 100)                     # outside -> warning
PROFILE_AGE_TOLERANCE_YEARS = 2                   # vs party_*.AGE
PROFILE_TENURE_TOLERANCE_DAYS = 30                # vs run date - access_method_subs_start_dt
PROFILE_MIN_COVERAGE = 0.95                       # share of the base with a profile row (below -> warning)

# ---------------------------------------------------------------------------
# Calendar (BR notation: D = run date, T = campaign anchor)
# ---------------------------------------------------------------------------
REFERENCE_THURSDAY = date(2026, 3, 12)            # T = D - (((D - 2026-03-12) MOD 7) + 7) MOD 7
SNAPSHOT_LAG_DAYS = 2                             # BR-01 ACTIVITY_DATE = D-2
HISTORY_DAYS_USAGE = 97                           # daily history D-96..D: deepest window is paygo M3 = T-90, T >= D-6
HISTORY_DAYS_FULFILMENT = 270                     # BR-24 interactions D-270..D-1

# ---------------------------------------------------------------------------
# Eligibility (BR-01 .. BR-12)
# ---------------------------------------------------------------------------
MSISDN_MIN_EXCLUSIVE = "69999999"                 # BR-02 string comparison
# BR-12 (SUBSTR(MSISDN, 4)) is applied upstream in the SQL exports: every MSISDN in the JSON files,
# including access_method_id_auto.MSISDN, is already the 8-digit ACCESS_METHOD_VAL. No prefix handling here.
EXCLUDED_PLANS = [                                # BR-10 internal / employee / business plans (13)
    "FSDR Internal",
    "Employee Data with Voice",
    "Shahry Business",
    "FSDR Shahry",
    "Employee superfast 100 Mbps plan",
    "Employee superfast 200 Mbps plan",
    "Employee superfast 400 Mbps plan",
    "5G Employee Plan 100 Mbps",
    "5G Employee Plan 50 Mbps",
    "Employee Fast HBB 150 GB Plan",
    "Employee Fast HBB 300 GB Plan",
    "Employee Fast HBB 600 GB Plan",
    "Fast  Home Broadband 150GB (Employee)",      # double space as in production
]

# ---------------------------------------------------------------------------
# Demographics (BR-13 .. BR-15)
# ---------------------------------------------------------------------------
NATIONALITY_KEEP = ["Oman", "India", "Bangladesh", "Pakistan"]   # BR-15; the rest -> "others"
NATIONALITY_OTHERS = "others"

# ---------------------------------------------------------------------------
# Segments and actions (BR-36 .. BR-45)
# ---------------------------------------------------------------------------
# BR-38. Fixed order: the bandit's probability vector is returned in this order. Never shuffle it.
ACTIONS = ["CROSS SELL DATA", "UPSELL DATA", "RETAIN", "MAINTAIN", "UPSELL", "CROSS SELL VOICE", "UPSELL VOICE"]
NO_CONTACT = "NO_CONTACT"                         # llm-arm only, never a VW action (plan Q10)

EXCLUDED_SEGMENTS = {"nursering_early_churn_at_risk", "nursering_early_churn_not_at_risk"}   # BR-36

# BR-39, lower-case keys so "RGS_..." and "Rgs_..." resolve to the same entry.
SEGMENT_TO_ACTION = {
    "retention_low_probability_to_reconnect": "RETAIN",
    "retention_med_probability_to_reconnect": "RETAIN",
    "retention_high_probability_to_reconnect": "RETAIN",                 # CHECK (plan Q1): was UPSELL then RETAIN
    "retention_rgs_at_risk": "RETAIN",
    "retention_rgs_fragile": "UPSELL",
    "rgs_not_at_risk_dualplay_upgrade": "MAINTAIN",
    "rgs_not_at_risk_data_only_voice_appetency": "UPSELL VOICE",          # CHECK (plan Q1b): or CROSS SELL VOICE
    "rgs_not_at_risk_data_only_no_voice_appetency": "CROSS SELL VOICE",
    "rgs_not_at_risk_voice_only_data_appetency": "UPSELL DATA",
    "rgs_not_at_risk_voice_only_no_data_appetency": "CROSS SELL DATA",
    "onboarding_early_churn_at_risk": "UPSELL",
    "onboarding_early_churn_not_at_risk": "UPSELL",
}

# Canonical segment spellings, and the mixed-case variants production also emits.
SEGMENTS = [
    "Retention_Low_Probability_To_Reconnect", "Retention_Med_Probability_To_Reconnect",
    "Retention_High_Probability_To_Reconnect", "Retention_RGS_At_Risk", "Retention_RGS_Fragile",
    "RGS_Not_At_Risk_DualPlay_Upgrade", "RGS_Not_At_Risk_Data_Only_Voice_Appetency",
    "RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "RGS_Not_At_Risk_Voice_Only_Data_Appetency",
    "RGS_Not_At_Risk_Voice_Only_No_Data_Appetency", "Onboarding_Early_Churn_At_Risk",
    "Onboarding_Early_Churn_Not_At_Risk",
    "Nursering_Early_Churn_At_Risk", "Nursering_Early_Churn_Not_At_Risk",
]
NURSERING_MAX_TENURE_DAYS = 45                    # BR-36
ONBOARDING_MAX_TENURE_DAYS = 90

ACTION_PRIOR = {                                  # BR-45 share of historical targeting volume
    "RETAIN": 0.5086577,
    "UPSELL": 0.2624428,
    "MAINTAIN": 0.1015183,
    "CROSS SELL VOICE": 0.0528180,
    "UPSELL DATA": 0.0422645,
    "UPSELL VOICE": 0.0239859,
    "CROSS SELL DATA": 0.0083128,
}
MISSING_SCORE_RANGE = (0.0, 0.02)                 # BR-47, drawn per row

# ---------------------------------------------------------------------------
# Model-score registry (BR-46). Adding / removing a model = editing this list.
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class ModelScore:
    table: str                      # exact file / table name
    feature: str                    # feature name in the NBA context
    combine: str = "max"            # how rows mapping to the same feature are merged
    signal: str = ""                # hidden-world signal the generator drives it with
    tenure_range: tuple[int, int] | None = None   # generator: only lines in this tenure band
    coverage: tuple[float, float] = (0.70, 0.90)  # generator: share of the base scored


MODEL_SCORE_REGISTRY: list[ModelScore] = [
    ModelScore("DAIM_CHURN_GLOBAL_SCORING_TD.prediction", "PROBA_CHURN", "max", "churn"),
    ModelScore("DAIM_EARLY_CHURN_0_30_SCORING_TD.Early_Churn_0_30_Prediction", "PROBA_CHURN", "max", "churn",
               tenure_range=(0, 30), coverage=(0.90, 0.98)),
    ModelScore("DAIM_EARLY_CHURN_30_60_SCORING_TD.scoring_data_scored", "PROBA_CHURN", "max", "churn",
               tenure_range=(30, 60), coverage=(0.90, 0.98)),
    ModelScore("DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_TD.Scored", "PROBA_REACT_30", "max", "reactivation_30"),
    ModelScore("PROPENSITY_REACTIVATION_15D_401_SCORING.Scored", "PROBA_REACT_15", "max", "reactivation_15"),
    ModelScore("APPETENCY_DATA_204_SCORING_TD.prediction", "PROBA_APP_DATA", "max", "data_appetite"),
    ModelScore("APPETENCY_VOICE_204_SCORING_TD.Scored", "PROBA_APP_VOICE", "max", "voice_appetite"),
    ModelScore("CHURN_ONE_DAY_ACTIVE_107_SCORING.Churn_One_Day_Scoring", "PROBA_CHURN_1D", "max", "churn_1d"),
    ModelScore("REV_DECREASE_204_SCORING_TD.predictions", "PROBA_REV_DECR", "max", "revenue_decrease"),
    ModelScore("DAIM_FRAGILITY_DATA_SCORING_TD.prediction", "PROBA_FRAG_DATA", "max", "fragility_data"),
    ModelScore("DAIM_FRAGILITY_VOICE_SCORING_TD.prediction", "PROBA_FRAG_VOICE", "max", "fragility_voice"),
]


def proba_features(registry: list[ModelScore] | None = None) -> list[str]:
    """Distinct feature names, in registry order (PROBA_CHURN first)."""
    out: list[str] = []
    for m in registry or MODEL_SCORE_REGISTRY:
        if m.feature not in out:
            out.append(m.feature)
    return out


# ---------------------------------------------------------------------------
# Experiment arms (BR-59 .. BR-62)
# ---------------------------------------------------------------------------
ARMS = ["cg", "free", "cstr", "llm"]              # cg = control (no contact); llm = new arm
ARM_SHARES = {"cg": 0.25, "free": 0.25, "cstr": 0.25, "llm": 0.25}
TARGET_SAMPLE_TABLE = os.getenv("FSDR_TARGET_SAMPLE_TABLE", "sample_cg_tg_new_7oct2026")   # plan Q16
TARGET_SAMPLE_SHARE = 0.40                        # generator: share of the base in the test sample
STRATA_FIELDS = ["nationality_group", "revenue_band", "voice_group", "data_group", "plan"]   # BR-62

# ---------------------------------------------------------------------------
# Offers, ARPU anchor and price bands (BR-63 .. BR-82); used from Phase 4
# ---------------------------------------------------------------------------
SERVICES = ["DATA", "VOICE", "COMBO"]
GENEROSITY_LEVELS = ["L0", "L1", "L2"]            # BR-65 L0 < L1 < L2
VALIDITIES = [7, 30]                              # BR-64
GENEROSITY_MAPPING = os.getenv("FSDR_GENEROSITY_MAPPING", "all_L2")   # BR-66 "all_L2" (active) | "normal" (plan Q3)
ACTION_FAMILIES = {                               # BR-66: (service A, service B) per action
    "MAINTAIN": ("COMBO", "DATA"),
    "UPSELL": ("COMBO", "DATA"),
    "RETAIN": ("COMBO", "DATA"),
    "UPSELL VOICE": ("DATA", "COMBO"),
    "CROSS SELL VOICE": ("DATA", "COMBO"),
    "CROSS SELL DATA": ("VOICE", "COMBO"),
    "UPSELL DATA": ("VOICE", "COMBO"),
}
ACTION_GENEROSITY_NORMAL = {                      # BR-66 "normal" (commented out in production): (gen A, gen B)
    "MAINTAIN": ("L0", "L0"),
    "UPSELL": ("L1", "L1"),
    "RETAIN": ("L2", "L2"),
    "UPSELL VOICE": ("L0", "L1"),
    "CROSS SELL VOICE": ("L0", "L2"),
    "CROSS SELL DATA": ("L0", "L2"),
    "UPSELL DATA": ("L0", "L2"),
}
FREE_ARM_GENEROSITY = "L2"                        # BR-67
FREE_ARM_SERVICES = ("COMBO", "DATA")             # BR-67 VOICE excluded
ARPU_DATA_SHARE = 0.9                             # BR-74
ARPU_VOICE_SHARE = 0.1                            # BR-75
PRICE_BAND_K = {                                  # BR-78 pass-1 upper margin
    "UPSELL": 3.0, "MAINTAIN": 2.0, "RETAIN": 2.0,
    "UPSELL VOICE": 3.0, "CROSS SELL DATA": 3.0, "CROSS SELL VOICE": 3.0, "UPSELL DATA": 3.0,
    "free": 3.0,
}
MAX_OFFERS_PER_VALIDITY = 2                       # BR-82
MAX_OFFERS_TOTAL = 4                              # BR-82

# ---------------------------------------------------------------------------
# Campaign windows (BR-21 .. BR-25), in days relative to D
# ---------------------------------------------------------------------------
TARGETING_WINDOW = (-8, -2)                       # BR-21 targeting_dataiku_auto.date1
ARBITRATION_WINDOW = (-7, 0)                      # BR-22 targeting_auto.start_date
REWARD_WINDOW = (-7, -1)                          # BR-23 fulfilment success date
INTERACTION_WINDOW = (-270, -1)                   # BR-24 sub_btl_3months

# ---------------------------------------------------------------------------
# Dummy-data generator defaults
# ---------------------------------------------------------------------------
GENERATOR_DEFAULT_N = 5000
GENERATOR_MAX_N = 2_000_000                       # V2.0: 1M designed, 2M tested on the P16 (plan §16)
GENERATOR_MODES = {"small": 2_000, "medium": 100_000, "full": 1_000_000}
GENERATOR_SEED = 42
GENERATOR_CHUNK = 20_000                          # customers per daily-history chunk (memory bound)
SCENARIOS = ["baseline", "churn_wave", "data_growth", "price_sensitive", "new_offers",
             "campaign_fatigue", "data_quality_issues"]
SCENARIO_RAMP_DAYS = 28                           # scenario drifts build up over the last 4 weeks
TARGETED_SHARE = 0.55                             # share of the base targeted in last week's campaign

# ---------------------------------------------------------------------------
# Features (BR-46 .. BR-53)
# ---------------------------------------------------------------------------
USAGE_COLS = ["LOCAL_VOICE_MIN", "DATA_VOLUME_MB", "PG_DATA_AMOUNT"]   # columns the model reads from activities_*
# BR-53 trend pairs as (recent, older): change FROM older TO recent (W1 = most recent week).
TREND_PAIRS = {
    "REV": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2"), ("M2", "M3")],
    "VOICE": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2")],
    "DATA": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2")],
}
TREND_SOURCE = {"REV": "REV_{p}", "VOICE": "LOCAL_VOICE_MIN_{p}", "DATA": "DATA_VOLUME_MB_{p}"}   # BR-49 REV = SUB_AMT + PG_DATA
PCT_CLIP = (-100.0, 500.0)                        # BR-50 % change, capped when the older period is tiny
# M3 last-campaign context, from the campaign log (campaign_log.last_campaign_features). Part of the model's
# feature list, so the compatibility check refuses a model trained without them (and the reverse).
LAST_CAMPAIGN_FEATURES = os.getenv("FSDR_LAST_CAMPAIGN_FEATURES", "true").lower() == "true"
LAST_CAMPAIGN_NUMERIC = ["PREV_OFFER_TAKEN", "DAYS_SINCE_LAST_BTL", "NB_CONTACTS_LAST_4W"]
PREV_ACTION_NONE = "NONE"                          # PREV_ACTION of a line never contacted
# V1.0 review #7: nationality as a targeting feature is a legal question (review F4); false drops it from the
# model (the fairness report per nationality group is produced either way).
USE_NATIONALITY = os.getenv("FSDR_USE_NATIONALITY", "true").lower() == "true"
CATEGORICAL_FEATURES = (["NATIONALITY_GROUP"] if USE_NATIONALITY else []) + (["PREV_ACTION"] if LAST_CAMPAIGN_FEATURES else [])
# V1.0 review #11 / #12: one PROBA_<x>_MISSING flag per score (BR-47's random fill stays, the model now sees it)
# and LOG_TENURE (tenure of the model-score files). false = the production feature list (parity).
EXTENDED_FEATURES = os.getenv("FSDR_EXTENDED_FEATURES", "true").lower() == "true"

# ---------------------------------------------------------------------------
# Bandit, training and scoring (BR-54 .. BR-58)
# ---------------------------------------------------------------------------
VW_EPSILON = 0.2                                  # BR-54 (allowed range for the training agent below)
VW_EPSILON_RANGE = (0.05, 0.3)
VW_ARGS_TEMPLATE = "--cb_explore_adf -q UA --quiet --epsilon {epsilon}"   # BR-54
TRAIN_MODE = "logged"                             # "logged" (all targeted, recommended) | "simulation" (BR-43/55/56)
TRAIN_PASSES = 2
TRAIN_PASSES_RANGE = (1, 5)
SIM_ITERATIONS = 20_000                           # BR-55 simulation mode
HOLDOUT_SHARE = 0.2
REWARD_DEFINITIONS = ["any_btl", "same_service", "exact_offer"]
REWARD_DEFINITION = "any_btl"                     # BR-40 production (plan Q4)
# BR-40: a taker is a targeted line with ANY catalogue BTL fulfilled in D-7..D-1 (production). Set true to also
# require the fulfilment on or after the line's targeting date1 (plan Q2 option; the two windows overlap).
REWARD_AFTER_TARGETING = os.getenv("FSDR_REWARD_AFTER_TARGETING", "false").lower() == "true"
PROPENSITY_CLIP = (0.01, 1.0)
SCORING_POLICY = "sample"                         # BR-57 sample (production) | greedy
BASE_SCORING_POLICY = "greedy"
MAX_MODEL_AGE_DAYS = 14                           # plan Q18
PROMOTION_TOLERANCE = 0.0                         # V0 rule: challenger must reach champion IPS - tolerance
# V1.0 review #2 / #3: the holdout is evaluated for the policy that is deployed (sampling from the PMF, BR-57), and
# the challenger is compared with the champion on the SAME holdout (out-of-time for the champion) by a paired
# bootstrap of the SNIPS gain. "paired_ci": refused only when the challenger is significantly worse (upper bound of
# the gain < -PROMOTION_MARGIN); significantly better -> promoted; inconclusive -> promoted (the fresher model wins
# when the data cannot separate them; scoring refuses a champion older than 14 days anyway). "point" = V0 (point
# estimates, which promote and refuse on noise: 0.215 vs 0.133 on 221 rows).
PROMOTION_RULE = os.getenv("FSDR_PROMOTION_RULE", "paired_ci").lower()     # paired_ci | point (V0)
PROMOTION_MARGIN = float(os.getenv("FSDR_PROMOTION_MARGIN", "0.0"))
BOOTSTRAP_SAMPLES = 500
CONFIDENCE = 0.95
# V1.0 review #1: the legacy campaign assigns the action from the segment (BR-39), so P(action | context) = 1 for
# the action sent. "deterministic" logs 1.0 (and flags the week as not usable for off-policy estimates);
# "share" is V0 (this week's action share, which the review found biased).
TARGETING_PROPENSITY = os.getenv("FSDR_TARGETING_PROPENSITY", "deterministic").lower()
# V1.0 review #4: uplift reward. The control group's organic take rate is modelled from the features (logistic
# regression on the cg rows of the campaign log); the bandit's cost becomes -(reward - baseline), so it learns the
# lift of a contact instead of "who buys anyway". false = V0 (cost = -reward).
UPLIFT_REWARD = os.getenv("FSDR_UPLIFT_REWARD", "true").lower() == "true"
PSI_BINS = 10
PSI_ALERT = 0.25                                  # PSI above this on a feature = drift alert
GROUP_TO_FLAG = {"free": "free", "cstr": "with_constraint", "llm": "llm"}   # BR-59; cg is not scored (control)

# ---------------------------------------------------------------------------
# Offer ranker (BR-83 .. BR-86)
# ---------------------------------------------------------------------------
RANKER = os.getenv("FSDR_RANKER", "bpr")          # bpr (implicit, default) | lightfm (if installed) (plan Q12)
RANKER_TOP_N = 100                                # BR-85
# Which offers the ranker may propose (BR-83/85). "history" = production Codes 3/4: only offers bought in
# the 270-day history after the action's family + validity filter (item_id = interaction1.BTL_NAME.unique()).
# "catalogue" = every catalogue offer of the family (the commented-out production line); needed for brand-new
# offers, which have no history yet (scenario new_offers).
RANKER_ITEMS = os.getenv("FSDR_RANKER_ITEMS", "history").lower()
BPR_PARAMS = {"factors": 32, "iterations": 60, "learning_rate": 0.05, "regularization": 0.01, "random_state": 42,
              "num_threads": 1}               # single thread: same data -> same ranking (offline llm arm = cstr arm)
CONTACT_MIN_GAP_DAYS = 2                          # see CONTACT_* below
# V1.0 review #10 / #14: offer arbitration. "value" ranks each family's offers on expected revenue
# P(accept | customer, offer) x price x OFFER_MARGIN, P from an acceptance model trained on last week's
# campaign (offers sent -> fulfilments), on offer ATTRIBUTES, so new offers get a score too (cold start); a share
# OFFER_EXPLORATION of the picks is a random family offer, so new offers get logged data. Families (BR-66/67),
# max offers (BR-82) and conflicts stay hard rules. "price_band" = production's 3 passes (BR-78..81), used as
# the fallback while the acceptance model has too little data.
OFFER_POLICY = os.getenv("FSDR_OFFER_POLICY", "value").lower()
OFFER_MARGIN = float(os.getenv("FSDR_OFFER_MARGIN", "1.0"))       # no cost data in the contract yet (review Q2)
OFFER_EXPLORATION = float(os.getenv("FSDR_OFFER_EXPLORATION", "0.05"))
ACCEPTANCE_MIN_POSITIVES = 20
# Price floor of the value policy. "anchor" keeps production's floor: every pick is priced above the line's ARPU
# anchor (as in the 3 passes, BR-78..80), so a line whose anchor is above every family price gets no offer (BR-82)
# and only the choice among the allowed offers changes. "none" lets the value policy also offer cheaper than the
# anchor (more lines contacted, more revenue in the simulation) -- a change of BR-78..82 that needs business sign-off.
OFFER_PRICE_FLOOR = os.getenv("FSDR_OFFER_PRICE_FLOOR", "anchor").lower()
LIGHTFM_PARAMS = {"no_components": 400, "loss": "warp", "k": 15, "epochs": 30}   # BR-84, used only by LightFMRanker

# ---------------------------------------------------------------------------
# Recommender agent: freedom, hard and soft rules (prompt §5.2)
# ---------------------------------------------------------------------------
EPSILON_LLM = float(os.getenv("FSDR_EPSILON_LLM", "0.1"))   # share of llm-arm decisions replaced by a bandit sample
CHURN_BLOCK_THRESHOLD = 0.60                      # plan Q9: PROBA_CHURN >= this blocks upsell-family actions
UPSELL_FAMILY = {"UPSELL", "UPSELL DATA", "UPSELL VOICE"}
# Contact cap (plan Q8, adjusted): campaigns are weekly and last week's targeting is D-8..D-2,
# so "1 contact per 7 days" would block most of the base. Hard rule = no new contact within
# CONTACT_MIN_GAP_DAYS of the last one, and at most CONTACT_CAP_28D in 28 days (campaign log, Phase 6).
CONTACT_CAP_28D = 3                               # L4 hard rule: at most 3 contacts logged in the last 28 days
CONTACT_CAP_DAYS = 28
# Training labels (H5): "auto" uses the campaign log when it holds last week's campaign (action actually sent,
# logged propensity), else the targeting tables with this week's action shares; "targeting" / "campaign_log" force one.
LABEL_SOURCE = os.getenv("FSDR_LABEL_SOURCE", "auto").lower()
# M4 price-escalation guard: cap the ARPU anchor at (1 + g) x the anchor logged for the line last time
# (campaign purchases raise next week's spend, so anchors could climb week after week).
# V1.0 review #9: on by default (10 % a week); "off" disables it (V0).
_GROWTH = os.getenv("FSDR_ANCHOR_MAX_GROWTH", "0.10").strip().lower()
ANCHOR_MAX_WEEKLY_GROWTH = None if _GROWTH in ("", "off", "none") else float(_GROWTH)
# V1.0 review #6: consent / do-not-contact (consent_contact table). A line with dnd = 1 or marketing_opt_in = 0 is
# never contacted, in any arm: hard rule, like eligibility.
CONSENT_TABLE = "consent_contact"

# ---------------------------------------------------------------------------
# Batch add-on offers: roaming and international (not in the BR catalogue; added for the batch flow)
# They are recommended BESIDE the main offers: BR-82 (max 4) applies to the main offers only,
# and an add-on may never conflict with a main offer.
# ---------------------------------------------------------------------------
ROAMING_SERVICES = [s.strip().upper() for s in os.getenv("FSDR_ROAMING_SERVICES", "ROAMING").split(",") if s.strip()]
INTERNATIONAL_SERVICES = [s.strip().upper() for s in os.getenv("FSDR_INTERNATIONAL_SERVICES", "INTERNATIONAL").split(",") if s.strip()]
ADDON_TOP_ROAMING = 2
ADDON_TOP_INTERNATIONAL = 2
INTL_MIN_CALLS_M1 = 3          # international calls in M1 (Call_Count - Local_Call_Count) to offer international packs
BATCH_SIZE = 25                # customers per batch in the batch flow (UI / API)
BATCH_MAX = 2000
# Customers decided in parallel by the agents. vLLM batches concurrent requests on the GPU, so it gets more.
# The code runs each agent's required evidence tools before the first LLM call and hands the results
# over as prior evidence: same tools, same trace, but the LLM only reasons and decides (fewer, faster
# calls, and small models stop wasting steps on refused or missing tool calls). 0 = the LLM calls them.
AGENT_PREFETCH = os.getenv("FSDR_AGENT_PREFETCH", "1") not in ("0", "false", "no")
AGENT_CONCURRENCY = int(os.getenv("FSDR_AGENT_CONCURRENCY", "8" if nbx_config.LLM_BACKEND == "vllm" else "4"))

# ---------------------------------------------------------------------------
# Validation thresholds
# ---------------------------------------------------------------------------
BLOCKING_KEY_ISSUE_SHARE = 0.01                   # null / duplicate / orphan keys above 1 % block the run
MIN_SCORE_COVERAGE = 0.70                         # below -> warning

# ---------------------------------------------------------------------------
# Weekly completeness per pipeline (master prompt §17.9, "which table is used where")
# required: missing = the run is blocked; optional: missing = warning (a fallback exists)
# ---------------------------------------------------------------------------
_ACT_AUTO = ["activities_w1_auto", "activities_w2_auto", "activities_w3_auto", "activities_w4_auto",
             "activities_m1_auto", "activities_auto_m2", "activities_m3_auto"]
_ACT_SCORE = ["activities_w1_score", "activities_w2_score", "activities_w3_score", "activities_w4_score",
              "activities_m1_score", "activities_m2_score", "activities_m3_score"]
_SCORES = [m.table for m in MODEL_SCORE_REGISTRY]
PIPELINE_TABLES = {
    "train_week0": {
        "required": ["base_prepaid", "consumer_aact_account", "access_method_id_auto", "party_auto",
                     "subs_amt_auto", *_ACT_AUTO, *_SCORES, "subscriptions_tg",
                     "usage_data_voice_payg_tg", "targeting_dataiku_auto", "targeting_auto", "subs_btl_auto",
                     "sub_btl_3months", "btl_catalog"],
        # customer_profile missing -> AGE from the party table, tenure from access_method_subs_start_dt (§2.1).
        # V4.0: week 0 reads only the *_auto history, so the D-anchored *_score tables are optional here: a weekly
        # export of fresh *_auto tables (one folder, no *_score) is a complete week-0 run.
        "optional": [PROFILE_TABLE, "location_auto", "DEVICEBEHAVIORSEGMENTATION.dv_segment", "paygo_activity_auto",
                     CONSENT_TABLE, "party_scoring", "subs_amt_score", *_ACT_SCORE],
    },
    "train_weekly": {
        "required": ["base_prepaid", "consumer_aact_account", "party_scoring", "subs_amt_score", *_ACT_SCORE, *_SCORES,
                     "subscriptions_tg", "usage_data_voice_payg_tg", "targeting_auto", "subs_btl_auto",
                     "sub_btl_3months", "btl_catalog"],
        "optional": [PROFILE_TABLE, "location_auto", "DEVICEBEHAVIORSEGMENTATION.dv_segment", "targeting_dataiku_auto",
                     CONSENT_TABLE],
    },
    "score": {   # + the validated training bundle (checked by fsdr-score itself)
        "required": ["subscriptions_tg", "usage_data_voice_payg_tg", "targeting_auto", "sub_btl_3months", "btl_catalog",
                     "BTL_script_english_arabic_ok", "conflicting_offers_h"],
        "optional": [TARGET_SAMPLE_TABLE, CONSENT_TABLE],
    },
}

# ---------------------------------------------------------------------------
# V2.0 Phase 3: features (B-01 .. B-12), S-rules, bands, micro-segments, the two actions (master prompt §6 - §8)
# ---------------------------------------------------------------------------
FEATURE_WORKERS = int(os.getenv("FSDR_FEATURE_WORKERS", "2"))     # partitions in parallel (8 on the P16, plan §15)
# B-04: the 3 churn tables are stacked in registry order (global, 0-30, 30-60); the first value per line is kept
CHURN_FEATURE = "PROBA_CHURN"
# B-05: a missing score is flagged and imputed with the median of the line's nationality group (full base)
MISSING_FLAG_SUFFIX = "_MISSING"
# B-11 / §9: the context the LLM NBA agent reads besides the scores and trends (customer_360), fixed order
CUSTOMER_360_CONTEXT = ["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND", "MICRO_SEGMENT", "VALUE_BAND", "USAGE_TYPE",
                        "REGION", "DV_SEGMENT", "SEGMENT", "RULE_ACTION", "ACTION_ORIGINAL", "ACTION_AGENT",
                        "ACTION_AGENT_RANK2", "ARPU_M1", "TENURE_DAYS", "AGE"]

# S-rules (§7): current segment of every line, first match wins
S_TENURE_NURSERING = 45                           # S-1/2: tenure < 45 days -> excluded
S_TENURE_ONBOARDING = 90                          # S-3/4
S_CHURN_AT_RISK = 0.5                             # S-1..4 and S-8 (PROBA_CHURN)
S_CHURN_1D_AT_RISK = 0.5                          # S-8 (PROBA_CHURN_1D)
S_RECONNECT_HIGH = 0.6                            # S-5
S_RECONNECT_MED = 0.3                             # S-6 (below -> S-7)
S_FRAGILITY = 0.5                                 # S-9
S_REV_DROP_PCT = -30.0                            # S-9 revenue M1 vs M2
S_VOICE_USER_MIN = 1.0                            # Local_Voice_min_M1 >= 1
S_DATA_USER_MB = 10.0                             # Data_Volume_MB_M1 >= 10
S_APPETENCY = 0.5                                 # S-11/12 (PROBA_APP_VOICE), S-13/14 (PROBA_APP_DATA)
# (rule id, segment, rule action); None = excluded (B-02). The order is the evaluation order.
S_RULES = [
    ("S-1", "Nursering_Early_Churn_At_Risk", None),
    ("S-2", "Nursering_Early_Churn_Not_At_Risk", None),
    ("S-3", "Onboarding_Early_Churn_At_Risk", "UPSELL"),
    ("S-4", "Onboarding_Early_Churn_Not_At_Risk", "UPSELL"),
    ("S-5", "Retention_High_Probability_To_Reconnect", "RETAIN"),
    ("S-6", "Retention_Med_Probability_To_Reconnect", "RETAIN"),
    ("S-7", "Retention_Low_Probability_To_Reconnect", "RETAIN"),
    ("S-8", "Retention_RGS_At_Risk", "RETAIN"),
    ("S-9", "Retention_RGS_Fragile", "UPSELL"),
    ("S-10", "RGS_Not_At_Risk_DualPlay_Upgrade", "MAINTAIN"),
    ("S-11", "Data_Only_Voice_Appetency", "UPSELL VOICE"),
    ("S-12", "Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("S-13", "Voice_Only_Data_Appetency", "UPSELL DATA"),
    ("S-14", "Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
]
S_SEGMENTS = [s for _, s, _ in S_RULES]

# B-03: the original dict_segm_actions of code 1, entry by entry in source order. It is a Python dict literal
# with repeated keys, so the LAST value wins (open item 1) and keys are case-sensitive (RGS_ vs Rgs_).
DICT_SEGM_ACTIONS_SOURCE = [
    ("Retention_Low_Probability_To_Reconnect", "RETAIN"), ("RGS_Not_At_Risk_DualPlay_Upgrade", "MAINTAIN"),
    ("Retention_RGS_At_Risk", "RETAIN"), ("Retention_RGS_Fragile", "UPSELL"),
    ("RGS_Not_At_Risk_Data_Only_Voice_Appetency", "UPSELL VOICE"), ("Retention_High_Probability_To_Reconnect", "UPSELL"),
    ("RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"), ("Onboarding_Early_Churn_At_Risk", "UPSELL"),
    ("RGS_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Onboarding_Early_Churn_Not_At_Risk", "UPSELL"),
    ("RGS_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"), ("Retention_Rgs_Fragile", "UPSELL"),
    ("Retention_Rgs_At_Risk", "RETAIN"), ("Rgs_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("Rgs_Not_At_Risk_Dualplay_Upgrade", "MAINTAIN"), ("Rgs_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
    ("Rgs_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Rgs_Not_At_Risk_Data_Only_Voice_Appetency", "UPSELL VOICE"),
    ("Retention_Med_Probability_To_Reconnect", "RETAIN"), ("Retention_Low_Probability_To_Reconnect", "RETAIN"),
    ("Retention_High_Probability_To_Reconnect", "RETAIN"), ("Rgs_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("Rgs_Not_At_Risk_Dualplay_Upgrade", "MAINTAIN"), ("Retention_Rgs_Fragile", "UPSELL"),
    ("Onboarding_Early_Churn_Not_At_Risk", "UPSELL"), ("Rgs_Not_At_Risk_Data_Only_Voice_Appetency", "CROSS SELL VOICE"),
    ("Onboarding_Early_Churn_At_Risk", "UPSELL"), ("Retention_Rgs_At_Risk", "RETAIN"),
    ("Rgs_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Rgs_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
]
DICT_SEGM_ACTIONS = dict(DICT_SEGM_ACTIONS_SOURCE)                 # last value wins, as in code 1
# §7.1: S-rule segment -> original segment name, for lines without a segment-engine segment this week.
# Every line is a CHECK item (plan §11): the S-rules re-create the segment engine, they are not the engine.
SEGMENT_CROSSWALK = {
    "Nursering_Early_Churn_At_Risk": ("Nursering_Early_Churn_At_Risk", "same name; excluded (B-02)"),
    "Nursering_Early_Churn_Not_At_Risk": ("Nursering_Early_Churn_Not_At_Risk", "same name; excluded (B-02)"),
    "Onboarding_Early_Churn_At_Risk": ("Onboarding_Early_Churn_At_Risk", "same name"),
    "Onboarding_Early_Churn_Not_At_Risk": ("Onboarding_Early_Churn_Not_At_Risk", "same name"),
    "Retention_High_Probability_To_Reconnect": ("Retention_High_Probability_To_Reconnect",
                                                "same name; original action RETAIN by last-value-wins (was UPSELL)"),
    "Retention_Med_Probability_To_Reconnect": ("Retention_Med_Probability_To_Reconnect", "same name"),
    "Retention_Low_Probability_To_Reconnect": ("Retention_Low_Probability_To_Reconnect", "same name"),
    "Retention_RGS_At_Risk": ("Retention_RGS_At_Risk", "same name (RGS_ spelling)"),
    "Retention_RGS_Fragile": ("Retention_RGS_Fragile", "same name (RGS_ spelling)"),
    "RGS_Not_At_Risk_DualPlay_Upgrade": ("RGS_Not_At_Risk_DualPlay_Upgrade", "same name (RGS_ spelling)"),
    "Data_Only_Voice_Appetency": ("RGS_Not_At_Risk_Data_Only_Voice_Appetency",
                                  "RGS_ spelling -> UPSELL VOICE (the Rgs_ spelling gives CROSS SELL VOICE)"),
    "Data_Only_No_Voice_Appetency": ("RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "RGS_ spelling"),
    "Voice_Only_Data_Appetency": ("RGS_Not_At_Risk_Voice_Only_Data_Appetency",
                                  "S-13 also takes lines with no usage; the engine may not"),
    "Voice_Only_No_Data_Appetency": ("RGS_Not_At_Risk_Voice_Only_No_Data_Appetency",
                                     "S-14 also takes lines with no usage; the engine may not"),
}

# §8 bands. Micro-segment = nationality group x age band x tenure band (5 x 4 x 6 = 120 cells)
NATIONALITY_GROUPS = ["Oman", "Pakistan", "India", "Bangladesh", "others"]
AGE_MIN, AGE_MAX = 15, 100                        # outside -> unknown
AGE_BANDS = [("young", 0, 40), ("middle", 40, 60), ("senior", 60, 200)]   # [lo, hi)
AGE_BAND_NAMES = [b for b, _, _ in AGE_BANDS] + ["unknown"]
YOUNG_SPLIT_AGE = 25                              # open item 5: under 25 / 25-39, tested in Phase 4
TENURE_BANDS = [("t0_lt3m", 0, 90), ("t1_3_6m", 90, 183), ("t2_6_12m", 183, 366), ("t3_1_2y", 366, 731),
                ("t4_gt2y", 731, 10**9)]          # days, [lo, hi)
TENURE_BAND_NAMES = [b for b, _, _ in TENURE_BANDS] + ["unknown"]
VALUE_BANDS = ["zero", "low", "mid", "high"]      # ARPU_M1 = 0, then tertiles of the paying lines (context only)
USAGE_TYPES = ["dual", "data_only", "voice_only", "none"]
AGENT_DICTIONARY_PREFIX = "dict_segm_actions_agent_v"

# ---------------------------------------------------------------------------
# V3.0: 100k contacted in four groups of 25k + control (update prompt PROMPT_FSDR_update_100k_targets.md)
# 2 x 2 design: rules (production / challenger) x offer universe (constraint = code 3 / free = code 4)
# ---------------------------------------------------------------------------
T1_CONSTRAINT_SIZE = int(os.getenv("FSDR_T1_CONSTRAINT_SIZE", "25000"))
T1_FREE_SIZE = int(os.getenv("FSDR_T1_FREE_SIZE", "25000"))
# V4.0: Target 2 only by default (the LLM agents' target), 50k a (T2C) + 50k b (T2F); Target 1 is off (FSDR_TARGETS)
T2_CONSTRAINT_SIZE = int(os.getenv("FSDR_T2_CONSTRAINT_SIZE", "50000"))
T2_FREE_SIZE = int(os.getenv("FSDR_T2_FREE_SIZE", "50000"))
CONTROL_SIZE = int(os.getenv("FSDR_CONTROL_SIZE", "0"))            # V4.0: no control group by default; 20000 = a baseline
T2_SPLIT = os.getenv("FSDR_T2_SPLIT", "true").lower() == "true"     # false -> one 50k T2 with the CC rules
GROUP_SIZE_TOLERANCE = 0.02                       # A-30: sizes within +-2 %
BALANCE_MAX_POINTS = 2.0                          # A-31: stratum shares differ by at most 2 points between groups

# One entry per contacted group. universe: "constraint" (code 3 families by action) | "free" (code 4: L2,
# DATA / COMBO, action not used). rules: "code3" | "code4" | "CC" | "CF". rule_action: the NBA hint and fallback.
GROUPS = {
    "t1c": {"panel_group": "t1_constraint", "target": "target1", "universe": "constraint", "rules": "code3",
            "agent": "6A-C", "tg_flag": "constraint", "rule_action": "ACTION_ORIGINAL", "nbo_sees_action": True,
            "consistency_loop": True, "rules_version": "production_constraint",
            "scored_table": "TG_40K_constraint_scored_nba", "recos_monthly": "recos_monthly_40k_constraint",
            "recos_weekly": "recos_weekly_40k_constraint", "action_column": True},
    "t1f": {"panel_group": "t1_free", "target": "target1", "universe": "free", "rules": "code4",
            "agent": "6A-F", "tg_flag": "free", "rule_action": "ACTION_ORIGINAL",
            "nbo_sees_action": os.getenv("FSDR_T1F_NBO_SEES_ACTION", "false").lower() == "true",
            "consistency_loop": False, "rules_version": "production_free",
            "scored_table": "TG_40k_free_scored_nba", "recos_monthly": "recos_monthly_tg_free",
            "recos_weekly": "recos_weekly_tg_free", "action_column": False},
    "t2c": {"panel_group": "t2_constraint", "target": "target2", "universe": "constraint", "rules": "CC",
            "agent": "6B-C", "tg_flag": "challenger_constraint", "rule_action": "ACTION_AGENT", "nbo_sees_action": True,
            "consistency_loop": True, "rules_version": "CC-v1",
            "scored_table": "TG_challenger_constraint_scored_nba", "recos_monthly": "recos_monthly_challenger_constraint",
            "recos_weekly": "recos_weekly_challenger_constraint", "action_column": True},
    "t2f": {"panel_group": "t2_free", "target": "target2", "universe": "free", "rules": "CF",
            "agent": "6B-F", "tg_flag": "challenger_free", "rule_action": "ACTION_AGENT",
            "nbo_sees_action": os.getenv("FSDR_T2F_NBO_SEES_ACTION", "false").lower() == "true",
            "consistency_loop": False, "rules_version": "CF-v1",
            "scored_table": "TG_challenger_free_scored_nba", "recos_monthly": "recos_monthly_challenger_free",
            "recos_weekly": "recos_weekly_challenger_free", "action_column": False},
}
CONTROL_GROUP = "control"
NOT_SELECTED = "not_selected"
PANEL_GROUPS = [g["panel_group"] for g in GROUPS.values()] + [CONTROL_GROUP, NOT_SELECTED]
# the provided test sample (master §2.4): its "group" column -> our groups; T2C / T2F come from the rest of the base
SAMPLE_GROUP_MAP = {"cstr": "t1c", "free": "t1f", "cg": "control"}
# Tables of the earlier update prompts, replaced by the 4-group names above (kept out of every new output)
SUPERSEDED_TABLES = ["TG_40k_challenger_scored_nba", "recos_monthly_challenger", "recos_weekly_challenger",
                     "TG_40k_challenger_constraint_scored_nba", "TG_40k_challenger_free_scored_nba"]


def group_sizes() -> dict[str, int]:
    """Configured size of every panel group of the ACTIVE targets (t2_split false: one T2 of both sizes, CC rules)."""
    sizes = {}
    if "target1" in TARGETS:
        sizes.update(t1c=T1_CONSTRAINT_SIZE, t1f=T1_FREE_SIZE)
    if "target2" in TARGETS:
        if T2_SPLIT:
            sizes.update(t2c=T2_CONSTRAINT_SIZE, t2f=T2_FREE_SIZE)
        else:
            sizes["t2c"] = T2_CONSTRAINT_SIZE + T2_FREE_SIZE
    if CONTROL_SIZE > 0:                                # V4.0 default 0: no control list, no control file
        sizes[CONTROL_GROUP] = CONTROL_SIZE
    return sizes


# Code 4 / FR rules (T1F) and the free universe of T2F
FREE_UNIVERSE_GENEROSITY = "L2"                   # FR-02
FREE_UNIVERSE_SERVICES = ("COMBO", "DATA")        # FR-03 no VOICE; order = (type A, type B) as code 4
FREE_PRICE_K = 3.0                                # FR-09
RECOMMENDABLE_HISTORY_DAYS = 270                  # B-20 / B-20b (sub_btl_3months)
ONLY_OWN_PAST_OFFERS = os.getenv("FSDR_ONLY_OWN_PAST_OFFERS", "false").lower() == "true"   # B-20b switch
MAX_CANDIDATES_PER_VALIDITY = 25                  # §10: candidates per validity sent to the NBO agent
TARGET1_FREE_SHARE = 0.0                          # B-23: the code-4 part now lives in its own group (T1F)

# Challenger CC-v1 (T2C) and CF-v1 (T2F): open item 4, approved before the first campaign
CHALLENGER_BAND = (1.0, 1.5)                      # CC-02 / CF-02: price between 1.0x and 1.5x the anchor
CHALLENGER_BAND_WIDENED = (1.0, 2.0)              # CC-04 / CF-04: widened once, upwards only
CHALLENGER_UPSELL_STEP = 0.5                      # minimum +0.5 OMR above the anchor for the UPSELL actions
UPSELL_ACTIONS = ["UPSELL", "UPSELL DATA", "UPSELL VOICE"]
DEFAULT_DATA_SHARE = 0.9                          # CC-03 / CF-03: no spend history -> 90 / 10
DO_NOT_DISTURB_IGNORED = 3                        # CC-05 / CF-05
GUARD_MODE_TARGET1 = os.getenv("FSDR_GUARD_MODE_TARGET1", "repair")   # open item 3: "repair" | "report"

# LLM budget at 100k (update §6)
LLM_WEEKLY_BUDGET_H = float(os.getenv("FSDR_LLM_WEEKLY_BUDGET_H", "10"))
LEVEL2_MAX_OUTPUT_TOKENS = 120
LEVEL2_MAX_INPUT_TOKENS = 1500

# ---------------------------------------------------------------------------
# V3.0 target upload folders and the final output (code 5, F-01 .. F-19)
# ---------------------------------------------------------------------------
# V4.0: the lists sit directly in the scoring folder, one sub-folder per target and per a / b group:
#   data/scoring/<D>/target1/a_constraint/target1_a.csv.gz   T1C   target1/b_free/target1_b.csv.gz   T1F
#   data/scoring/<D>/target2/a_constraint/target2_a.csv.gz   T2C   target2/b_free/target2_b.csv.gz   T2F
#   data/scoring/<D>/control/control.csv.gz                   never contacted
TARGETS_FOLDER = os.getenv("FSDR_TARGETS_FOLDER", ".")
ALL_TARGET_FILES = {"t1c": "target1/a_constraint/target1_a", "t1f": "target1/b_free/target1_b",
                    "t2c": "target2/a_constraint/target2_a", "t2f": "target2/b_free/target2_b", "control": "control/control"}
ALL_TARGETS = {"target1": ["t1c", "t1f"], "target2": ["t2c", "t2f"]}   # scored separately (V4.0)
GROUP_SUBFOLDER = {k: v.split("/")[1] for k, v in ALL_TARGET_FILES.items() if k != "control"}   # t1c -> a_constraint
# V4.0: the targets this deployment scores. Default: Target 2 only (where the LLM agents decide); Target 1 (the
# production rules) is not scored, its lists are not read and its folders are not written. FSDR_TARGETS=target1,target2
# brings it back.
TARGETS: dict[str, list[str]] = {}
TARGET_FILES: dict[str, str] = {}


def set_targets(names) -> None:
    """Activates these targets (in place, so every module sees it): TARGETS and the list files they read."""
    names = [n for n in names if n]
    unknown = [n for n in names if n not in ALL_TARGETS]
    if unknown or not names:
        raise ValueError(f"FSDR_TARGETS must name one or more of {list(ALL_TARGETS)}, got {names}")
    TARGETS.clear()
    TARGETS.update({t: ALL_TARGETS[t] for t in ALL_TARGETS if t in names})
    TARGET_FILES.clear()
    TARGET_FILES.update({k: v for k, v in ALL_TARGET_FILES.items()
                         if (k == CONTROL_GROUP and CONTROL_SIZE > 0) or any(k in g for g in TARGETS.values())})


set_targets([t.strip() for t in os.getenv("FSDR_TARGETS", "target2").split(",")])
# V4.0: recommended offers per customer. 6 = 3 monthly + 3 weekly (slots 1-2 monthly, 3-4 weekly, 5 = monthly
# rank 3, 6 = weekly rank 3; a favourite / forced offer only fills a slot 5 / 6 left empty). 4 = the V3.0 file.
RECO_OFFERS = int(os.getenv("FSDR_RECO_OFFERS", "6"))
if RECO_OFFERS not in (4, 6):
    raise ValueError("FSDR_RECO_OFFERS must be 4 or 6")
RECOS_PER_VALIDITY = RECO_OFFERS // 2
TARGETS_MANIFEST = "_targets_manifest.json"
# F-08 forced bundles in slot 6, active version (code 5 "Change the forced offers above to L2 offers"), in order:
# (anchor column, low, high, offer, popular list that blocks the forcing)
_POP_CM = ["CM_3R_8GB_240MIN", "CM_4R_10GB_300MIN", "CM_5R_14GB_360MIN"]
_POP_DM = _POP_CM + ["DM_3R_10GB", "DM_4R_14GB", "DM_5R_18GB"]
FORCED_BUNDLES = [("F05_MONTHLY", 0, 3, "CM_3R_8GB_240MIN", _POP_CM), ("F05_MONTHLY", 3, 4, "CM_4R_10GB_300MIN", _POP_CM),
                  ("F05_MONTHLY", 4, 5, "CM_5R_14GB_360MIN", _POP_CM), ("F05_DATA_MONTHLY", 0, 3, "DM_3R_10GB", _POP_DM),
                  ("F05_DATA_MONTHLY", 3, 4, "DM_4R_14GB", _POP_DM), ("F05_DATA_MONTHLY", 4, 5, "DM_5R_18GB", _POP_DM)]
FAVOURITE_ARPU_FILTER = False                     # F-07 (code 5 commented it out)
OPPORTUNITY_PREFIX = "nbo_LFM_"                   # F-18, expected by the campaign platform
INPUT_FLYTXT_COLUMNS = (["date", "start_date", "end_date", "msisdn"] + [f"channel_{i}" for i in range(1, 7)]
                        + [f"opportunity_name_{i}" for i in range(1, 7)] + [f"destination_offer_name_{i}" for i in range(1, 7)]
                        + ["flag_splio"] + [f"offer_script_en_{i}" for i in range(1, 7)]
                        + [f"offer_script_ar_{i}" for i in range(1, 7)] + ["segment", "segment_btl_taker", "segment_btl_taker_upsell"])


In [ ]:
%%engine nbx.fsdr.calendar
"""Anchor date and every aggregation window of the business rules (BR §4).

D = run date, T = campaign anchor = most recent Thursday on or before D.
Windows are inclusive [start, end] date ranges. The generator aggregates its
daily history with exactly these windows, and the validation / features use
them to know what each column covers, so the two can never disagree.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta

from nbx.fsdr import config

WEEKS = ["W1", "W2", "W3", "W4"]      # W1 = most recent week
MONTHS = ["M1", "M2", "M3"]           # M1 = most recent month
PERIODS = WEEKS + MONTHS


@dataclass(frozen=True)
class Window:
    start: date
    end: date

    @property
    def days(self) -> int:
        return (self.end - self.start).days + 1

    def contains(self, d: date) -> bool:
        return self.start <= d <= self.end

    def as_dict(self) -> dict:
        return {"start": self.start.isoformat(), "end": self.end.isoformat(), "days": self.days}


def anchor_date(run_date: date) -> date:
    """T = D - (((D - 2026-03-12) MOD 7) + 7) MOD 7 -- the last Thursday on or before D."""
    return run_date - timedelta(days=(((run_date - config.REFERENCE_THURSDAY).days % 7) + 7) % 7)


def _w(anchor: date, start_off: int, end_off: int) -> Window:
    return Window(anchor + timedelta(days=start_off), anchor + timedelta(days=end_off))


# Offsets (start, end) relative to the anchor, per BR §4.
_SUBS_OFFSETS = {                     # BR 4.1 -- window INCLUDES the anchor day
    "W1": (-6, 0), "W2": (-13, -7), "W3": (-20, -14), "W4": (-27, -21),
    "M1": (-29, 0), "M2": (-59, -30), "M3": (-89, -60),
}
_ACTIVITY_OFFSETS = {                 # BR 4.2 -- window EXCLUDES the anchor day
    "W1": (-7, -1), "W2": (-14, -8), "W3": (-21, -15), "W4": (-28, -22),
    "M1": (-28, -1), "M2": (-56, -29), "M3": (-85, -57),
}
_SUBSCRIPTIONS_TG_OFFSETS = {         # BR 4.3 subscriptions_tg, anchored on D (2-day lag)
    "W1": (-8, -2), "W2": (-15, -9), "W3": (-22, -16), "W4": (-29, -23),
    "M1": (-31, -2), "M2": (-61, -32), "M3": (-91, -62),
}
_USAGE_TG_VOLUME_OFFSETS = {          # BR 4.3 usage_data_voice_payg_tg voice & data p1..p3
    "p1": (-31, -2), "p2": (-61, -32), "p3": (-91, -62),
}
_PAYG_OFFSETS = {                     # BR 4.3 PAYG windows (usage_data_voice_payg_tg on D, paygo_activity_auto on T)
    "M1": (-30, -1), "M2": (-60, -31), "M3": (-90, -61),
    "W1": (-7, -1), "W2": (-14, -8), "W3": (-21, -15), "W4": (-28, -22),
}


def subs_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _SUBS_OFFSETS.items()}


def activity_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _ACTIVITY_OFFSETS.items()}


def subscriptions_tg_windows(run_date: date) -> dict[str, Window]:
    return {p: _w(run_date, *o) for p, o in _SUBSCRIPTIONS_TG_OFFSETS.items()}


def usage_tg_volume_windows(run_date: date) -> dict[str, Window]:
    return {p: _w(run_date, *o) for p, o in _USAGE_TG_VOLUME_OFFSETS.items()}


def payg_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _PAYG_OFFSETS.items()}


def campaign_windows(run_date: date) -> dict[str, Window]:
    """BR-21 .. BR-24, relative to D."""
    return {
        "targeting": _w(run_date, *config.TARGETING_WINDOW),      # BR-21
        "arbitration": _w(run_date, *config.ARBITRATION_WINDOW),  # BR-22
        "reward": _w(run_date, *config.REWARD_WINDOW),            # BR-23
        "interactions": _w(run_date, *config.INTERACTION_WINDOW),  # BR-24
    }


def snapshot_date(run_date: date) -> date:
    """BR-01: base snapshot is ACTIVITY_DATE = D-2."""
    return run_date - timedelta(days=config.SNAPSHOT_LAG_DAYS)


def location_month(run_date: date) -> tuple[int, int]:
    """BR-13: previous calendar month of D-2 -> (month, year); January rolls back to December."""
    snap = snapshot_date(run_date)
    return (12, snap.year - 1) if snap.month == 1 else (snap.month - 1, snap.year)


def usage_history_start(run_date: date) -> date:
    """First day any usage / subscription window can need (D-96)."""
    return run_date - timedelta(days=config.HISTORY_DAYS_USAGE - 1)


def all_windows(run_date: date) -> dict:
    """Every window for D, for the manifest, the UI and the docs."""
    t = anchor_date(run_date)
    d = run_date
    return {
        "run_date": d.isoformat(),
        "anchor_date": t.isoformat(),
        "snapshot_date": snapshot_date(d).isoformat(),
        "location_month": dict(zip(("month", "year"), location_month(d))),
        "subs_amt_auto": {p: w.as_dict() for p, w in subs_windows(t).items()},
        "subs_amt_score": {p: w.as_dict() for p, w in subs_windows(d).items()},
        "activities_auto": {p: w.as_dict() for p, w in activity_windows(t).items()},
        "activities_score": {p: w.as_dict() for p, w in activity_windows(d).items()},
        "subscriptions_tg": {p: w.as_dict() for p, w in subscriptions_tg_windows(d).items()},
        "usage_tg_volume": {p: w.as_dict() for p, w in usage_tg_volume_windows(d).items()},
        "usage_tg_payg": {p: w.as_dict() for p, w in payg_windows(d).items()},
        "paygo_activity_auto": {p: w.as_dict() for p, w in payg_windows(t).items()},
        "campaign": {k: w.as_dict() for k, w in campaign_windows(d).items()},
    }


In [ ]:
%%engine nbx.fsdr.schemas
"""Data contract: the exact raw schema of every input file.

One file per table, file name = table name + ".json" (array), ".jsonl", ".csv" or ".parquet", filed in one of the
5 input folders (usage / models / subscription / catalog / others, plan §13): TableSchema.folder.
Field names, case and production quirks are kept on purpose
(`UBSCRIPTION_ID`, `activities_auto_m2`, `Access_Method_id`, "Price data",
numbers stored as strings in `btl_catalog` and in the test sample), so the
same code runs on the dummy data and on real exports.

Raw types follow the production tables: string, int, bigint, tinyint,
double, date ("YYYY-MM-DD") and timestamp (ISO 8601). In JSON, every value
may also be null unless the field is a key.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS, WEEKS

INT_TYPES = {"int", "bigint", "tinyint"}
RAW_TYPES = INT_TYPES | {"string", "double", "date", "timestamp"}
_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


@dataclass(frozen=True)
class Field:
    name: str
    raw_type: str
    key: bool = False           # join key: must be present and non-null


@dataclass
class TableSchema:
    name: str
    fields: list[Field]
    description: str
    group: str
    anchor: str | None = None               # "T", "D" or None
    primary_key: tuple[str, ...] = ()       # duplicates on it are reported
    duplicates_blocking: bool = True        # False: duplicates are expected / resolved downstream
    used_by_model: bool = True              # BR §12: some tables are loaded but not used
    family: str | None = None               # e.g. "model_score", "activities"
    required: bool = True                  # False: a missing file is a warning (a fallback exists)

    def __post_init__(self):
        for f in self.fields:
            if f.raw_type not in RAW_TYPES:
                raise ValueError(f"{self.name}.{f.name}: unknown raw type {f.raw_type}")

    @property
    def field_names(self) -> list[str]:
        return [f.name for f in self.fields]

    @property
    def key_fields(self) -> list[str]:
        return [f.name for f in self.fields if f.key]

    def types(self) -> dict[str, str]:
        return {f.name: f.raw_type for f in self.fields}

    @property
    def folder(self) -> str:
        """Input sub-folder of the table (plan §13)."""
        return folder_of(self.name, self.family)


def folder_of(table: str, family: str | None = None) -> str:
    if family == "model_score":
        return "models"
    for folder, names in config.TABLE_FOLDERS.items():
        if table in names:
            return folder
    return "others"


def _f(name: str, raw_type: str, key: bool = False) -> Field:
    return Field(name, raw_type, key)


# ---------------------------------------------------------------------------
# Field builders for the repetitive table families
# ---------------------------------------------------------------------------
ACTIVITY_METRICS = [  # (name stem, raw type) -- SQL order of activities_*
    ("Call_Count", "int"), ("Data_Session_Count", "int"), ("Voice_Traffic_min", "double"),
    ("Data_Volume_MB", "double"), ("PG_Total_Amount", "double"), ("PG_Voice_Amount", "double"),
    ("PG_Data_Amount", "double"), ("Onnet_Call_Count", "int"), ("Onnet_Voice_min", "double"),
    ("Offnet_Call_Count", "int"), ("Offnet_Voice_min", "double"), ("Local_Call_Count", "int"),
    ("Local_Voice_min", "double"), ("Active_Days_Count", "int"), ("Active_Days_Voice_Count", "int"),
    ("Active_Days_Data_Count", "int"),
]


def activity_table_name(period: str, anchor: str) -> str:
    """activities_w1_auto ... activities_m3_auto, except M2 auto = activities_auto_m2 (production name)."""
    p = period.lower()
    if anchor == "T":
        return "activities_auto_m2" if p == "m2" else f"activities_{p}_auto"
    return f"activities_{p}_score"


def _activity_fields(period: str) -> list[Field]:
    return [_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "bigint", True),
            _f("ACCOUNT_ID", "int", True)] + [_f(f"{stem}_{period}", t) for stem, t in ACTIVITY_METRICS]


def _subs_amt_fields() -> list[Field]:
    return ([_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True),
             _f("ACCOUNT_ID", "double", True)]
            + [_f(f"Sub_Amt_{p.lower()}", "double") for p in WEEKS + MONTHS])


def _subscriptions_tg_fields() -> list[Field]:
    out = [_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True), _f("ACCOUNT_ID", "double", True)]
    for w in WEEKS:
        out += [_f(f"{k}_Sub_Amt_{w.lower()}", "double") for k in ("Total", "Data", "Voice")]
    for m in MONTHS:
        out += [_f(f"Sub_Count_{m}", "int")] + [_f(f"{k}_Sub_Amt_{m}", "double") for k in ("Total", "Data", "Voice")]
    return out


def _payg_fields(prefix_cols: list[Field]) -> list[Field]:
    return prefix_cols + [_f(f"PG_Total_Amount_{p.lower()}", "double") for p in MONTHS + WEEKS]


def _targeting_auto_fields() -> list[Field]:
    out = [_f("DATE1", "date"), _f("START_DATE", "date"), _f("END_DATE", "date"), _f("MSISDN", "string", True)]
    first, last = range(1, 5), range(5, 7)
    out += [_f(f"CHANNEL_{i}", "string") for i in first]
    out += [_f(f"OPPORTUNITY_NAME_{i}", "string") for i in first]
    out += [_f(f"DESTINATION_OFFER_NAME_{i}", "string") for i in first]
    out += [_f("FLAG_SPLIO", "string")]
    out += [_f(f"OFFER_SCRIPT_EN_{i}", "string") for i in first]
    out += [_f(f"OFFER_SCRIPT_AR_{i}", "string") for i in first]
    out += [_f("SEGMENT", "string"), _f("FILENAME", "string"), _f("ETL_CREATE_DT", "date"), _f("JOB_ID", "int")]
    for i in last:
        out += [_f(f"CHANNEL_{i}", "string"), _f(f"OPPORTUNITY_NAME_{i}", "string"),
                _f(f"DESTINATION_OFFER_NAME_{i}", "string"), _f(f"OFFER_SCRIPT_EN_{i}", "string"),
                _f(f"OFFER_SCRIPT_AR_{i}", "string")]
    return out + [_f("SEGMENT_BTL_TAKER", "string"), _f("SEGMENT_BTL_TAKER_UPSELL", "string")]


_CONFLICT_SIDE = ["Offer", "Gen_Level", "Price", "Data", "Min", "Validity", "Offer_Type"]

# The 11 model-score files share ONE identical structure.
MODEL_SCORE_FIELDS = [
    _f("subscription_id", "string"), _f("access_method_val", "bigint", True), _f("access_method_id", "bigint", True),
    _f("tenure", "bigint"), _f("life_in_nw_range", "string"), _f("proba_0", "double"), _f("proba_1", "double"),
    _f("prediction", "string"),
]

SAMPLE_FIELDS = [  # every column is a string in production
    "access_method_val", "access_method_id", "account_id", "tenure", "activation_date", "plan_name", "payg_revenue",
    "subscription_revenue", "total_revenue", "total_revenue_3m", "voice_traffic", "data_volume", "nationality",
    "region", "ARPU", "ARPU_3m", "voice_trf_group", "data_trf_group", "Final", "channel", "group",
    "strata", "counts",
]
_SAMPLE_KEYS = {"access_method_val", "access_method_id", "account_id", "group"}


def model_score_schema(table: str) -> TableSchema:
    return TableSchema(table, list(MODEL_SCORE_FIELDS), "Model score (one shared structure; proba_1 is the NBA score).",
                       "model_scores", primary_key=("access_method_val", "access_method_id"),
                       duplicates_blocking=False, family="model_score")


def _build_tables() -> dict[str, TableSchema]:
    t: list[TableSchema] = [
        # --- Base & reference --------------------------------------------------
        TableSchema("base_prepaid", [
            _f("UBSCRIPTION_ID", "string"), _f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True),
            _f("ACCOUNT_ID", "int", True), _f("PARTY_ID", "int", True), _f("access_method_subs_start_dt", "string"),
            _f("CURRENT_PLAN", "string")],
            "Active prepaid consumer lines, snapshot D-2 (BR-01..BR-10).", "base", primary_key=("ACCESS_METHOD_VAL",)),
        TableSchema("consumer_aact_account", [_f("ACCOUNT_ID", "bigint", True), _f("PARTY_ID", "bigint", True)],
                    "Active consumer account -> party (BR-11).", "base", primary_key=("ACCOUNT_ID",)),
        TableSchema("access_method_id_auto", [
            _f("ACCESS_METHOD_ID", "int", True), _f("MSISDN", "string", True), _f("ACTIVATION_DATE", "string")],
            "Line id -> MSISDN with 3-character country prefix (BR-12).", "base",
            primary_key=("ACCESS_METHOD_ID", "MSISDN")),
        *[TableSchema(name, [_f("party_id", "int", True), _f("AGE", "int"), _f("GENDER", "string"),
                             _f("NATIONALITY", "string"), _f("nationality_identifier", "string")],
                      "Customer demographics (BR-15).", "base", primary_key=("party_id",))
          for name in ("party_auto", "party_scoring")],
        TableSchema("location_auto", [
            _f("Access_Method_id", "int", True), _f("Calling_Month", "int"), _f("Calling_Year", "int"),
            _f("REGION", "string"), _f("WILAYAT", "string"), _f("AREA", "string")],
            "Home location, previous calendar month of D-2 (BR-13/14). Not used by the model.", "base",
            primary_key=("Access_Method_id", "REGION", "WILAYAT", "AREA"), used_by_model=False),
        TableSchema("btl_catalog", [_f("btl_name", "string", True)] + [_f(n, "string") for n in (
            "type", "generosity", "gb", "min", "price", "Price data", "Price voice", "service", "validity")],
            "BTL offer catalogue; every column is a string (BR-63..65).", "base", primary_key=("btl_name",)),
        TableSchema("BTL_script_english_arabic_ok", [
            _f("Sl No.", "string"), _f("destination_offer_name", "string", True), _f("English Script", "string"),
            _f("Arabic Script", "string")],
            "English / Arabic script per offer.", "base", primary_key=("destination_offer_name",), used_by_model=False),
        TableSchema("conflicting_offers_h",
                    [_f(f"{c}_A", "string", c == "Offer") for c in _CONFLICT_SIDE]
                    + [_f(f"{c}_B", "string", c == "Offer") for c in _CONFLICT_SIDE]
                    + [_f(n, "string") for n in ("Price_Difference", "Data_Difference", "SAME_VALIDITY", "SAME_OFFER_TYPE")],
                    "Pairs of offers that must never be sent together.", "base", primary_key=("Offer_A", "Offer_B")),
        TableSchema(config.CONSENT_TABLE, [
            _f("access_method_val", "string", True), _f("dnd", "tinyint"), _f("marketing_opt_in", "tinyint"),
            _f("updated_at", "date")],
            "V1.0 consent / do-not-contact per line (review #6): dnd = 1 or marketing_opt_in = 0 -> never contacted. "
            "Optional: without it no consent check runs (warning).", "base", primary_key=("access_method_val",),
            used_by_model=False),
        TableSchema(config.PROFILE_TABLE, [_f("msisdn", "string", True), _f("age", "int"), _f("tenure", "int")],
                    f"V2.0 age and tenure per line ({config.PROFILE_TENURE_UNIT}), provided weekly: age and tenure bands "
                    "of the micro-segments. Overrides party_*.AGE and the activation-date tenure. Optional: without it "
                    "AGE comes from party_scoring and tenure from access_method_subs_start_dt.", "base",
                    primary_key=("msisdn",), required=False),
        TableSchema("DEVICEBEHAVIORSEGMENTATION.dv_segment", [_f("ACCESS_METHOD_ID", "int", True), _f("DV_SEGMENT", "string")],
                    "Device behaviour segment. Not used by the model.", "base", primary_key=("ACCESS_METHOD_ID",),
                    used_by_model=False),
        # --- Revenue & usage ---------------------------------------------------
        TableSchema("subs_amt_auto", _subs_amt_fields(), "Subscription amounts, windows on T (BR 4.1).", "usage", "T",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="subs_amt"),
        TableSchema("subs_amt_score", _subs_amt_fields(), "Subscription amounts, windows on D (BR 4.1).", "usage", "D",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="subs_amt"),
        *[TableSchema(activity_table_name(p, a), _activity_fields(p),
                      f"Usage activity {p}, windows on {a} (BR 4.2).", "usage", a,
                      ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="activities")
          for a in ("T", "D") for p in WEEKS + MONTHS],
        TableSchema("paygo_activity_auto", _payg_fields([
            _f("ACCESS_METHOD_ID", "bigint", True), _f("ACCOUNT_ID", "int", True), _f("ACCESS_METHOD_VAL", "string", True)]),
            "PAYG amounts on T (BR 4.3). Loaded but not used.", "usage", "T",
            ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), used_by_model=False),
        TableSchema("subscriptions_tg", _subscriptions_tg_fields(),
                    "Offer-pricing subscriptions on D, 2-day lag (BR 4.3, BR-19 SUBSCRIPTION_AMT).", "pricing", "D",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID")),
        TableSchema("usage_data_voice_payg_tg", _payg_fields([
            _f("ACCESS_METHOD_ID", "bigint", True), _f("ACCOUNT_ID", "int", True), _f("ACCESS_METHOD_VAL", "string", True)]
            + [_f(f"local_Voice_min_p{i}", "double") for i in (1, 2, 3)]
            + [_f(f"data_volume_mb_p{i}", "double") for i in (1, 2, 3)]),
            "Offer-pricing usage + positive PAYG on D (BR 4.3, BR-33/34).", "pricing", "D",
            ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID")),
        # --- Campaign ----------------------------------------------------------
        TableSchema("targeting_dataiku_auto", [
            _f("date1", "string"), _f("access_method_id", "int", True), _f("account_id", "int"), _f("segment", "string")]
            + [_f(f"destination_offer_name_{i}", "string") for i in range(1, 7)],
            "Last campaign targeting, date1 in D-8..D-2 (BR-21, BR-36..39).", "campaign", "D",
            ("access_method_id", "date1"), duplicates_blocking=False),
        TableSchema("targeting_auto", _targeting_auto_fields(),
                    "Arbitration targeting, start_date in D-7..D (BR-22). Loaded but not used.", "campaign", "D",
                    duplicates_blocking=False, used_by_model=False),
        TableSchema("subs_btl_auto", [
            _f("access_method_val", "string", True), _f("PRODUCTNAME", "string", True),
            _f("FULFILMENTSUCCESSDATE_TIME", "timestamp"), _f("sub_flag", "tinyint"), _f("nb_sub", "int")],
            "Fulfilments D-7..D-1, MSISDN without prefix (BR-23/25).", "campaign", "D",
            ("access_method_val", "PRODUCTNAME", "FULFILMENTSUCCESSDATE_TIME")),
        TableSchema("sub_btl_3months", [
            _f("access_method_val", "string", True), _f("PRODUCTNAME", "string", True), _f("sub_flag", "tinyint"),
            _f("nb_sub", "int")],
            "Fulfilments D-270..D-1 despite the name (BR-24).", "campaign", "D", ("access_method_val", "PRODUCTNAME")),
        # --- Test sample -------------------------------------------------------
        TableSchema(config.TARGET_SAMPLE_TABLE, [_f(n, "string", n in _SAMPLE_KEYS) for n in SAMPLE_FIELDS],
                    "Campaign test sample: control cg + arms free / cstr / llm, stratified (BR-59..62). All strings.",
                    "sample", "D", ("access_method_val",)),
    ]
    tables = {s.name: s for s in t}
    for m in config.MODEL_SCORE_REGISTRY:
        tables.setdefault(m.table, model_score_schema(m.table))
    return tables


TABLES: dict[str, TableSchema] = _build_tables()

# Accepted alternative file names (production typos) -> canonical table.
TABLE_ALIASES = {"activities_w1_score_hfds": "activities_w1_score"}


def refresh() -> None:
    """Rebuilds TABLES after config.MODEL_SCORE_REGISTRY was changed (tests, plug-in models)."""
    global TABLES
    TABLES = _build_tables()


def model_score_tables() -> list[str]:
    return [m.table for m in config.MODEL_SCORE_REGISTRY]


# ---------------------------------------------------------------------------
# Cross-table key relations (validated: child keys must exist in the parent)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Relation:
    child: str
    child_col: str
    parent: str
    parent_col: str
    blocking: bool = True
    note: str = ""


def relations() -> list[Relation]:
    rels = [
        Relation("base_prepaid", "ACCESS_METHOD_ID", "access_method_id_auto", "ACCESS_METHOD_ID"),
        Relation("base_prepaid", "PARTY_ID", "party_auto", "party_id"),
        Relation("base_prepaid", "ACCOUNT_ID", "consumer_aact_account", "ACCOUNT_ID", blocking=False),
        Relation("targeting_dataiku_auto", "access_method_id", "access_method_id_auto", "ACCESS_METHOD_ID"),
        Relation(config.TARGET_SAMPLE_TABLE, "access_method_id", "base_prepaid", "ACCESS_METHOD_ID"),
        Relation(config.TARGET_SAMPLE_TABLE, "account_id", "consumer_aact_account", "ACCOUNT_ID"),
        Relation("subs_amt_auto", "ACCESS_METHOD_VAL", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        Relation("activities_w1_auto", "ACCESS_METHOD_VAL", "access_method_id_auto", "MSISDN", blocking=False),
        # B6: every fulfilment line must belong to a known line (reported, not blocking: churned lines buy too)
        Relation("subs_btl_auto", "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        Relation("sub_btl_3months", "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        *[Relation("targeting_dataiku_auto", f"destination_offer_name_{i}", "btl_catalog", "btl_name", blocking=False)
          for i in range(1, 7)],
        Relation("conflicting_offers_h", "Offer_A", "btl_catalog", "btl_name", blocking=False),
        Relation("conflicting_offers_h", "Offer_B", "btl_catalog", "btl_name", blocking=False),
        Relation(config.PROFILE_TABLE, "msisdn", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False,
                 note="profile of a line outside the base"),
    ]
    rels += [Relation(t, "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False,
                      note="model score for a line outside the base") for t in model_score_tables()]
    return rels


# ---------------------------------------------------------------------------
# Raw-type checks and coercion
# ---------------------------------------------------------------------------
def value_matches(raw_type: str, v) -> bool:
    """Does one non-null JSON value match the raw type?"""
    if raw_type == "string":
        return isinstance(v, str)
    if raw_type in INT_TYPES:
        return isinstance(v, (int, np.integer)) and not isinstance(v, bool)
    if raw_type == "double":
        return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)
    if raw_type == "date":
        return isinstance(v, str) and bool(_DATE_RE.match(v))
    if raw_type == "timestamp":
        if not isinstance(v, str):
            return False
        try:
            datetime.fromisoformat(v)
            return True
        except ValueError:
            return False
    return False


def type_errors(series: pd.Series, raw_type: str, sample: int = 3) -> tuple[int, list]:
    """(number of non-null values with the wrong raw type, a few examples).

    value_matches depends only on the Python type of the value, except for dates and timestamps (content), so
    the check runs once per type present in the column, and the content checks on strings only (review #15:
    value_matches was called once per value, 45 M times at 100 k lines)."""
    s = series[series.notna()]
    if s.empty:
        return 0, []
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind in ("string", "integer", "floating", "boolean"):
        groups = [(s.iloc[0].__class__, np.ones(len(s), bool))]
    else:
        types = s.map(type)
        groups = [(t, (types == t).to_numpy()) for t in types.unique()]
    ok = np.zeros(len(s), bool)
    for t, m in groups:
        sub = s[m]
        if raw_type == "date" and issubclass(t, str):
            ok[m] = sub.str.match(_DATE_RE.pattern).fillna(False).to_numpy(dtype=bool)
        elif raw_type == "timestamp" and issubclass(t, str):
            res = {u: value_matches(raw_type, u) for u in pd.unique(sub)}
            ok[m] = sub.map(res).to_numpy(dtype=bool)
        else:
            ok[m] = value_matches(raw_type, sub.iloc[0])
    bad = s[~ok]
    return int(len(bad)), [repr(v) for v in bad.head(sample).tolist()]


def coerce(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Raw (object) frame -> typed frame for the pipelines: ints -> Int64,
    doubles -> float64, strings -> pandas string, dates / timestamps kept as
    strings. Values that cannot be converted become missing (the validation
    report already lists them)."""
    schema = TABLES[table]
    out = df.copy()
    for f in schema.fields:
        if f.name not in out.columns:
            continue
        col = out[f.name]
        if f.raw_type in INT_TYPES:
            out[f.name] = pd.to_numeric(col, errors="coerce").round().astype("Int64")
        elif f.raw_type == "double":
            out[f.name] = pd.to_numeric(col, errors="coerce").astype("float64")
        else:
            out[f.name] = col.astype("string")
    return out


# ---------------------------------------------------------------------------
# JSON Schema export (docs/fsdr_json_schemas/<table>.json)
# ---------------------------------------------------------------------------
_JSON_TYPE = {"string": "string", "int": "integer", "bigint": "integer", "tinyint": "integer",
              "double": "number", "date": "string", "timestamp": "string"}


def json_schema(table: str) -> dict:
    s = TABLES[table]
    props = {}
    for f in s.fields:
        p: dict = {"type": _JSON_TYPE[f.raw_type] if f.key else [_JSON_TYPE[f.raw_type], "null"],
                   "x-raw-type": f.raw_type}
        if f.raw_type == "date":
            p["pattern"] = _DATE_RE.pattern
        if f.raw_type == "timestamp":
            p["format"] = "date-time"
        props[f.name] = p
    return {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "$id": f"fsdr/{table}.json",
        "title": table,
        "description": s.description,
        "type": "array",
        "items": {"type": "object", "properties": props, "required": s.field_names, "additionalProperties": True},
        "x-schema-version": config.SCHEMA_VERSION,
        "x-group": s.group,
        "x-folder": s.folder,
        "x-anchor": s.anchor,
        "x-primary-key": list(s.primary_key),
        "x-used-by-model": s.used_by_model,
    }


def export_json_schemas(out_dir: Path | None = None) -> list[Path]:
    out_dir = Path(out_dir or config.JSON_SCHEMA_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for name in TABLES:
        p = out_dir / f"{name}.json"
        p.write_text(json.dumps(json_schema(name), indent=2, ensure_ascii=False), encoding="utf-8")
        paths.append(p)
    index = {"schema_version": config.SCHEMA_VERSION, "generated": date.today().isoformat(),
             "tables": {n: {"group": s.group, "folder": s.folder, "anchor": s.anchor, "fields": len(s.fields), "used_by_model": s.used_by_model}
                        for n, s in TABLES.items()}}
    (out_dir / "_index.json").write_text(json.dumps(index, indent=2), encoding="utf-8")
    return paths


In [ ]:
%%engine nbx.fsdr.datasource
"""Where the FSDR pipelines get their tables.

    JsonSource("data/input/2026-10-22")   a run folder, one file per table (real exports or generator output)
    SyntheticSource(n=5000, ...)          the same tables generated in memory

Both return the same thing for a table: a raw frame (python values, None =
null, schema columns first) exactly as a JSON file holds it, so swapping
dummy data for real data is a path change, not a code change. Use
schemas.coerce() to get typed columns.

Layout (plan §13): the tables sit in the 5 input folders (usage/, models/,
subscription/, catalog/, others/) or, as in V1.0, all in the run folder itself.

Formats: "<table>.json" (array of records; JSON Lines is detected from the
content), "<table>.jsonl", "<table>.csv" (text: numbers are parsed with the
raw type of the contract) or "<table>.parquet".
"""
from __future__ import annotations

import json
import re
from datetime import date
from pathlib import Path
from typing import Protocol

import pandas as pd

from nbx.fsdr import config, schemas


class DataSource(Protocol):
    def tables(self) -> list[str]: ...
    def has(self, table: str) -> bool: ...
    def read(self, table: str) -> pd.DataFrame: ...
    def manifest(self) -> dict: ...


def _raw_frame(records: list[dict], table: str) -> pd.DataFrame:
    """Records -> object frame: schema columns first (in schema order), then any extra fields."""
    schema = schemas.TABLES.get(table)
    # dtype=object: no type inference, so an int column with nulls stays ints (from_records would make 7 -> 7.0)
    df = pd.DataFrame(records, dtype=object) if records else pd.DataFrame()
    df = df.where(pd.notna(df), None) if len(df) else df
    if schema is None:
        return df
    extra = [c for c in df.columns if c not in schema.field_names]
    present = [c for c in schema.field_names if c in df.columns]
    return df[present + extra] if len(df.columns) else pd.DataFrame(columns=schema.field_names)


def compact_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Raw frame -> numeric columns as numpy int64 (no null) / float64 (nulls, or ints mixed with floats); text
    stays object. For the pipelines only, once validation has seen the raw values (review #15: a column of Python
    objects costs ~4x the memory of a numpy column, ~30 GB for a 1.7 M-line run)."""
    out = {}
    for c in df.columns:
        s = df[c]
        kind = pd.api.types.infer_dtype(s, skipna=True) if s.dtype == object else ""
        try:
            if kind == "integer":
                out[c] = s.astype("int64") if s.notna().all() else s.astype("float64")
            elif kind in ("floating", "mixed-integer-float"):
                out[c] = s.astype("float64")
            else:
                out[c] = s
        except (OverflowError, TypeError, ValueError):     # e.g. an int beyond int64: keep the raw values
            out[c] = s
    return pd.DataFrame(out, index=df.index)


def read_json_records(path: Path) -> list[dict]:
    """Reads a JSON array or JSON Lines file (auto-detected from the first non-blank character)."""
    text = Path(path).read_text(encoding="utf-8-sig")
    stripped = text.lstrip()
    if not stripped:
        return []
    if stripped[0] == "[":
        data = json.loads(stripped)
        if not isinstance(data, list):
            raise ValueError(f"{path}: expected a JSON array of records")
        return data
    return [json.loads(line) for line in text.splitlines() if line.strip()]


_INT_TEXT = re.compile(r"^-?\d+(\.0+)?$")


def _conform(df: pd.DataFrame, table: str, from_text: bool) -> pd.DataFrame:
    """CSV / Parquet frame -> the raw values a JSON file would hold: None for nulls, python ints for integer
    fields (a Parquet int column with nulls arrives as float), floats for doubles, ISO strings for dates. CSV
    text that does not parse keeps its string, so validation reports it as a type error."""
    schema = schemas.TABLES.get(table)
    types = schema.types() if schema else {}
    out = {}
    for c in df.columns:
        s = df[c]
        rt = types.get(c)
        if pd.api.types.is_datetime64_any_dtype(s):
            fmt = "%Y-%m-%d" if rt == "date" else "%Y-%m-%dT%H:%M:%S"
            vals = s.dt.strftime(fmt).astype(object)
        elif from_text:
            vals = s.astype(object).where(s.astype(str) != "", None)
            if rt in schemas.INT_TYPES or rt == "double":
                num = pd.to_numeric(vals, errors="coerce")
                ok = num.notna()
                if rt in schemas.INT_TYPES:
                    ok &= vals.astype(str).str.match(_INT_TEXT.pattern)
                    conv = [int(v) for v in num[ok]]
                else:
                    conv = [float(v) for v in num[ok]]
                vals = vals.copy()
                vals[ok] = pd.Series(conv, index=num[ok].index, dtype=object)
        else:
            vals = s.astype(object)
            if rt in schemas.INT_TYPES and pd.api.types.is_float_dtype(s):
                whole = s.notna() & (s % 1 == 0)
                vals = vals.copy()
                vals[whole] = pd.Series([int(v) for v in s[whole]], index=s[whole].index, dtype=object)
            elif pd.api.types.is_numeric_dtype(s):
                vals = pd.Series(s.tolist(), index=s.index, dtype=object)       # numpy scalars -> python
        out[c] = vals.where(pd.notna(vals), None)
    return pd.DataFrame(out, index=df.index, columns=df.columns)


def _frame_raw(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Conformed frame -> schema columns first (in schema order), then any extra fields."""
    schema = schemas.TABLES.get(table)
    if schema is None:
        return df
    extra = [c for c in df.columns if c not in schema.field_names]
    present = [c for c in schema.field_names if c in df.columns]
    return df[present + extra] if len(df.columns) else pd.DataFrame(columns=schema.field_names)


def read_table_file(path: Path, table: str) -> pd.DataFrame:
    """One table file, any supported format -> raw frame."""
    from nbx.fsdr import csvstd

    suffix = csvstd.file_format(path)
    if suffix in (csvstd.SUFFIX, ".csv"):                  # V3.0: .csv.gz (FSDR CSV standard), text in, typed here
        return _frame_raw(_conform(csvstd.read_csv(path), table, from_text=True), table)
    if suffix == ".parquet":
        return _frame_raw(_conform(pd.read_parquet(path), table, from_text=False), table)
    return _raw_frame(read_json_records(path), table)


class JsonSource:
    """A run folder: one file per table (file name = exact table name), in the 5 input folders or flat."""

    def __init__(self, folder: str | Path, compact: bool = False):
        """compact=True: numeric columns typed (compact_frame), for the pipelines; False: raw values, for validation."""
        self.folder = Path(folder)
        if not self.folder.is_dir():
            raise FileNotFoundError(f"No such data folder: {self.folder}")
        self.compact = compact
        self._cache: dict[str, pd.DataFrame] = {}
        self._files = self._index()

    def _index(self) -> dict[str, Path]:
        """Table -> file. Fills found_in (table -> input folder, "" = the run folder itself) and duplicate_files
        (a table found more than once: validation blocks, since nobody can tell which file is the right one)."""
        from nbx.fsdr import csvstd

        # V4.0: data/training/<D> and data/scoring/<D> are read as one run (config.run_twins); single folder otherwise
        self.twins = config.run_twins(self.folder) or {"": self.folder}
        found: dict[str, dict[str, list[tuple[str, Path]]]] = {}
        for side, root in self.twins.items():
            dirs = [(root, "")] + [(root / f, f) for f in config.INPUT_FOLDERS if (root / f).is_dir()]
            for d, sub in dirs:
                for p in sorted(d.iterdir()):
                    if (not p.is_file() or not csvstd.file_format(p) or p.name.startswith("_")
                            or p.name.endswith(csvstd.SCHEMA_SUFFIX)):
                        continue
                    name = csvstd.table_name(p)
                    found.setdefault(schemas.TABLE_ALIASES.get(name, name), {}).setdefault(side, []).append((sub, p))
        files: dict[str, Path] = {}
        self.found_in: dict[str, str] = {}
        self.side_of: dict[str, str] = {}
        self.duplicate_files = {}
        for t, by_side in found.items():
            side = config.table_side(t) if config.table_side(t) in by_side else next(iter(by_side))
            cands = by_side[side]                                                  # a table in both twins: its own side
            if len(cands) > 1:
                self.duplicate_files[t] = [str(p) for _, p in cands]
            home = schemas.TABLES[t].folder if t in schemas.TABLES else None
            sub, p = next(((s, p) for s, p in cands if s == home), cands[0])     # its own folder first
            files[t], self.found_in[t], self.side_of[t] = p, sub, side
        return files

    def misfiled(self) -> dict[str, tuple[str, str]]:
        """V4.0: tables in the wrong twin (a scoring table under training/ or the reverse): table -> (found, expected).
        A warning only: the run still reads them."""
        if set(self.twins) == {""}:
            return {}
        return {t: (s, config.table_side(t)) for t, s in self.side_of.items()
                if t in schemas.TABLES and s != config.table_side(t)}

    @property
    def layout(self) -> str:
        """"folders" (plan §13), "flat" (V1.0: every file in the run folder) or "mixed"."""
        subs = set(self.found_in.values())
        return "flat" if subs <= {""} else "folders" if "" not in subs else "mixed"

    def misplaced(self) -> dict[str, tuple[str, str]]:
        """Contract tables filed in a folder that is not theirs: table -> (found in, expected)."""
        return {t: (sub, schemas.TABLES[t].folder) for t, sub in self.found_in.items()
                if sub and t in schemas.TABLES and sub != schemas.TABLES[t].folder}

    def tables(self) -> list[str]:
        return list(self._files)

    def unknown_files(self) -> list[str]:
        return [t for t in self._files if t not in schemas.TABLES]

    def path(self, table: str) -> Path | None:
        return self._files.get(table)

    def has(self, table: str) -> bool:
        return table in self._files

    def read(self, table: str) -> pd.DataFrame:
        if table not in self._cache:
            if table not in self._files:
                raise FileNotFoundError(f"{table}: no {table}.csv.gz (or a test format) in {self.folder}")
            df = read_table_file(self._files[table], table)
            self._cache[table] = compact_frame(df) if self.compact else df
        return self._cache[table]

    def manifest(self) -> dict:
        for root in [self.folder, *self.twins.values()]:
            p = root / config.MANIFEST_FILE
            if p.exists():
                return json.loads(p.read_text(encoding="utf-8"))
        return {}

    def manifest_files(self) -> list[Path]:
        return [r / config.MANIFEST_FILE for r in dict.fromkeys([self.folder, *self.twins.values()])]

    def compact_table(self, table: str) -> None:
        """Replaces a cached raw table by its typed version (validation, once its raw checks are done)."""
        if table in self._cache:
            self._cache[table] = compact_frame(self._cache[table])


class SyntheticSource:
    """Generates the same tables in memory (no files) -- tests, demos, simulation."""

    def __init__(self, n: int = config.GENERATOR_DEFAULT_N, run_date: date | None = None, scenarios=None,
                 seed: int = config.GENERATOR_SEED, world=None, as_of: date | None = None):
        from nbx.fsdr.synth import generator
        from nbx.fsdr.synth import scenarios as scen
        from nbx.fsdr.synth.world import World

        run_date = run_date or date.today()
        world = world or World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=as_of or run_date)
        self.run = generator.generate_run(world, run_date)
        self._cache: dict[str, pd.DataFrame] = {}

    def tables(self) -> list[str]:
        return list(self.run.tables)

    def has(self, table: str) -> bool:
        return table in self.run.tables

    def read(self, table: str) -> pd.DataFrame:
        if table not in self._cache:
            # same path as JsonSource: records -> raw frame
            self._cache[table] = _raw_frame(self.run.tables[table].to_dict("records"), table)
        return self._cache[table]

    def manifest(self) -> dict:
        return dict(self.run.manifest)

    def write(self, folder: str | Path, fmt: str = "json", layout: str = "folders") -> Path:
        from nbx.fsdr.synth import generator

        return generator.write_run(self.run, Path(folder), fmt, layout)


FolderSource = JsonSource          # V2.0 name: the folder holds JSON, JSONL, CSV or Parquet files


In [ ]:
%%engine nbx.fsdr.features
"""NBA feature engineering -- port of docs/fsdr_reference/nba_lib.py, without Dataiku.

One function, build_features(), is used for BOTH training (T-anchored *_auto
tables) and scoring (D-anchored *_score tables), so the two can never drift.

Fixes kept from nba_lib (README_NBA_pipelines.md):
  - one key type everywhere: ACCESS_METHOD_VAL is a canonical 8-digit string,
    ids are Int64 (the original merged strings with integers);
  - missing model score -> uniform(0, 0.02) drawn PER ROW (BR-47), seeded;
  - trends go from the older period to the recent one, W1 = most recent week,
    % change clipped to [-100, 500] (BR-50..53).

V1.0 (review #11 / #12, FSDR_EXTENDED_FEATURES): PROBA_<x>_MISSING flags (the BR-47 fill hides missingness) and
LOG_TENURE (tenure carried by the model-score files). Off = production's feature list.
"""
from __future__ import annotations

from datetime import date

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import PERIODS
from nbx.fsdr.datasource import DataSource

KEY_ID_COLS = ["ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"]


# ---------------------------------------------------------------------------
# Feature names (derived from the registry / config, never hard-coded)
# ---------------------------------------------------------------------------
def trend_feature_names() -> list[str]:
    names = []
    for k, pairs in config.TREND_PAIRS.items():
        for recent, older in pairs:
            names += [f"VAR_{k}_{recent}_{older}", f"LOG_{k}_{recent}_{older}", f"{k}_{older}_ZERO"]
    return names


def numeric_features(registry: list[config.ModelScore] | None = None, extra: list[str] | tuple = ()) -> list[str]:
    """PROBA_* (one per registry feature) + trends (+ extra): the 48 numeric features of production."""
    return config.proba_features(registry) + trend_feature_names() + list(extra)


def extended_feature_names(registry: list[config.ModelScore] | None = None) -> list[str]:
    return [f"{p}_MISSING" for p in config.proba_features(registry)] + ["LOG_TENURE"]


def model_features(registry: list[config.ModelScore] | None = None) -> list[str]:
    """What the bandit reads: the production features + the last-campaign features (M3) when enabled
    + the V1.0 extended features when enabled."""
    extra = list(config.LAST_CAMPAIGN_NUMERIC if config.LAST_CAMPAIGN_FEATURES else ())
    if config.EXTENDED_FEATURES:
        extra += extended_feature_names(registry)
    return numeric_features(registry, extra)


# ---------------------------------------------------------------------------
# Key normalisation
# ---------------------------------------------------------------------------
def canonical_keys(s: pd.Series, strip_zero: bool, empty_none: bool) -> pd.Series:
    """Key values -> canonical strings, vectorised per Python type (a column holds one or two types):
    ints -> str(int); floats -> str(int) when integral, else str; NaN / None -> None; strings stripped
    (strip_zero: '91234567.0' -> '91234567'; empty_none: '' -> None). Same rules as the per-value functions it
    replaces (review #15: they ran once per value, 15-45 M calls at 100 k lines)."""
    out = np.full(len(s), None, dtype=object)       # numpy, not a Series: masked Series assignment turns None into NaN
    if not len(s):
        return pd.Series(out, index=s.index, dtype=object)
    if s.dtype.kind in "iu":
        return s.astype(str).astype(object)
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind == "string" or (kind == "empty" and s.dtype == object):
        groups = [(str, s.notna().to_numpy())]
    elif s.dtype.kind == "f" or kind == "floating":
        groups = [(float, s.notna().to_numpy())]
    elif kind == "integer":
        groups = [(int, s.notna().to_numpy())]
    else:
        types = s.map(type)
        groups = [(t, (types == t).to_numpy()) for t in types.unique() if t is not type(None)]
    for t, m in groups:
        if not m.any():
            continue
        sub = s[m]
        if issubclass(t, str):
            x = sub.str.strip()
            if strip_zero:
                drop = x.str.endswith(".0") & x.str[:-2].str.isdigit()
                x = x.where(~drop, x.str[:-2])
            if empty_none:
                x = x.where(x != "", None)
            out[m] = x.to_numpy(dtype=object)
        elif issubclass(t, (bool, int, np.integer)):
            out[m] = [str(int(v)) for v in sub]
        elif issubclass(t, (float, np.floating)):
            arr = sub.to_numpy(dtype=float)
            nan = np.isnan(arr)
            integral = ~nan & np.isfinite(arr) & (np.mod(arr, 1) == 0)
            vals = np.empty(len(arr), dtype=object)
            vals[nan] = None
            vals[integral] = [str(int(v)) for v in arr[integral]]
            rest = ~nan & ~integral
            vals[rest] = [str(v) for v in sub.to_numpy(dtype=object)[rest]]
            out[m] = vals
        else:
            out[m] = [_canonical_one(v, strip_zero, empty_none) for v in sub]
    return pd.Series(out, index=s.index, dtype=object)


def _canonical_one(v, strip_zero: bool, empty_none: bool):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return str(int(v)) if float(v).is_integer() else str(v)
    out = str(v).strip()
    if strip_zero and out.endswith(".0") and out[:-2].isdigit():
        out = out[:-2]
    return (out or None) if empty_none else out


def norm_val(s: pd.Series) -> pd.Series:
    """MSISDN -> canonical string: 91234567, 91234567.0 and '91234567' all become '91234567'."""
    return canonical_keys(s, strip_zero=True, empty_none=True)


def _norm_val_per_value(s: pd.Series) -> pd.Series:
    """Reference implementation of norm_val (one value at a time), kept for the equivalence test."""
    def one(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        if isinstance(v, (int, np.integer)):
            out = str(int(v))
        elif isinstance(v, (float, np.floating)):
            out = str(int(v)) if float(v).is_integer() else str(v)
        else:
            out = str(v).strip()
            if out.endswith(".0") and out[:-2].isdigit():
                out = out[:-2]
        return out or None
    return s.map(one).astype(object)


def norm_id(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")


def upper(df: pd.DataFrame) -> pd.DataFrame:
    """Upper-case column names (production tables mix cases) and normalise the join keys."""
    out = df.copy()
    out.columns = [str(c).upper() for c in out.columns]
    if "ACCESS_METHOD_VAL" in out.columns:
        out["ACCESS_METHOD_VAL"] = norm_val(out["ACCESS_METHOD_VAL"])
    for c in KEY_ID_COLS:
        if c in out.columns:
            out[c] = norm_id(out[c])
    return out


def _numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64") if c in df.columns else 0.0
    return df


# ---------------------------------------------------------------------------
# Loaders (each returns one row per ACCESS_METHOD_VAL)
# ---------------------------------------------------------------------------
def load_party(src: DataSource, table: str) -> pd.DataFrame:
    df = upper(src.read(table))
    return df[["PARTY_ID", "NATIONALITY"]].dropna(subset=["PARTY_ID"]).drop_duplicates("PARTY_ID")


def load_model_scores(src: DataSource, registry: list[config.ModelScore] | None = None) -> tuple[pd.DataFrame, dict]:
    """One row per line with one column per registry feature. Rows of the same
    feature (e.g. the 3 churn files) and stacked duplicates are merged by max."""
    registry = registry or config.MODEL_SCORE_REGISTRY
    per_feature: dict[str, list[pd.DataFrame]] = {}
    coverage: dict[str, int] = {}
    for m in registry:
        if not src.has(m.table):
            coverage[m.table] = 0
            continue
        df = upper(src.read(m.table))
        df = df[["ACCESS_METHOD_VAL", "PROBA_1"]].dropna(subset=["ACCESS_METHOD_VAL"])
        df["PROBA_1"] = pd.to_numeric(df["PROBA_1"], errors="coerce")
        coverage[m.table] = int(df["ACCESS_METHOD_VAL"].nunique())
        per_feature.setdefault(m.feature, []).append(df.rename(columns={"PROBA_1": m.feature}))
    out: pd.DataFrame | None = None
    for feature, frames in per_feature.items():
        agg = "max" if all(m.combine == "max" for m in registry if m.feature == feature) else "mean"
        f = pd.concat(frames, ignore_index=True).groupby("ACCESS_METHOD_VAL", as_index=False)[feature].agg(agg)
        out = f if out is None else out.merge(f, on="ACCESS_METHOD_VAL", how="outer")
    if out is None:
        out = pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    return out, coverage


def load_tenure(src: DataSource, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """Tenure (days) per line from the model-score files (they share one structure with a tenure field), max."""
    frames = []
    for m in registry or config.MODEL_SCORE_REGISTRY:
        if src.has(m.table):
            df = upper(src.read(m.table))
            if "TENURE" in df.columns:
                frames.append(pd.DataFrame({"ACCESS_METHOD_VAL": df["ACCESS_METHOD_VAL"],
                                            "TENURE": pd.to_numeric(df["TENURE"], errors="coerce")}))
    if not frames:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL", "TENURE"])
    return pd.concat(frames).dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL", as_index=False)["TENURE"].max()


def load_usage(src: DataSource, anchor: str) -> pd.DataFrame:
    """anchor 'T' -> activities_*_auto (training), 'D' -> activities_*_score (scoring)."""
    out: pd.DataFrame | None = None
    for p in PERIODS:
        table = schemas.activity_table_name(p, anchor)
        if not src.has(table):
            continue
        df = upper(src.read(table))
        cols = [f"{u}_{p}" for u in config.USAGE_COLS if f"{u}_{p}" in df.columns]
        df = _numeric(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
        df = df.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()
        out = df if out is None else out.merge(df, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def load_subs(src: DataSource, anchor: str) -> pd.DataFrame:
    table = "subs_amt_auto" if anchor == "T" else "subs_amt_score"
    if not src.has(table):
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    df = upper(src.read(table))
    cols = [f"SUB_AMT_{p}" for p in PERIODS if f"SUB_AMT_{p}" in df.columns]
    df = _numeric(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
    return df.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()


# ---------------------------------------------------------------------------
# Eligible base (BR-02, BR-08, BR-10)
# ---------------------------------------------------------------------------
def eligible_base(src: DataSource) -> tuple[pd.DataFrame, dict]:
    return eligible_frame(upper(src.read("base_prepaid")))


def eligible_frame(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """BR-02 / BR-08 / BR-10 on an upper-cased base_prepaid frame (the whole base, or one cache partition)."""
    n0 = len(df)
    df = df.dropna(subset=["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID"])
    null_keys = n0 - len(df)
    val = df["ACCESS_METHOD_VAL"].astype(str)
    ok_msisdn = val.str.fullmatch(r"\d{8}") & (val > config.MSISDN_MIN_EXCLUSIVE)        # BR-02
    excluded_plan = df["CURRENT_PLAN"].isin(config.EXCLUDED_PLANS)                       # BR-10
    df = df[ok_msisdn & ~excluded_plan]
    before = len(df)
    df = df.sort_values("ACCESS_METHOD_SUBS_START_DT").drop_duplicates("ACCESS_METHOD_VAL", keep="last")   # BR-08
    report = {"base_rows": n0, "eligible": len(df), "null_keys": null_keys, "excluded_plan": int(excluded_plan.sum()),
              "bad_msisdn": int((~ok_msisdn).sum()), "duplicates_removed": before - len(df)}
    return df.reset_index(drop=True), report


def nationality_groups(src: DataSource) -> pd.Series:
    """ACCESS_METHOD_VAL -> nationality group (BR-15), for the fairness report (review #7)."""
    base = upper(src.read("base_prepaid"))[["ACCESS_METHOD_VAL", "PARTY_ID"]].dropna()
    party = load_party(src, "party_scoring")
    df = base.merge(party, on="PARTY_ID", how="left").drop_duplicates("ACCESS_METHOD_VAL")
    grp = np.where(df["NATIONALITY"].isin(config.NATIONALITY_KEEP), df["NATIONALITY"], config.NATIONALITY_OTHERS)
    return pd.Series(grp, index=df["ACCESS_METHOD_VAL"].astype(str).to_numpy())


def consent_blocked(src: DataSource) -> set[str]:
    """V1.0 review #6: lines that must never be contacted (dnd = 1 or marketing_opt_in = 0 in consent_contact).
    No table = nothing blocked (validation warns that the consent check could not run)."""
    if not src.has(config.CONSENT_TABLE):
        return set()
    df = upper(src.read(config.CONSENT_TABLE))
    dnd = pd.to_numeric(df.get("DND"), errors="coerce").fillna(0) == 1
    opt_out = pd.to_numeric(df.get("MARKETING_OPT_IN"), errors="coerce").fillna(1) == 0
    return set(df.loc[dnd | opt_out, "ACCESS_METHOD_VAL"].dropna().astype(str))


# ---------------------------------------------------------------------------
# Feature engineering (identical for training and scoring)
# ---------------------------------------------------------------------------
def build_features(pop: pd.DataFrame, party: pd.DataFrame, scores: pd.DataFrame, usage: pd.DataFrame, subs: pd.DataFrame,
                   seed: int = config.GENERATOR_SEED, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """pop: one row per line with at least ACCESS_METHOD_VAL and PARTY_ID."""
    proba_cols = config.proba_features(registry)
    df = pop.drop(columns=[c for c in ["NATIONALITY"] if c in pop.columns])
    df = df.dropna(subset=["ACCESS_METHOD_VAL"]).drop_duplicates("ACCESS_METHOD_VAL").copy()
    df = (df.merge(party, on="PARTY_ID", how="left")
            .merge(scores, on="ACCESS_METHOD_VAL", how="left")
            .merge(usage, on="ACCESS_METHOD_VAL", how="left")
            .merge(subs, on="ACCESS_METHOD_VAL", how="left"))

    # BR-47: missing model scores -> small random value, drawn PER ROW
    rng = np.random.default_rng(seed)
    lo, hi = config.MISSING_SCORE_RANGE
    for c in proba_cols:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
        miss = df[c].isna()
        if config.EXTENDED_FEATURES:
            df[f"{c}_MISSING"] = miss.astype(int)
        df.loc[miss, c] = rng.uniform(lo, hi, int(miss.sum()))

    # BR-48 / BR-49: missing usage = 0; NBA revenue = subscriptions + PAYG data
    for p in PERIODS:
        for c in [f"SUB_AMT_{p}", f"PG_DATA_AMOUNT_{p}", f"LOCAL_VOICE_MIN_{p}", f"DATA_VOLUME_MB_{p}"]:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype("float64") if c in df.columns else 0.0
        df[f"REV_{p}"] = df[f"SUB_AMT_{p}"] + df[f"PG_DATA_AMOUNT_{p}"]

    add_trends(df)

    # BR-15 nationality group
    df["NATIONALITY_GROUP"] = np.where(df["NATIONALITY"].isin(config.NATIONALITY_KEEP), df["NATIONALITY"], config.NATIONALITY_OTHERS)
    return df.reset_index(drop=True)


def add_trends(df: pd.DataFrame) -> pd.DataFrame:
    """BR-50..53 (V2.0 B-07): % change, log change and "older period was zero" flag, from older to recent, in place."""
    for k, pairs in config.TREND_PAIRS.items():
        for recent, older in pairs:
            cur = df[config.TREND_SOURCE[k].format(p=recent)]
            prev = df[config.TREND_SOURCE[k].format(p=older)]
            zero = prev == 0
            df[f"{k}_{older}_ZERO"] = zero.astype(int)
            pct = np.where(zero, 0.0, (cur - prev) / prev.where(~zero, 1.0) * 100.0)
            df[f"VAR_{k}_{recent}_{older}"] = np.clip(pct, *config.PCT_CLIP)
            df[f"LOG_{k}_{recent}_{older}"] = np.log1p(cur.clip(lower=0)) - np.log1p(prev.clip(lower=0))
    return df


def build_feature_frame(src: DataSource, pop: pd.DataFrame, anchor: str, seed: int = config.GENERATOR_SEED,
                        registry: list[config.ModelScore] | None = None, log_root=None,
                        context_date=None) -> tuple[pd.DataFrame, dict]:
    """Loads every input for `anchor` ('T' training, 'D' scoring) and builds the features of `pop`.
    log_root / context_date: campaign log and the decision date of the last-campaign features (M3);
    without a log every line gets the "never contacted" values."""
    party = load_party(src, "party_auto" if anchor == "T" else "party_scoring")
    scores, coverage = load_model_scores(src, registry)
    feat = build_features(pop, party, scores, load_usage(src, anchor), load_subs(src, anchor), seed, registry)
    if config.EXTENDED_FEATURES:
        ten = load_tenure(src, registry)
        feat = feat.drop(columns=[c for c in ("TENURE", "LOG_TENURE") if c in feat.columns]).merge(ten, on="ACCESS_METHOD_VAL", how="left")
        feat["LOG_TENURE"] = np.log1p(feat["TENURE"].clip(lower=0)).fillna(0.0)
    if config.LAST_CAMPAIGN_FEATURES:
        from nbx.fsdr import campaign_log
        run_date = src.manifest().get("run_date")
        when = context_date or (date.fromisoformat(run_date) if run_date else date.today())
        lc = campaign_log.last_campaign_features(log_root, feat["ACCESS_METHOD_VAL"], when)
        feat = feat.drop(columns=[c for c in lc.columns if c != "ACCESS_METHOD_VAL" and c in feat.columns])
        feat = feat.merge(lc, on="ACCESS_METHOD_VAL", how="left")
    pop_vals = set(pop["ACCESS_METHOD_VAL"].dropna())
    scored = set(scores["ACCESS_METHOD_VAL"].dropna()) if len(scores) else set()
    report = {"lines": len(feat), "anchor": anchor, "numeric_features": len(numeric_features(registry)), "model_features": len(model_features(registry)),
              "score_coverage": round(len(pop_vals & scored) / len(pop_vals), 4) if pop_vals else 0.0,
              "model_rows": coverage}
    return feat, report


In [ ]:
%%engine nbx.fsdr.base_features
"""T2_features (master prompt §6, B-01 … B-12): the features of the full base, partition by partition.

    df, stats = partition_features(CacheSource(D), part=3, run_date=D, anchor="T")   # raw: scores may be NaN
    medians = score_medians(paths)                                                   # B-05, on the whole base
    impute(df, medians)

Every line of one cache partition is in the same partition of every table (contract/cache.py), so a partition is
built with joins inside it only, and 8 partitions run in parallel on the P16 (plan §15). Two passes, because B-05
imputes a missing score with the median of the nationality group over the WHOLE base:
  pass 1  per partition: population, joins, revenue, trends, profile        -> base_features/part-NN.parquet
  between the score medians per nationality group and the ARPU tertiles are computed from those files
  pass 2  per partition: impute the missing scores, keep the *_MISSING flags -> the same file

What changes from V1.0's features.py (the rules of code 1 are the same):
  B-04  the 3 churn tables are stacked in registry order and the FIRST value per line is kept (V1.0: max)
  B-05  missing score -> flag + nationality-group median (V1.0: uniform(0, 0.02) draw, documented as replaced)
        A missing score never triggers "at risk": the S-rules read the flags (segmentation/rules.py)
  §8    AGE and TENURE_DAYS come from customer_profile when present, else party AGE and run date - activation
"""
from __future__ import annotations

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import PERIODS
from nbx.fsdr.features import add_trends, eligible_frame, norm_id, norm_val, upper

USAGE_V2 = ["LOCAL_VOICE_MIN", "DATA_VOLUME_MB", "PG_DATA_AMOUNT", "PG_TOTAL_AMOUNT", "ACTIVE_DAYS_COUNT"]
KEYS = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"]
_REFERENCE: dict = {}                              # reference tables, read once per worker process


def anchor_of(pipeline: str) -> str:
    """B-01: week 0 trains on the T-anchored *_auto tables, every later week on the D-anchored *_score tables."""
    return "T" if pipeline == "train_week0" else "D"


def party_table(anchor: str, cache) -> str:
    first, second = ("party_auto", "party_scoring") if anchor == "T" else ("party_scoring", "party_auto")
    return first if cache.has(first) else second


def _part(cache, table: str, part: int) -> pd.DataFrame | None:
    return upper(cache.read(table, part)) if cache.has(table) else None


def _reference(cache, table: str) -> pd.DataFrame | None:
    key = (str(cache.dir), table)
    if key not in _REFERENCE:
        _REFERENCE[key] = upper(cache.read(table)) if cache.has(table) else None
    return _REFERENCE[key]


def _num(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    return df


def model_scores(cache, part: int, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """One row per line, one column per score feature (NaN = not scored). B-04: churn = first row of the stack."""
    per_feature: dict[str, list[pd.DataFrame]] = {}
    for m in registry or config.MODEL_SCORE_REGISTRY:
        df = _part(cache, m.table, part)
        if df is None or "PROBA_1" not in df.columns:
            continue
        d = pd.DataFrame({"ACCESS_METHOD_VAL": df["ACCESS_METHOD_VAL"],
                          m.feature: pd.to_numeric(df["PROBA_1"], errors="coerce")}).dropna()
        per_feature.setdefault(m.feature, []).append(d)
    out: pd.DataFrame | None = None
    for feature, frames in per_feature.items():
        st = pd.concat(frames, ignore_index=True)
        if feature == config.CHURN_FEATURE:
            st = st.drop_duplicates("ACCESS_METHOD_VAL", keep="first")
        else:
            st = st.groupby("ACCESS_METHOD_VAL", as_index=False)[feature].max()
        out = st if out is None else out.merge(st, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def usage(cache, part: int, anchor: str) -> pd.DataFrame:
    out: pd.DataFrame | None = None
    for p in PERIODS:
        df = _part(cache, schemas.activity_table_name(p, anchor), part)
        if df is None:
            continue
        cols = [f"{u}_{p}" for u in USAGE_V2 if f"{u}_{p}" in df.columns]
        d = _num(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
        d = d.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()
        out = d if out is None else out.merge(d, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def subscriptions(cache, part: int, anchor: str) -> pd.DataFrame:
    df = _part(cache, "subs_amt_auto" if anchor == "T" else "subs_amt_score", part)
    if df is None:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    cols = [f"SUB_AMT_{p}" for p in PERIODS if f"SUB_AMT_{p}" in df.columns]
    d = _num(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
    return d.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()


def profile(cache, part: int) -> pd.DataFrame:
    df = _part(cache, config.PROFILE_TABLE, part)
    if df is None or "MSISDN" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL", "AGE_PROFILE", "TENURE_PROFILE"])
    d = pd.DataFrame({"ACCESS_METHOD_VAL": norm_val(df["MSISDN"]),
                      "AGE_PROFILE": pd.to_numeric(df.get("AGE"), errors="coerce"),
                      "TENURE_PROFILE": pd.to_numeric(df.get("TENURE"), errors="coerce")})
    return d.dropna(subset=["ACCESS_METHOD_VAL"]).drop_duplicates("ACCESS_METHOD_VAL", keep="last")


def region(cache, part: int) -> pd.DataFrame:
    df = _part(cache, "location_auto", part)
    if df is None or "REGION" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_ID", "REGION"])
    d = df.assign(_Y=pd.to_numeric(df.get("CALLING_YEAR"), errors="coerce"),
                  _M=pd.to_numeric(df.get("CALLING_MONTH"), errors="coerce"))
    d = d.dropna(subset=["ACCESS_METHOD_ID"]).sort_values(["_Y", "_M"]).drop_duplicates("ACCESS_METHOD_ID", keep="last")
    return d[["ACCESS_METHOD_ID", "REGION"]]


def device(cache, part: int) -> pd.DataFrame:
    df = _part(cache, "DEVICEBEHAVIORSEGMENTATION.dv_segment", part)
    if df is None or "DV_SEGMENT" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_ID", "DV_SEGMENT"])
    return df[["ACCESS_METHOD_ID", "DV_SEGMENT"]].dropna(subset=["ACCESS_METHOD_ID"]).drop_duplicates("ACCESS_METHOD_ID", keep="last")


def partition_features(cache, part: int, run_date: date, anchor: str,
                       registry: list[config.ModelScore] | None = None) -> tuple[pd.DataFrame, dict]:
    """Pass 1 of one partition: B-01 population and left joins in code-1 order, B-05 (usage), B-06, B-07, B-08,
    profile. Scores stay NaN where missing (pass 2 imputes them)."""
    proba = config.proba_features(registry)
    base = _part(cache, "base_prepaid", part)
    if base is None:
        raise FileNotFoundError("base_prepaid is not in the cache")
    base, elig = eligible_frame(base)
    df = base[[c for c in KEYS + ["ACCESS_METHOD_SUBS_START_DT", "CURRENT_PLAN"] if c in base.columns]].copy()

    accounts = _reference(cache, "consumer_aact_account")             # B-01: party through the account if missing
    if accounts is not None and {"ACCOUNT_ID", "PARTY_ID"} <= set(accounts.columns) and df["PARTY_ID"].isna().any():
        acc = accounts.dropna(subset=["ACCOUNT_ID"]).drop_duplicates("ACCOUNT_ID").set_index("ACCOUNT_ID")["PARTY_ID"]
        df["PARTY_ID"] = df["PARTY_ID"].fillna(df["ACCOUNT_ID"].map(acc))
    party = _reference(cache, party_table(anchor, cache))
    if party is not None:
        pty = party[[c for c in ("PARTY_ID", "NATIONALITY", "AGE") if c in party.columns]]
        pty = pty.dropna(subset=["PARTY_ID"]).drop_duplicates("PARTY_ID").rename(columns={"AGE": "AGE_PARTY"})
        df = df.merge(pty, on="PARTY_ID", how="left")
    for c in ("NATIONALITY", "AGE_PARTY"):
        if c not in df.columns:
            df[c] = np.nan

    df = (df.merge(model_scores(cache, part, registry), on="ACCESS_METHOD_VAL", how="left")
            .merge(usage(cache, part, anchor), on="ACCESS_METHOD_VAL", how="left")
            .merge(subscriptions(cache, part, anchor), on="ACCESS_METHOD_VAL", how="left")
            .merge(profile(cache, part), on="ACCESS_METHOD_VAL", how="left")
            .merge(region(cache, part), on="ACCESS_METHOD_ID", how="left")
            .merge(device(cache, part), on="ACCESS_METHOD_ID", how="left"))

    cols: dict[str, pd.Series] = {}
    for f in proba:                                                    # B-05 flags (imputed in pass 2)
        s = pd.to_numeric(df[f], errors="coerce").astype("float64") if f in df.columns else pd.Series(np.nan, index=df.index)
        cols[f] = s
        cols[f"{f}{config.MISSING_FLAG_SUFFIX}"] = s.isna().astype("int8")
    for p in PERIODS:                                                  # B-05 missing usage / spend = 0
        for c in [f"SUB_AMT_{p}"] + [f"{u}_{p}" for u in USAGE_V2]:
            cols[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype("float64") if c in df.columns \
                else pd.Series(0.0, index=df.index)
        cols[f"REV_{p}"] = cols[f"SUB_AMT_{p}"] + cols[f"PG_DATA_AMOUNT_{p}"]       # B-06
    df = pd.concat([df.drop(columns=[c for c in cols if c in df.columns]), pd.DataFrame(cols, index=df.index)], axis=1)
    trends = add_trends(pd.DataFrame(index=df.index).assign(**{c: df[c] for c in df.columns
                                                                  if c.startswith(("REV_", "LOCAL_VOICE_MIN_", "DATA_VOLUME_MB_"))}))
    df = pd.concat([df, trends.drop(columns=[c for c in trends.columns if c in df.columns])], axis=1)   # B-07

    nat = df["NATIONALITY"].astype("string").str.strip()
    df["NATIONALITY_GROUP"] = np.where(nat.isin(config.NATIONALITY_KEEP), nat, config.NATIONALITY_OTHERS)   # B-08
    df["ARPU_M1"] = df["SUB_AMT_M1"] + df["PG_TOTAL_AMOUNT_M1"]          # §7 REV_M1 of the S-rules (and value band)

    age_p = pd.to_numeric(df["AGE_PROFILE"], errors="coerce")           # §8 sources
    df["AGE"] = age_p.fillna(pd.to_numeric(df["AGE_PARTY"], errors="coerce"))
    df["AGE_SOURCE"] = np.where(age_p.notna(), "profile", np.where(df["AGE"].notna(), "party", "missing"))
    start = pd.to_datetime(df["ACCESS_METHOD_SUBS_START_DT"], errors="coerce")
    by_activation = (pd.Timestamp(run_date) - start).dt.days.astype("float64")
    ten_p = pd.to_numeric(df["TENURE_PROFILE"], errors="coerce")
    df["TENURE_DAYS"] = ten_p.fillna(by_activation)
    df["TENURE_SOURCE"] = np.where(ten_p.notna(), "profile", np.where(by_activation.notna(), "activation", "missing"))
    df = df.drop(columns=["AGE_PROFILE", "TENURE_PROFILE"])

    stats = {"part": part, **elig, "lines": len(df),
             "missing_scores": {f: int(df[f"{f}{config.MISSING_FLAG_SUFFIX}"].sum()) for f in proba},
             "age_source": df["AGE_SOURCE"].value_counts().to_dict(),
             "tenure_source": df["TENURE_SOURCE"].value_counts().to_dict(),
             "no_nationality": int(nat.isna().sum()), "no_region": int(df["REGION"].isna().sum()),
             "no_device": int(df["DV_SEGMENT"].isna().sum())}
    return df.reset_index(drop=True), stats


def score_medians(paths: list[Path], registry: list[config.ModelScore] | None = None) -> dict:
    """B-05 medians of the observed scores per nationality group, over every partition (+ "_global")."""
    proba = config.proba_features(registry)
    frames = [pd.read_parquet(p, columns=["NATIONALITY_GROUP"] + proba) for p in paths]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["NATIONALITY_GROUP"] + proba)
    med = {g: {f: (None if pd.isna(v) else round(float(v), 6)) for f, v in row.items()}
           for g, row in df.groupby("NATIONALITY_GROUP")[proba].median().iterrows()}
    med["_global"] = {f: (None if pd.isna(df[f].median()) else round(float(df[f].median()), 6)) for f in proba}
    return med


def arpu_cutoffs(paths: list[Path]) -> list[float]:
    """Tertile cut points of ARPU_M1 among the paying lines (value band, context only: §8)."""
    arpu = pd.concat([pd.read_parquet(p, columns=["ARPU_M1"])["ARPU_M1"] for p in paths], ignore_index=True)
    paying = arpu[arpu > 0]
    return [round(float(q), 4) for q in paying.quantile([1 / 3, 2 / 3])] if len(paying) else [0.0, 0.0]


def impute(df: pd.DataFrame, medians: dict, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """Pass 2 (B-05): every missing score = the median of its nationality group (else global, else 0). In place."""
    glob = medians.get("_global", {})
    for f in config.proba_features(registry):
        miss = df[f].isna()
        if miss.any():
            by_group = {g: m.get(f) for g, m in medians.items() if g != "_global"}
            fill = df.loc[miss, "NATIONALITY_GROUP"].map(by_group).astype("float64")
            df.loc[miss, f] = fill.fillna(glob.get(f) if glob.get(f) is not None else 0.0).fillna(0.0)
    return df


def column_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Per numeric column: count, nulls, sum, min, max (summed over partitions into feature_summary.csv)."""
    num = df.select_dtypes("number")
    return pd.DataFrame({"count": num.count(), "nulls": num.isna().sum(), "sum": num.sum(), "min": num.min(),
                         "max": num.max()})


In [ ]:
%%engine nbx.fsdr.campaign_log
"""Campaign log and rewards: what was sent to whom, and what came back (prompt §6, audit M1..M4).

    nba_campaign_log.jsonl   one row per (run_date, arm, line): action, offers (code, price, validity),
                             policy_source, propensity, bandit PMF, model version, LLM model,
                             justification id, the line's ARPU anchors. Append-only.
    nba_rewards.jsonl        one row per logged line once the reward window D+7 is known: the three reward
                             definitions (any_btl / same_service / exact_offer), revenue, last BTL date.
                             Append-only.

Both files live in <data root>/state/campaign/, next to the dated run folders, so one log follows the
weekly cycle. Appending never rewrites a row: a key (run_date, arm, msisdn) already present is skipped,
so re-running a step is safe.

Uses: training labels with the logged propensity (H5), last-campaign features (M3), the 28-day contact
cap (L4), the anchor growth guard (M4), and the weekly report per arm.

V1.0 (review #15): the parsed rows and the key set of each file are cached on (size, mtime), and an append only
parses the bytes added since, so appending is no longer a full re-read of a growing file (quadratic over time).
"""
from __future__ import annotations

import hashlib
import json
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config

LOG_FILE = "nba_campaign_log.jsonl"
REWARDS_FILE = "nba_rewards.jsonl"
KEY = ("run_date", "arm", "msisdn")
LOG_FIELDS = ["run_date", "arm", "msisdn", "action", "offers", "policy_source", "propensity", "bandit_pmf",
              "model_version", "llm_model", "justification_id", "anchors", "logged_at"]


def root_of(data_root: str | Path | None = None) -> Path:
    """<data root>/state/campaign (default data root: config.FSDR_DATA_DIR)."""
    return Path(data_root or config.FSDR_DATA_DIR).resolve() / config.FSDR_STATE_DIR.name / config.CAMPAIGN_DIR


def root_for(data_dir: str | Path) -> Path:
    """The log of the run folder <data root>/<D>."""
    return root_of(Path(data_dir).resolve().parent)


_CACHE: dict[str, dict] = {}          # resolved path -> {"size", "mtime", "rows", "keys"}


def _key(r: dict) -> tuple:
    return tuple(str(r.get(k)) for k in KEY)


def _load(path: Path) -> dict:
    """Rows + keys of a JSONL file; re-parses only the bytes appended since the cached state."""
    p = str(path.resolve())
    if not path.exists():
        _CACHE.pop(p, None)
        return {"size": 0, "mtime": 0, "rows": [], "keys": set()}
    st = path.stat()
    c = _CACHE.get(p)
    if c and c["size"] == st.st_size and c["mtime"] == st.st_mtime_ns:
        return c
    start = c["size"] if c and st.st_size > c["size"] else 0       # grown: read the tail; else (rewritten) all
    rows, keys = (list(c["rows"]), set(c["keys"])) if start else ([], set())
    with path.open("rb") as fh:
        fh.seek(start)
        for line in fh.read().decode("utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                rows.append(r)
                keys.add(_key(r))
    c = _CACHE[p] = {"size": st.st_size, "mtime": st.st_mtime_ns, "rows": rows, "keys": keys}
    return c


def _read_jsonl(path: Path) -> list[dict]:
    return list(_load(path)["rows"])


def _append(path: Path, rows: list[dict]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    seen = set(_load(path)["keys"])
    new = []
    for r in rows:
        k = _key(r)
        if k not in seen:
            seen.add(k)
            new.append(r)
    with path.open("a", encoding="utf-8") as fh:
        for r in new:
            fh.write(json.dumps(r, ensure_ascii=False, default=_json_default) + "\n")
    return len(new)


def _json_default(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return round(float(v), 6)
    if isinstance(v, (date, datetime)):
        return v.isoformat()
    return str(v)


def append_log(root: Path, rows: list[dict]) -> int:
    """Adds campaign rows (missing fields become null); returns how many were new."""
    stamp = datetime.now().isoformat(timespec="seconds")
    return _append(Path(root) / LOG_FILE, [{**{f: None for f in LOG_FIELDS}, "logged_at": stamp, **r} for r in rows])


def append_rewards(root: Path, rows: list[dict]) -> int:
    return _append(Path(root) / REWARDS_FILE, rows)


def read_log(root: Path, start: date | None = None, end: date | None = None, arm: str | None = None) -> pd.DataFrame:
    df = pd.DataFrame(_read_jsonl(Path(root) / LOG_FILE), columns=LOG_FIELDS)
    return _filter(df, start, end, arm)


def read_rewards(root: Path, start: date | None = None, end: date | None = None, arm: str | None = None) -> pd.DataFrame:
    df = pd.DataFrame(_read_jsonl(Path(root) / REWARDS_FILE))
    return _filter(df, start, end, arm) if len(df) else df


def _filter(df: pd.DataFrame, start, end, arm) -> pd.DataFrame:
    if not len(df):
        return df
    d = pd.to_datetime(df["run_date"]).dt.date
    keep = pd.Series(True, index=df.index)
    if start:
        keep &= d >= start
    if end:
        keep &= d <= end
    if arm:
        keep &= df["arm"] == arm
    return df[keep].reset_index(drop=True)


def justification_id(*parts) -> str:
    return hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:12]


# ---------------------------------------------------------------------------
# Rows from each arm
# ---------------------------------------------------------------------------
def _pmf(row: pd.Series) -> dict:
    from nbx.core.vw_format import action_token
    return {a: round(float(row.get(f"P_{action_token(a)}", np.nan)), 6) for a in config.ACTIONS
            if pd.notna(row.get(f"P_{action_token(a)}", np.nan))}


def _anchors(anchors: pd.DataFrame | None, val: str) -> dict | None:
    if anchors is None or val not in anchors.index:
        return None
    return {"monthly": round(float(anchors.at[val, "AVG_REV_MONTHLY"]), 6), "weekly": round(float(anchors.at[val, "AVG_REV_WEEKLY"]), 6)}


def rows_from_recos(recos: pd.DataFrame, scored: pd.DataFrame, arm: str, run_date: date,
                    anchors: pd.DataFrame | None = None) -> list[dict]:
    """cstr: the bandit's sampled action and its PMF probability; free: the ranker picks the offers (BR-60),
    the scored action is logged with its probability so the row stays usable for off-policy learning."""
    flag = {"free": "free", "cstr": "with_constraint"}[arm]
    pop = scored[scored["TG_FLAG"] == flag].copy()
    pop["ACCESS_METHOD_VAL"] = pop["ACCESS_METHOD_VAL"].astype(str)
    by_val = {v: g for v, g in recos.groupby("ACCESS_METHOD_VAL")} if len(recos) else {}
    rows = []
    for r in pop.itertuples(index=False):
        s = pd.Series(r._asdict())
        val = s["ACCESS_METHOD_VAL"]
        g = by_val.get(val)
        offers = [] if g is None else [{"code": o.BTL_NAME, "price": float(o.PRICE), "validity": int(o.VALIDITY)}
                                       for o in g.itertuples(index=False)]
        rows.append({"run_date": run_date.isoformat(), "arm": arm, "msisdn": val, "action": s["ACTION"], "offers": offers,
                     "policy_source": "bandit" if arm == "cstr" else "ranker", "propensity": round(float(s["PROBA"]), 6),
                     "bandit_pmf": _pmf(s), "model_version": s.get("MODEL_VERSION"), "llm_model": None,
                     "justification_id": None, "anchors": _anchors(anchors, val)})
    return rows


def rows_from_decisions(decisions: list[dict], anchors: pd.DataFrame | None = None) -> list[dict]:
    """llm arm: one row per agent decision (recommender_agent.decide)."""
    rows = []
    for d in decisions:
        just = d.get("justification") or {}
        rows.append({"run_date": d["run_date"], "arm": "llm", "msisdn": str(d["msisdn"]), "action": d["action"],
                     "offers": [{"code": o["offer"], "price": float(o["price"]), "validity": int(o["validity"])} for o in d.get("offers", [])],
                     "policy_source": d["policy_source"], "propensity": d["propensity"], "bandit_pmf": d.get("bandit_pmf"),
                     "model_version": d.get("model_version"), "llm_model": d.get("model"),
                     "justification_id": justification_id(d["msisdn"], d["run_date"], json.dumps(just, sort_keys=True)),
                     "anchors": _anchors(anchors, str(d["msisdn"]))})
    return rows


def rows_control(sample_vals: list[str], run_date: date) -> list[dict]:
    """cg: logged as not contacted, so the report can compare every arm with the control."""
    return [{"run_date": run_date.isoformat(), "arm": "cg", "msisdn": str(v), "action": config.NO_CONTACT, "offers": [],
             "policy_source": "control", "propensity": 1.0} for v in sample_vals]


# ---------------------------------------------------------------------------
# Rewards (feedback at D+7) and last-campaign features
# ---------------------------------------------------------------------------
def compute_rewards(log: pd.DataFrame, fulfilments: pd.DataFrame, catalog: pd.DataFrame) -> list[dict]:
    """Rewards of logged rows from the fulfilments of the reward window that follows them (BR-23, BR-40).
    fulfilments: ACCESS_METHOD_VAL, PRODUCTNAME, DATE, SERVICE (labels.load_fulfilments)."""
    price = dict(zip(catalog["BTL_NAME"], catalog["PRICE"]))
    service = dict(zip(catalog["BTL_NAME"], catalog["SERVICE"]))
    by_val = {v: g for v, g in fulfilments.groupby("ACCESS_METHOD_VAL")} if len(fulfilments) else {}
    rows = []
    for r in log.itertuples(index=False):
        val, sent = str(r.msisdn), [o["code"] for o in (r.offers or [])]
        f = by_val.get(val)
        bought = [] if f is None else list(f["PRODUCTNAME"])
        sent_services = {service.get(c) for c in sent}
        rows.append({"run_date": r.run_date, "arm": r.arm, "msisdn": val, "action": r.action,
                     "propensity": r.propensity, "policy_source": r.policy_source,
                     "reward_any_btl": int(bool(bought)),
                     "reward_same_service": int(any(service.get(b) in sent_services for b in bought)),
                     "reward_exact_offer": int(any(b in sent for b in bought)),
                     "revenue": round(float(sum(price.get(b, 0.0) for b in bought)), 3),
                     "last_btl_date": None if f is None else max(f["DATE"]).isoformat()})
    return rows


def last_campaign_features(root: Path | None, vals: pd.Series, run_date: date) -> pd.DataFrame:
    """M3, for every line: PREV_ACTION (last logged action, 'NONE' if never contacted), PREV_OFFER_TAKEN
    (exact offer taken after that contact), DAYS_SINCE_LAST_BTL (capped at the 270-day history),
    NB_CONTACTS_LAST_4W (contacts logged in D-28..D-1, control rows excluded)."""
    cap = -config.INTERACTION_WINDOW[0]
    out = pd.DataFrame({"ACCESS_METHOD_VAL": vals.astype(str).to_numpy(), "PREV_ACTION": config.PREV_ACTION_NONE,
                        "PREV_OFFER_TAKEN": 0, "DAYS_SINCE_LAST_BTL": float(cap), "NB_CONTACTS_LAST_4W": 0.0})
    if root is None:
        return out
    log = read_log(root, end=run_date - timedelta(days=1))
    if not len(log):
        return out
    contacts = log[(log["arm"] != "cg") & (log["action"] != config.NO_CONTACT)].copy()
    contacts["d"] = pd.to_datetime(contacts["run_date"]).dt.date
    last = contacts.sort_values("d").drop_duplicates("msisdn", keep="last").set_index("msisdn")
    window = contacts[contacts["d"] >= run_date - timedelta(days=config.CONTACT_CAP_DAYS)]
    n4w = window.groupby("msisdn").size()
    rew = read_rewards(root, end=run_date - timedelta(days=1))
    taken, last_btl = {}, {}
    if len(rew):
        rew = rew.assign(d=pd.to_datetime(rew["run_date"]).dt.date).sort_values("d")
        taken = rew[rew["arm"] != "cg"].drop_duplicates("msisdn", keep="last").set_index("msisdn")["reward_exact_offer"].to_dict()
        lb = rew.dropna(subset=["last_btl_date"])
        last_btl = lb.drop_duplicates("msisdn", keep="last").set_index("msisdn")["last_btl_date"].to_dict()
    v = out["ACCESS_METHOD_VAL"]
    out["PREV_ACTION"] = v.map(last["action"]).fillna(config.PREV_ACTION_NONE).str.replace(" ", "_")
    out["PREV_OFFER_TAKEN"] = v.map(taken).fillna(0).astype(int)
    days = v.map(last_btl).map(lambda s: (run_date - date.fromisoformat(s)).days if isinstance(s, str) else cap)
    out["DAYS_SINCE_LAST_BTL"] = days.clip(upper=cap).astype(float)
    out["NB_CONTACTS_LAST_4W"] = v.map(n4w).fillna(0).astype(float)
    return out


def previous_anchors(root: Path | None, run_date: date) -> dict[str, dict]:
    """Latest anchors logged before run_date, per line (for the M4 growth guard)."""
    if root is None:
        return {}
    log = read_log(root, end=run_date - timedelta(days=1)).dropna(subset=["anchors"])
    if not len(log):
        return {}
    log = log.assign(d=pd.to_datetime(log["run_date"]).dt.date).sort_values("d").drop_duplicates("msisdn", keep="last")
    return dict(zip(log["msisdn"].astype(str), log["anchors"]))


In [ ]:
%%engine nbx.fsdr.contract package
"""V2.0 input contract (plan §4, §13-§15).

    schemas.py (nbx/fsdr/schemas.py)   the 47 tables: fields, raw types, keys, input folder
    datasource.py                      reads a run folder: 5 input folders or flat, JSON / JSONL / CSV / Parquet
    validation.py                      per-table, cross-table, layout and customer_profile checks
    contract/completeness.py           which tables each pipeline needs this week (master prompt §17.9)
    contract/cache.py                  Parquet cache, hash-partitioned on the line id (plan §15)
    contract/data_agent.py             ① data agent: step T0_load = validate + completeness + cache, or block
"""


In [ ]:
%%engine nbx.fsdr.steps
"""Step folders (plan §14): every pipeline step saves its output before the next one starts.

    data/runs/<D>/train/T0_load/  ...  data/runs/<D>/score/P10_report/  ...  data/runs/<D>/final/

    with steps.Step(runs_dir(D) / "train", "T0_load", inputs=fingerprint(files)) as st:
        if st.done:                       # same inputs, finished before: skipped (resume)
            return st.read_json("summary.json")
        st.write_table(df, "base_features")         # Parquet + CSV (full or preview)
        st.write_json("summary.json", {...})
        st.rows(rows_in=..., rows_out=...)

Each step folder holds its files and _step.json: status (running / done / failed), start, duration, rows in / out,
the fingerprint of its inputs and the list of its outputs. A step is skipped when its _step.json says done with
the same inputs; force=True re-runs it (fsdr-* --from <step>).
"""
from __future__ import annotations

import hashlib
import json
import time
from datetime import date, datetime
from pathlib import Path

import pandas as pd

from nbx.fsdr import config

STATUSES = ("running", "done", "failed", "blocked")


def runs_dir(run_date: date | str, root: Path | None = None) -> Path:
    d = run_date.isoformat() if isinstance(run_date, date) else str(run_date)
    return Path(root or config.FSDR_RUNS_DIR) / d


def fingerprint(paths: list[Path] | dict, extra: dict | None = None) -> str:
    """Cheap, stable id of a step's inputs: name, size and modification time of each file (hashing 12 GB of JSON
    every Thursday would cost minutes), plus any parameters. A dict is hashed as it is."""
    if isinstance(paths, dict):
        payload = paths
    else:
        payload = {str(p): [p.stat().st_size, int(p.stat().st_mtime)] for p in sorted(map(Path, paths)) if p.exists()}
    blob = json.dumps({"inputs": payload, "extra": extra or {}}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


class Step:
    """One step folder. Context manager: status running -> done, or failed (with the error) on an exception."""

    def __init__(self, parent: Path, name: str, inputs: str = "", force: bool = False):
        self.dir = Path(parent) / name
        self.name = name
        self.inputs = inputs
        self.force = force
        self.outputs: list[str] = []
        self._rows: dict = {}
        self._extra: dict = {}
        self._t0 = 0.0
        prev = read_step(self.dir)
        self.done = (not force and prev.get("status") == "done" and prev.get("inputs") == inputs
                     and all((self.dir / f).exists() for f in prev.get("outputs", [])))
        self.previous = prev

    # -- lifecycle -------------------------------------------------------------
    def __enter__(self) -> "Step":
        self._t0 = time.time()
        if not self.done:
            self.dir.mkdir(parents=True, exist_ok=True)
            self._write_meta("running")
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        if self.done:
            return False
        if exc is not None:
            self._write_meta("failed", error=f"{exc_type.__name__}: {exc}")
        else:
            self._write_meta(self._extra.pop("_status", "done"))
        return False

    def block(self, reason: str) -> None:
        """Ends the step as blocked (the run stops here; the outputs explain why)."""
        self._extra["_status"] = "blocked"
        self._extra["reason"] = reason

    def rows(self, **counts) -> None:
        self._rows.update({k: int(v) for k, v in counts.items()})

    def note(self, **info) -> None:
        self._extra.update(info)

    def _write_meta(self, status: str, error: str | None = None) -> None:
        meta = {"step": self.name, "status": status, "inputs": self.inputs,
                "started": datetime.fromtimestamp(self._t0).isoformat(timespec="seconds"),
                "seconds": round(time.time() - self._t0, 2), "rows": self._rows, "outputs": self.outputs,
                **{k: v for k, v in self._extra.items() if not k.startswith("_")}}
        if error:
            meta["error"] = error
        tmp = self.dir / (config.STEP_FILE + ".tmp")
        tmp.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
        tmp.replace(self.dir / config.STEP_FILE)

    # -- outputs ---------------------------------------------------------------
    def _add(self, name: str) -> Path:
        if name not in self.outputs:
            self.outputs.append(name)
        return self.dir / name

    def output(self, name: str) -> Path:
        """Registers an output written by other code (e.g. one partition file "base_features/part-03.parquet")."""
        p = self._add(name)
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def write_json(self, name: str, obj) -> Path:
        p = self._add(name)
        p.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        return p

    def write_text(self, name: str, text: str) -> Path:
        p = self._add(name)
        p.write_text(text, encoding="utf-8")
        return p

    def write_csv(self, df: pd.DataFrame, name: str) -> Path:
        p = self._add(name)
        df.to_csv(p, index=False, encoding="utf-8")
        return p

    def write_table(self, df: pd.DataFrame, stem: str, csv: str = "auto") -> list[Path]:
        """<stem>.parquet always, plus <stem>.csv in full (small tables, e.g. the 50k of scoring) or a
        preview.csv of the first rows (1M-row training tables). csv: "auto" | "full" | "preview" | "none"."""
        out = [self._add(f"{stem}.parquet")]
        _parquet_safe(df).to_parquet(out[0], index=False)
        mode = csv if csv != "auto" else ("full" if len(df) <= config.STEP_FULL_CSV_MAX_ROWS else "preview")
        if mode == "full":
            out.append(self._add(f"{stem}.csv"))
            df.to_csv(out[-1], index=False, encoding="utf-8")
        elif mode == "preview":
            out.append(self._add(f"{stem}_preview.csv"))
            df.head(config.STEP_PREVIEW_ROWS).to_csv(out[-1], index=False, encoding="utf-8")
        return out

    def read_json(self, name: str):
        return json.loads((self.dir / name).read_text(encoding="utf-8"))


def _parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Object columns mixing types (raw input values) cannot go to Parquet as they are: written as text."""
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            kinds = {type(v) for v in out[c].dropna().head(10_000)}
            if len(kinds) > 1:
                out[c] = out[c].map(lambda v: None if v is None else str(v)).astype("string")
    return out


def read_step(step_dir: Path) -> dict:
    p = Path(step_dir) / config.STEP_FILE
    try:
        return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}
    except (OSError, json.JSONDecodeError):
        return {}


def list_steps(run_dir: Path) -> list[dict]:
    """Every step of a run (train / score / final), in order, with its status and files: for the UI and the API."""
    out = []
    for phase in ("train", "score", "final"):
        d = Path(run_dir) / phase
        if not d.is_dir():
            continue
        for sd in sorted((p for p in d.iterdir() if p.is_dir()), key=lambda p: _step_order(p.name)):
            meta = read_step(sd)
            out.append({"phase": phase, "step": sd.name, "status": meta.get("status", "unknown"),
                        "seconds": meta.get("seconds"), "rows": meta.get("rows", {}),
                        "files": sorted(f.name for f in sd.iterdir() if f.is_file() and f.name != config.STEP_FILE)})
    return out


def _step_order(name: str) -> tuple:
    """T0 < T1 < ... < T8; P0 < P1 < ... < P6 < P6b < P7 < P7b < ... < P10."""
    head = name.split("_", 1)[0]
    digits = "".join(ch for ch in head[1:] if ch.isdigit())
    return (head[:1], int(digits) if digits else 0, head)


In [ ]:
%%engine nbx.fsdr.contract.cache
"""Parquet cache of a run (plan §4, §15): each input table converted once, typed, split into partitions.

    data/cache/<D>/<table>/part-00.parquet ... part-15.parquet    tables with a line (MSISDN or ACCESS_METHOD_ID)
    data/cache/<D>/<table>/part-all.parquet                        reference tables (catalogue, party, accounts, ...)
    data/cache/<D>/_cache_manifest.json                            rows and SHA-256 of every file

Every per-line table is partitioned on the SAME key: the line's ACCESS_METHOD_ID (taken from base_prepaid when the
table only has the MSISDN), canonicalised, then hashed. So all rows of one line, whatever the table, land in the
same partition number: features and segments run partition by partition (8 in parallel on the P16) and every join
stays inside a partition. A line outside the base is hashed on its own key (it joins nothing anyway).

    cache = build_cache(src, date(2026, 10, 21))
    CacheSource(date(2026, 10, 21)).read("activities_w1_score", part=3)
"""
from __future__ import annotations

import json
import time
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource, compact_frame
from nbx.fsdr.steps import _parquet_safe, sha256_file

CACHE_MANIFEST = "_cache_manifest.json"
LINE_ID_COLS = ("ACCESS_METHOD_ID", "access_method_id", "Access_Method_id")
LINE_VAL_COLS = ("ACCESS_METHOD_VAL", "access_method_val", "MSISDN", "msisdn")
ALL = "all"


def cache_dir(run_date: date | str, root: Path | None = None) -> Path:
    d = run_date.isoformat() if isinstance(run_date, date) else str(run_date)
    return Path(root or config.FSDR_CACHE_DIR) / d


def _canon(s: pd.Series) -> pd.Series:
    from nbx.fsdr.validation import norm_keys

    return norm_keys(s)


def line_map(src: DataSource) -> pd.Series:
    """Canonical MSISDN -> canonical ACCESS_METHOD_ID, from base_prepaid (first row wins on a duplicate MSISDN)."""
    if not src.has("base_prepaid"):
        return pd.Series(dtype=object)
    base = src.read("base_prepaid")
    if not {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID"} <= set(base.columns):
        return pd.Series(dtype=object)
    m = pd.Series(_canon(base["ACCESS_METHOD_ID"]).to_numpy(), index=_canon(base["ACCESS_METHOD_VAL"]).to_numpy())
    m = m[m.index.notna() & m.notna()]
    return m[~m.index.duplicated(keep="first")]


def line_key(df: pd.DataFrame, lines: pd.Series) -> pd.Series | None:
    """The partition key of every row: canonical ACCESS_METHOD_ID, or None for a table without a line column."""
    for c in LINE_ID_COLS:
        if c in df.columns:
            return _canon(df[c])
    for c in LINE_VAL_COLS:
        if c in df.columns:
            val = _canon(df[c])
            return val.map(lines).fillna("v:" + val.fillna(""))        # MSISDN outside the base: its own hash
    return None


def partition_of(keys: pd.Series, parts: int) -> np.ndarray:
    """Stable partition number of canonical keys (pandas' fixed-key hash: same key -> same partition, every run)."""
    h = pd.util.hash_pandas_object(keys.fillna("").astype(str), index=False).to_numpy()
    return (h % np.uint64(parts)).astype(np.int16)


def build_cache(src: DataSource, run_date: date, root: Path | None = None, parts: int | None = None,
                tables: list[str] | None = None) -> dict:
    """Writes every table of the source (or `tables`) to the cache; returns and writes the manifest."""
    parts = int(parts or config.CACHE_PARTITIONS)
    out = cache_dir(run_date, root)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    lines = line_map(src)
    man = {"run_date": run_date.isoformat(), "partitions": parts, "key": "ACCESS_METHOD_ID (via base_prepaid for MSISDN)",
           "built_at": datetime.now().isoformat(timespec="seconds"), "tables": {}}
    for t in tables or [t for t in src.tables() if t in schemas.TABLES]:
        df = compact_frame(src.read(t))
        tdir = out / t
        tdir.mkdir(exist_ok=True)
        for old in tdir.glob("part-*.parquet"):
            old.unlink()
        key = line_key(df, lines)
        files = {}
        if key is None:
            p = tdir / f"part-{ALL}.parquet"
            _parquet_safe(df).to_parquet(p, index=False)
            files[ALL] = {"rows": len(df), "sha256": sha256_file(p)}
        else:
            part = partition_of(key, parts)
            safe = _parquet_safe(df)
            for k in range(parts):
                p = tdir / f"part-{k:02d}.parquet"
                sub = safe[part == k]
                sub.to_parquet(p, index=False)
                files[f"{k:02d}"] = {"rows": len(sub), "sha256": sha256_file(p)}
        man["tables"][t] = {"rows": len(df), "partitioned": key is not None, "folder": schemas.TABLES[t].folder,
                            "files": files}
    man["seconds"] = round(time.time() - t0, 2)
    man["bytes"] = sum(p.stat().st_size for p in out.rglob("*.parquet"))
    (out / CACHE_MANIFEST).write_text(json.dumps(man, indent=2), encoding="utf-8")
    return man


def verify_cache(run_date: date, root: Path | None = None) -> list[str]:
    """Files whose SHA-256 no longer matches the manifest (empty list = intact)."""
    d = cache_dir(run_date, root)
    man = json.loads((d / CACHE_MANIFEST).read_text(encoding="utf-8"))
    bad = []
    for t, info in man["tables"].items():
        for k, f in info["files"].items():
            p = d / t / f"part-{k}.parquet"
            if not p.exists() or sha256_file(p) != f["sha256"]:
                bad.append(str(p.relative_to(d)))
    return bad


class CacheSource:
    """Reads the cache: a whole table, or one partition of a per-line table (reference tables have no partitions:
    any part returns the whole table)."""

    def __init__(self, run_date: date | str, root: Path | None = None):
        self.dir = cache_dir(run_date, root)
        p = self.dir / CACHE_MANIFEST
        if not p.exists():
            raise FileNotFoundError(f"No cache for {run_date}: run fsdr-load first ({p})")
        self._man = json.loads(p.read_text(encoding="utf-8"))
        self.partitions = int(self._man["partitions"])

    def tables(self) -> list[str]:
        return list(self._man["tables"])

    def has(self, table: str) -> bool:
        return table in self._man["tables"]

    def partitioned(self, table: str) -> bool:
        return bool(self._man["tables"][table]["partitioned"])

    def read(self, table: str, part: int | None = None) -> pd.DataFrame:
        if not self.has(table):
            raise FileNotFoundError(f"{table} is not in the cache {self.dir}")
        tdir = self.dir / table
        if not self.partitioned(table):
            return pd.read_parquet(tdir / f"part-{ALL}.parquet")
        if part is not None:
            return pd.read_parquet(tdir / f"part-{int(part):02d}.parquet")
        return pd.concat([pd.read_parquet(tdir / f"part-{k:02d}.parquet") for k in range(self.partitions)],
                         ignore_index=True)

    def manifest(self) -> dict:
        return dict(self._man)


In [ ]:
%%engine nbx.fsdr.contract.completeness
"""Weekly completeness (master prompt §5, §17.9): every table the pipeline about to run needs is present, has
rows and belongs to this week's run.

    rep = completeness.check(src, "train_week0", date(2026, 10, 21))
    rep["complete"]        False -> the run is blocked (a required table is missing, empty or stale)

Pipelines: train_week0 (_auto + _score + history), train_weekly (_score + purchases), score (the tables scoring reads
directly; the rest comes from the validated bundle). The lists live in config.PIPELINE_TABLES.
"""
from __future__ import annotations

from datetime import date

from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource

PIPELINES = tuple(config.PIPELINE_TABLES)


def detect_pipeline(src: DataSource) -> str:
    """Week 0 when the history tables of the segment engine are there (_auto + targeting_dataiku_auto)."""
    week0 = ["activities_w1_auto", "subs_amt_auto", "targeting_dataiku_auto"]
    return "train_week0" if all(src.has(t) for t in week0) else "train_weekly"


def check(src: DataSource, pipeline: str, run_date: date | None = None) -> dict:
    if pipeline not in config.PIPELINE_TABLES:
        raise ValueError(f"pipeline must be one of {PIPELINES}")
    spec = config.PIPELINE_TABLES[pipeline]
    man = src.manifest()
    man_date = man.get("run_date")
    fresh = run_date is None or man_date is None or man_date == run_date.isoformat()
    rows = []
    for need in ("required", "optional"):
        for t in spec[need]:
            present = src.has(t)
            n = len(src.read(t)) if present else 0
            status = "ok" if present and n else "empty" if present else "missing"
            level = "ok" if status == "ok" else "blocking" if need == "required" else "warning"
            rows.append({"table": t, "folder": schemas.folder_of(t, schemas.TABLES[t].family if t in schemas.TABLES else None),
                         "need": need, "present": present, "rows": n, "status": status, "level": level})
    blocking = [r["table"] for r in rows if r["level"] == "blocking"]
    reasons = [f"required table {r['table']} is {r['status']}" for r in rows if r["level"] == "blocking"]
    if not fresh:
        reasons.append(f"stale data: the manifest is for {man_date}, the run is for {run_date}")
    return {"pipeline": pipeline, "run_date": run_date.isoformat() if run_date else None, "manifest_run_date": man_date,
            "fresh": fresh, "complete": not blocking and fresh, "blocking": blocking,
            "warnings": [r["table"] for r in rows if r["level"] == "warning"], "reasons": reasons, "tables": rows}


In [ ]:
%%engine nbx.fsdr.validation
"""Validation of a run folder (or any DataSource) against the data contract.

Checks, each graded blocking / warning / info:
  - files: every table present (missing model-used table = blocking)
  - fields: missing fields (blocking for model-used tables), unknown extra fields (warning)
  - raw types: wrong type in a key field = blocking, elsewhere = warning
  - keys: null keys and duplicates on the primary key (blocking above 1 % unless
    duplicates are expected, e.g. stacked model scores resolved by max)
  - cross-table: child keys missing from the parent table (BR joins)
  - coverage: share of the base scored by each model (< 70 % = warning)
  - consistency: activities M1 = W1+..+W4, Sub_Amt_m1 >= its weeks, dates inside
    their BR windows, manifest anchor date = calendar anchor

The report is JSON-serialisable and also rendered as Markdown; both are
written next to the data as _validation.json / _validation.md.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import calendar as cal
from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource, JsonSource

LEVELS = ("blocking", "warning", "info")


@dataclass
class Issue:
    level: str
    check: str
    table: str
    message: str
    count: int = 0
    column: str | None = None
    examples: list = field(default_factory=list)


@dataclass
class TableReport:
    table: str
    present: bool
    rows: int = 0
    missing_fields: list[str] = field(default_factory=list)
    unknown_fields: list[str] = field(default_factory=list)
    type_errors: dict[str, int] = field(default_factory=dict)
    null_keys: dict[str, int] = field(default_factory=dict)
    duplicate_rows: int = 0


@dataclass
class ValidationReport:
    run_date: str | None
    anchor_date: str | None
    source: str
    tables: dict[str, TableReport] = field(default_factory=dict)
    issues: list[Issue] = field(default_factory=list)
    coverage: dict[str, float] = field(default_factory=dict)
    relations: list[dict] = field(default_factory=list)
    layout: str | None = None                       # "folders" / "flat" / "mixed" (plan §13)
    profile: dict = field(default_factory=dict)     # customer_profile coverage and disagreement counts
    checked_at: str = field(default_factory=lambda: datetime.now().isoformat(timespec="seconds"))

    def add(self, level: str, check: str, table: str, message: str, count: int = 0, column: str | None = None,
            examples: list | None = None) -> None:
        self.issues.append(Issue(level, check, table, message, int(count), column, list(examples or [])))

    @property
    def status(self) -> str:
        if any(i.level == "blocking" for i in self.issues):
            return "blocked"
        return "warnings" if any(i.level == "warning" for i in self.issues) else "ok"

    @property
    def usable(self) -> bool:
        return self.status != "blocked"

    def counts(self) -> dict[str, int]:
        return {lv: sum(1 for i in self.issues if i.level == lv) for lv in LEVELS}

    def to_dict(self) -> dict:
        return {"status": self.status, "usable": self.usable, "run_date": self.run_date, "anchor_date": self.anchor_date,
                "source": self.source, "checked_at": self.checked_at, "issue_counts": self.counts(),
                "issues": [asdict(i) for i in self.issues], "coverage": self.coverage, "relations": self.relations,
                "layout": self.layout, "profile": self.profile,
                "tables": {k: asdict(v) for k, v in self.tables.items()}}

    def to_markdown(self) -> str:
        c = self.counts()
        icon = {"ok": "✅", "warnings": "⚠️", "blocked": "⛔"}[self.status]
        lines = [f"# FSDR validation report — {self.run_date or '?'}", "",
                 f"**Status:** {icon} {self.status}  ·  blocking {c['blocking']} · warnings {c['warning']} · info {c['info']}",
                 f"**Anchor date (T):** {self.anchor_date or '?'}  ·  **Source:** {self.source}  ·  checked {self.checked_at}", ""]
        for lv in LEVELS:
            items = [i for i in self.issues if i.level == lv]
            if not items:
                continue
            lines += [f"## {lv.capitalize()} ({len(items)})", "", "| Check | Table | Column | Count | Message |", "|---|---|---|---|---|"]
            for i in items:
                ex = f" e.g. {', '.join(map(str, i.examples[:3]))}" if i.examples else ""
                lines.append(f"| {i.check} | `{i.table}` | {i.column or ''} | {i.count} | {i.message}{ex} |")
            lines.append("")
        if self.coverage:
            lines += ["## Model-score coverage of the base", "", "| Model file | Coverage |", "|---|---|"]
            lines += [f"| `{k}` | {v:.1%} |" for k, v in self.coverage.items()]
            lines.append("")
        if self.profile:
            lines += [f"## {config.PROFILE_TABLE}", "", "| Check | Value |", "|---|---|"]
            lines += [f"| {k} | {v:.1%} |" if k == "coverage" else f"| {k} | {v} |" for k, v in self.profile.items()]
            lines.append("")
        lines += ["## Tables", "", "| Table | Present | Rows | Missing fields | Unknown fields | Type errors | Null keys | Duplicates |",
                  "|---|---|---|---|---|---|---|---|"]
        for t in self.tables.values():
            lines.append(f"| `{t.table}` | {'yes' if t.present else '**no**'} | {t.rows} | {len(t.missing_fields)} | "
                         f"{len(t.unknown_fields)} | {sum(t.type_errors.values())} | {sum(t.null_keys.values())} | {t.duplicate_rows} |")
        return "\n".join(lines) + "\n"


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def norm_keys(s: pd.Series) -> pd.Series:
    """_norm_key on a whole column, vectorised (review #15)."""
    from nbx.fsdr.features import canonical_keys

    return canonical_keys(s, strip_zero=False, empty_none=False)


def _key_set(s: pd.Series, drop_empty: bool = False) -> set:
    k = norm_keys(s).dropna()
    return set(k[k != ""]) if drop_empty else set(k)


def _norm_key(v) -> str | None:
    """Key value -> canonical string ('91234567', 1001, 1001.0 and '1001' compare equal)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (int, np.integer)):
        s = str(int(v))
    elif isinstance(v, (float, np.floating)):
        s = str(int(v)) if float(v).is_integer() else str(v)
    else:
        s = str(v).strip()
    return s


def _share(count: int, rows: int) -> float:
    return count / rows if rows else 0.0


def _over(count: int, rows: int) -> bool:
    return _share(count, rows) > config.BLOCKING_KEY_ISSUE_SHARE


def _num(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce") if col in df.columns else pd.Series(dtype=float)


# ---------------------------------------------------------------------------
# Checks
# ---------------------------------------------------------------------------
def _check_table(src: DataSource, schema: schemas.TableSchema, rep: ValidationReport,
                 pipeline: str | None = None) -> pd.DataFrame | None:
    name = schema.name
    if not src.has(name):
        level = "blocking" if schema.used_by_model and schema.required and schema.family != "model_score" else "warning"
        # V4.0: a weekly run does not need the week-0 history (*_auto): missing, it is a warning, not a block
        not_needed = (pipeline in config.PIPELINE_TABLES and level == "blocking"
                      and name not in config.PIPELINE_TABLES[pipeline]["required"])
        if not_needed:
            level = "warning"
        why = ("missing model file: its score falls back to random 0-0.02 (BR-47)" if schema.family == "model_score"
               else f"not needed by the {pipeline} pipeline (week-0 history)" if not_needed
               else "required table missing" if level == "blocking"
               else "optional table missing (fallback used)" if schema.used_by_model else "table missing (not used by the model)")
        rep.add(level, "file_present", name, why)
        rep.tables[name] = TableReport(name, present=False)
        return None
    df = src.read(name)
    tr = TableReport(name, present=True, rows=len(df))
    rep.tables[name] = tr
    if not len(df):
        rep.add("warning" if not schema.used_by_model else "blocking", "rows", name, "file has no rows")
        return df

    tr.missing_fields = [f for f in schema.field_names if f not in df.columns]
    tr.unknown_fields = [c for c in df.columns if c not in schema.field_names]
    if tr.missing_fields:
        rep.add("blocking" if schema.used_by_model else "warning", "fields_missing", name,
                f"{len(tr.missing_fields)} field(s) missing", len(tr.missing_fields), examples=tr.missing_fields)
    if tr.unknown_fields:
        rep.add("warning", "fields_unknown", name, "extra field(s) not in the contract (ignored)",
                len(tr.unknown_fields), examples=tr.unknown_fields)

    for f in schema.fields:
        if f.name not in df.columns:
            continue
        bad, examples = schemas.type_errors(df[f.name], f.raw_type)
        if bad:
            tr.type_errors[f.name] = bad
            rep.add("blocking" if f.key else "warning", "raw_type", name,
                    f"{bad} value(s) are not {f.raw_type}", bad, f.name, examples)
        if f.key:
            nulls = int(df[f.name].isna().sum())
            if nulls:
                tr.null_keys[f.name] = nulls
                rep.add("blocking" if _over(nulls, len(df)) else "warning", "null_key", name,
                        f"{nulls} null key value(s) ({_share(nulls, len(df)):.1%})", nulls, f.name)

    pk = [c for c in schema.primary_key if c in df.columns]
    if pk:
        keyed = df[pk].apply(norm_keys)
        dups = int(keyed.duplicated().sum())
        tr.duplicate_rows = dups
        if dups:
            blocking = schema.duplicates_blocking and _over(dups, len(df))
            note = "" if schema.duplicates_blocking else " (expected: resolved downstream, e.g. max of stacked scores)"
            rep.add("blocking" if blocking else "warning", "duplicates", name,
                    f"{dups} duplicate row(s) on ({', '.join(pk)}){note}", dups, ",".join(pk))
    return df


def _check_relations(src: DataSource, rep: ValidationReport) -> None:
    for r in schemas.relations():
        if not (src.has(r.child) and src.has(r.parent)):
            continue
        child, parent = src.read(r.child), src.read(r.parent)
        if r.child_col not in child.columns or r.parent_col not in parent.columns:
            continue
        parent_keys = _key_set(parent[r.parent_col])
        ck = norm_keys(child[r.child_col]).dropna().tolist()
        if not ck:
            continue
        orphans = sum(1 for k in ck if k not in parent_keys)
        rep.relations.append({"child": f"{r.child}.{r.child_col}", "parent": f"{r.parent}.{r.parent_col}",
                              "child_keys": len(ck), "orphans": orphans, "orphan_share": round(_share(orphans, len(ck)), 4)})
        if orphans:
            level = "blocking" if r.blocking and _over(orphans, len(ck)) else "warning"
            rep.add(level, "cross_table", r.child, f"{orphans} {r.child_col} value(s) not found in {r.parent}.{r.parent_col}"
                    + (f" -- {r.note}" if r.note else ""), orphans, r.child_col)


def _check_coverage(src: DataSource, rep: ValidationReport) -> None:
    if not src.has("base_prepaid"):
        return
    base = _key_set(src.read("base_prepaid")["ACCESS_METHOD_VAL"], drop_empty=True)
    for m in config.MODEL_SCORE_REGISTRY:
        if not src.has(m.table) or "access_method_val" not in src.read(m.table).columns:
            continue
        scored = _key_set(src.read(m.table)["access_method_val"], drop_empty=True)
        cov = _share(len(scored & base), len(base))
        rep.coverage[m.table] = round(cov, 4)
        if m.tenure_range is None and cov < config.MIN_SCORE_COVERAGE:
            rep.add("warning", "coverage", m.table, f"scores only {cov:.1%} of the base (missing -> random 0-0.02, BR-47)")


def _check_consistency(src: DataSource, rep: ValidationReport, run_date: date | None) -> None:
    # activities: M1 window = W1..W4 exactly (BR 4.2)
    for anchor in ("T", "D"):
        names = {p: schemas.activity_table_name(p, anchor) for p in ("W1", "W2", "W3", "W4", "M1")}
        if not all(src.has(t) for t in names.values()):
            continue
        m1 = src.read(names["M1"])
        if "ACCESS_METHOD_VAL" not in m1.columns or "Local_Voice_min_M1" not in m1.columns:
            continue
        tot = pd.Series(0.0, index=norm_keys(m1["ACCESS_METHOD_VAL"]))
        weeks = pd.Series(dtype=float)
        for w in ("W1", "W2", "W3", "W4"):
            df = src.read(names[w])
            col = f"Local_Voice_min_{w}"
            if col in df.columns:
                s = pd.Series(_num(df, col).to_numpy(), index=norm_keys(df["ACCESS_METHOD_VAL"]))
                weeks = weeks.add(s.groupby(level=0).sum(), fill_value=0)
        m1v = pd.Series(_num(m1, "Local_Voice_min_M1").to_numpy(), index=tot.index).groupby(level=0).sum()
        diff = (m1v - weeks.reindex(m1v.index).fillna(0)).abs()
        off = int((diff > 0.05 + 0.001 * m1v.abs()).sum())
        if off:
            rep.add("warning", "window_consistency", names["M1"], f"{off} line(s) where Local_Voice_min_M1 != W1+W2+W3+W4", off)

    # subscriptions: M1 (30 days incl. anchor) contains W1..W4 (28 days)
    for t in ("subs_amt_auto", "subs_amt_score"):
        if not src.has(t):
            continue
        df = src.read(t)
        cols = [f"Sub_Amt_w{i}" for i in range(1, 5)]
        if not set(cols + ["Sub_Amt_m1"]) <= set(df.columns):
            continue
        off = int((_num(df, "Sub_Amt_m1") + 0.01 < sum(_num(df, c).fillna(0) for c in cols)).sum())
        if off:
            rep.add("warning", "window_consistency", t, f"{off} line(s) where Sub_Amt_m1 < Sub_Amt_w1..w4", off)

    if run_date is None:
        return
    win = cal.campaign_windows(run_date)
    if src.has("targeting_dataiku_auto") and "date1" in src.read("targeting_dataiku_auto").columns:
        d1 = pd.to_datetime(src.read("targeting_dataiku_auto")["date1"], errors="coerce").dt.date
        out = int(((d1 < win["targeting"].start) | (d1 > win["targeting"].end)).sum())
        if out:
            rep.add("warning", "date_window", "targeting_dataiku_auto",
                    f"{out} date1 value(s) outside {win['targeting'].start}..{win['targeting'].end} (BR-21)", out, "date1")
    if src.has("subs_btl_auto") and "FULFILMENTSUCCESSDATE_TIME" in src.read("subs_btl_auto").columns:
        ts = pd.to_datetime(src.read("subs_btl_auto")["FULFILMENTSUCCESSDATE_TIME"], errors="coerce").dt.date
        out = int(((ts < win["reward"].start) | (ts > win["reward"].end)).sum())
        if out:
            rep.add("warning", "date_window", "subs_btl_auto",
                    f"{out} fulfilment(s) outside {win['reward'].start}..{win['reward'].end} (BR-23)", out,
                    "FULFILMENTSUCCESSDATE_TIME")
    if src.has("base_prepaid") and "ACCESS_METHOD_VAL" in src.read("base_prepaid").columns:
        vals = norm_keys(src.read("base_prepaid")["ACCESS_METHOD_VAL"]).dropna()
        bad = int((~vals.str.fullmatch(r"\d{8}") | (vals <= config.MSISDN_MIN_EXCLUSIVE)).sum())
        if bad:
            rep.add("warning", "msisdn_format", "base_prepaid",
                    f"{bad} MSISDN(s) not 8 digits > '{config.MSISDN_MIN_EXCLUSIVE}' (BR-02: filtered out downstream)", bad)


def _check_layout(src: JsonSource, rep: ValidationReport) -> None:
    """Plan §13: the 5 input folders. A misplaced table still loads (warning); a table found twice blocks."""
    rep.layout = src.layout
    if src.layout != "folders":
        rep.add("warning", "layout", "-", f"{src.layout} layout: tables are expected in the 5 folders "
                f"{', '.join(config.INPUT_FOLDERS)} (plan §13); files loaded as they are")
    for t, (found, expected) in sorted(src.misplaced().items()):
        rep.add("warning", "folder", t, f"found in {found}/, expected in {expected}/ (loaded anyway)")
    for t, (found, expected) in sorted(src.misfiled().items()):
        rep.add("warning", "twin", t, f"found under {found}/, expected under {expected}/ (V4.0 layout; loaded anyway)")
    root = src.folder.parent.parent if src.twins.keys() - {""} else src.folder
    for t, paths in sorted(src.duplicate_files.items()):
        rep.add("blocking", "file_duplicate", t, f"{len(paths)} files for one table: keep only one",
                len(paths), examples=[str(Path(p).relative_to(root)) for p in paths])
    _check_formats(src, rep)


def _check_formats(src: JsonSource, rep: ValidationReport) -> None:
    """Master §2 (V3.0): production tables are .csv.gz. Another format is accepted only for tests
    (FSDR_ALLOW_TEST_FORMATS=true) and for the dummy generator's data (its manifest says source=generator)."""
    from nbx.fsdr import csvstd

    other = {t: csvstd.file_format(src.path(t)) for t in src.tables()
             if src.path(t) is not None and csvstd.file_format(src.path(t)) != config.PRODUCTION_FORMAT}
    if not other:
        return
    if src.manifest().get("source") == "generator":            # the dummy generator's data: accepted as it is
        return
    fmts = sorted(set(other.values()))
    if config.ALLOW_TEST_FORMATS:
        rep.add("info", "format", "-", f"{len(other)} table(s) in a test format ({', '.join(fmts)}): accepted "
                f"(FSDR_ALLOW_TEST_FORMATS=true); production inputs must be {config.PRODUCTION_FORMAT}", len(other))
    else:
        rep.add("blocking", "format", "-", f"{len(other)} table(s) are not {config.PRODUCTION_FORMAT} "
                f"({', '.join(fmts)}): production inputs must be the Dataiku compressed CSV export (master §2)",
                len(other), examples=sorted(other)[:5])


def _check_profile(src: DataSource, rep: ValidationReport, run_date: date | None) -> None:
    """customer_profile (master prompt §5): coverage, ages outside 15-100, negative tenure, disagreement with
    party_*.AGE (> 2 years) and with the activation-date tenure (> 30 days). Warnings with counts: the profile
    wins downstream, so a disagreement is reported, never repaired."""
    t = config.PROFILE_TABLE
    if not src.has(t) or not src.has("base_prepaid"):
        return
    prof, base = src.read(t), src.read("base_prepaid")
    if not {"msisdn", "age", "tenure"} <= set(prof.columns) or "ACCESS_METHOD_VAL" not in base.columns:
        return
    p = pd.DataFrame({"msisdn": norm_keys(prof["msisdn"]), "age": _num(prof, "age"), "tenure": _num(prof, "tenure")})
    p = p.dropna(subset=["msisdn"]).drop_duplicates("msisdn", keep="last")
    b = pd.DataFrame({"msisdn": norm_keys(base["ACCESS_METHOD_VAL"]),
                      "party": norm_keys(base["PARTY_ID"]) if "PARTY_ID" in base.columns else None,
                      "start": base["access_method_subs_start_dt"] if "access_method_subs_start_dt" in base.columns else None})
    b = b.dropna(subset=["msisdn"]).drop_duplicates("msisdn")
    m = b.merge(p, on="msisdn", how="left")
    covered = int(m["msisdn"].isin(p["msisdn"]).sum())
    cov = _share(covered, len(b))
    rep.profile = {"rows": len(prof), "base_lines": len(b), "covered": covered, "coverage": round(cov, 4)}
    if cov < config.PROFILE_MIN_COVERAGE:
        rep.add("warning", "profile_coverage", t, f"profile for {cov:.1%} of the base (< {config.PROFILE_MIN_COVERAGE:.0%}): "
                "the others use party_scoring.AGE and the activation-date tenure", len(b) - covered)
    lo, hi = config.PROFILE_AGE_RANGE
    checks = {"age_out_of_range": (((p["age"] < lo) | (p["age"] > hi)).sum(), f"age outside {lo}-{hi}"),
              "negative_tenure": ((p["tenure"] < 0).sum(), "negative tenure")}
    party_t = "party_scoring" if src.has("party_scoring") else "party_auto" if src.has("party_auto") else None
    if party_t and "party" in m and {"party_id", "AGE"} <= set(src.read(party_t).columns):
        party = src.read(party_t)
        ages = pd.Series(_num(party, "AGE").to_numpy(), index=norm_keys(party["party_id"]))
        ages = ages[~ages.index.duplicated(keep="last")]
        diff = (m["age"] - m["party"].map(ages)).abs()
        tol = config.PROFILE_AGE_TOLERANCE_YEARS
        checks["age_vs_party"] = ((diff > tol).sum(), f"age differs from {party_t}.AGE by more than {tol} years")
    if run_date is not None and m["start"].notna().any():
        start = pd.to_datetime(m["start"], errors="coerce")
        days = (pd.Timestamp(run_date) - start).dt.days
        if config.PROFILE_TENURE_UNIT == "months":
            days = days / 30
        tol = config.PROFILE_TENURE_TOLERANCE_DAYS / (30 if config.PROFILE_TENURE_UNIT == "months" else 1)
        checks["tenure_vs_activation"] = (((m["tenure"] - days).abs() > tol).sum(),
                                          f"tenure differs from run date - access_method_subs_start_dt by more than "
                                          f"{config.PROFILE_TENURE_TOLERANCE_DAYS} days")
    for name, (count, what) in checks.items():
        count = int(count)
        rep.profile[name] = count
        if count:
            rep.add("warning", f"profile_{name}", t, f"{count} line(s): {what}", count)


def validate(src: DataSource, run_date: date | None = None, pipeline: str | None = None) -> ValidationReport:
    """pipeline (train_week0 / train_weekly / score): a table that pipeline does not need may be missing (V4.0)."""
    manifest = src.manifest()
    if run_date is None and manifest.get("run_date"):
        run_date = date.fromisoformat(manifest["run_date"])
    anchor = cal.anchor_date(run_date) if run_date else None
    rep = ValidationReport(run_date.isoformat() if run_date else None, anchor.isoformat() if anchor else None,
                           manifest.get("source", "unknown"))
    if not manifest:
        rep.add("warning", "manifest", config.MANIFEST_FILE, "no _manifest.json: run date taken from the folder name or argument")
    elif manifest.get("anchor_date") and anchor and manifest["anchor_date"] != anchor.isoformat():
        rep.add("blocking", "manifest", config.MANIFEST_FILE,
                f"manifest anchor_date {manifest['anchor_date']} != calendar anchor {anchor} for run date {run_date}")
    if run_date is None:
        rep.add("warning", "run_date", "-", "run date unknown: date-window checks skipped")

    for schema in schemas.TABLES.values():
        _check_table(src, schema, rep, pipeline)
        if isinstance(src, JsonSource):      # raw values are only needed by the per-table checks above (review #15)
            src.compact_table(schema.name)
    if isinstance(src, JsonSource):
        for t in src.unknown_files():
            rep.add("info", "file_unknown", t, "file not in the data contract (ignored)")
        _check_layout(src, rep)
    _check_relations(src, rep)
    _check_coverage(src, rep)
    _check_consistency(src, rep, run_date)
    _check_profile(src, rep, run_date)
    return rep


def validate_folder(folder: str | Path, run_date: date | None = None, write: bool = True) -> ValidationReport:
    """Validates a run folder; the run date comes from the manifest, the argument or the folder name."""
    folder = Path(folder)
    src = JsonSource(folder)
    if run_date is None and not src.manifest():
        try:
            run_date = date.fromisoformat(folder.name)
        except ValueError:
            pass
    rep = validate(src, run_date)
    if write:
        (folder / config.VALIDATION_JSON).write_text(json.dumps(rep.to_dict(), indent=2, ensure_ascii=False, default=str),
                                                     encoding="utf-8")
        (folder / config.VALIDATION_MD).write_text(rep.to_markdown(), encoding="utf-8")
    return rep


In [ ]:
%%engine nbx.fsdr.contract.data_agent
"""① Data agent (master prompt §5): step T0_load of the training pipeline.

    rep = run_t0(Path("data/input/2026-10-21"))            # pipeline detected: train_week0 or train_weekly
    rep["status"]     "done" (the cache is built) or "blocked" (the run stops: rep["explanation"] says why)

What it does, in order, each result saved in data/runs/<D>/train/T0_load/ (plan §14):
  1. validate every table against the contract (fields, raw types, keys, duplicates, cross-table orphans, model
     coverage, date windows, the 5-folder layout, customer_profile)    -> validation_report.json / .md
  2. weekly completeness for the pipeline about to run (§17.9)          -> completeness.csv / .json
  3. if nothing blocks: convert every table to the partitioned Parquet cache (plan §15) -> cache_manifest.json
  4. a plain-language summary of what blocks or warns                   -> summary.json

The checks are code; the agent only reports. Blocking issues stop the run here, before any feature is computed.
A re-run with the same input files is skipped (resume), unless force=True.
"""
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.contract import cache as cache_mod
from nbx.fsdr.contract import completeness
from nbx.fsdr.datasource import JsonSource
from nbx.fsdr.steps import Step, fingerprint, runs_dir
from nbx.fsdr.validation import ValidationReport, validate

STEP = "T0_load"


def run_date_of(folder: Path, src: JsonSource, run_date: date | None = None) -> date | None:
    if run_date:
        return run_date
    man = src.manifest()
    if man.get("run_date"):
        return date.fromisoformat(man["run_date"])
    try:
        return date.fromisoformat(Path(folder).name)
    except ValueError:
        return None


def explain(rep: ValidationReport, comp: dict) -> str:
    """Plain-language summary of the run's data: what blocks first, then the main warnings (deterministic)."""
    c = rep.counts()
    lines = []
    if rep.status == "blocked" or not comp["complete"]:
        lines.append("The run is BLOCKED: no feature is computed until these are fixed.")
        for i in [i for i in rep.issues if i.level == "blocking"][:10]:
            lines.append(f"- {i.table}{'.' + i.column if i.column else ''}: {i.message}")
        for r in comp["reasons"]:
            lines.append(f"- {r}")
    else:
        lines.append(f"The data of {rep.run_date} is usable for {comp['pipeline']}: "
                     f"{len(rep.tables)} tables checked, {c['warning']} warning(s), {c['info']} note(s).")
    warns = [i for i in rep.issues if i.level == "warning"]
    if warns:
        lines.append("Warnings (the run continues):")
        lines += [f"- {i.table}{'.' + i.column if i.column else ''}: {i.message}" for i in warns[:10]]
        if len(warns) > 10:
            lines.append(f"- ... and {len(warns) - 10} more in validation_report.md")
    if comp["warnings"]:
        lines.append(f"Optional tables missing (fallback used): {', '.join(comp['warnings'])}.")
    return "\n".join(lines)


def run_t0(folder: Path | str, pipeline: str | None = None, run_date: date | None = None,
           runs_root: Path | None = None, cache_root: Path | None = None, force: bool = False) -> dict:
    folder = Path(folder)
    src = JsonSource(folder)
    run_date = run_date_of(folder, src, run_date)
    if run_date is None:
        raise ValueError(f"{folder}: no run date (no _manifest.json, folder name is not YYYY-MM-DD, none given)")
    pipeline = pipeline or completeness.detect_pipeline(src)
    files = [src.path(t) for t in src.tables()] + src.manifest_files()
    inputs = fingerprint(files, {"pipeline": pipeline, "parts": config.CACHE_PARTITIONS, "cache": str(cache_root or "")})
    with Step(runs_dir(run_date, runs_root) / "train", STEP, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        rep = validate(src, run_date, pipeline)
        st.write_json("validation_report.json", rep.to_dict())
        st.write_text("validation_report.md", rep.to_markdown())
        comp = completeness.check(src, pipeline, run_date)
        st.write_json("completeness.json", comp)
        st.write_csv(pd.DataFrame(comp["tables"]), "completeness.csv")
        rows_in = sum(t.rows for t in rep.tables.values())
        blocked = rep.status == "blocked" or not comp["complete"]
        twins = config.run_twins(folder)
        if twins and src.misfiled():
            st.write_json("misfiled_tables.json", {t: {"found": f, "expected": e} for t, (f, e) in src.misfiled().items()})
        summary = {"run_date": run_date.isoformat(), "pipeline": pipeline, "source": str(folder),
                   "training_source": str(twins.get(config.TRAINING_FOLDER, folder)),
                   "scoring_source": str(twins.get(config.SCORING_FOLDER, folder)),
                   "misfiled_tables": len(src.misfiled()),
                   "layout": rep.layout, "status": "blocked" if blocked else "done",
                   "validation": rep.status, "issue_counts": rep.counts(), "complete": comp["complete"],
                   "tables": len(src.tables()), "rows_in": rows_in, "profile": rep.profile,
                   "explanation": explain(rep, comp), "step_dir": str(st.dir)}
        if blocked:
            st.block("; ".join(comp["reasons"] + [i.message for i in rep.issues if i.level == "blocking"][:5]))
        else:
            man = cache_mod.build_cache(src, run_date, cache_root)
            st.write_json("cache_manifest.json", man)
            summary["cache"] = {"dir": str(cache_mod.cache_dir(run_date, cache_root)), "bytes": man["bytes"],
                                "partitions": man["partitions"], "seconds": man["seconds"]}
        st.rows(rows_in=rows_in, tables=len(src.tables()))
        st.note(pipeline=pipeline, validation=rep.status)
        st.write_json("summary.json", summary)
        return summary


def load_summary(run_date: date, runs_root: Path | None = None) -> dict:
    p = runs_dir(run_date, runs_root) / "train" / STEP / "summary.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


In [ ]:
%%engine nbx.fsdr.csvstd
"""The FSDR CSV standard (master prompt §2, §20): the one format of every input AND output table file.

    write_csv(df, path)                 <name>.csv.gz (gzip level 6), + <name>.schema.json next to it
    read_csv(path)                      every column as text (IDs keep their leading zeros), "" for missing
    write_outputs_manifest(dir, files)  _outputs_manifest.json: file list, row counts, SHA-256, run id, versions

| Compression | gzip, level 6 (mtime 0: the same table gives the same bytes)  |
| Encoding    | UTF-8, no BOM                                                 |
| Separator   | ,   quoting only when needed (comma, quote, line break), "" inside |
| Line ending | \\n                                                            |
| Decimal     | dot, no thousands separator, never scientific notation        |
| Dates       | YYYY-MM-DD / YYYY-MM-DD HH:MM:SS                              |
| Missing     | empty field (never nan / None / NULL)                         |
| Index       | none                                                          |
"""
from __future__ import annotations

import csv
import gzip
import io
import json
import math
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config

SUFFIX = ".csv.gz"
SCHEMA_SUFFIX = ".schema.json"
OUTPUTS_MANIFEST = "_outputs_manifest.json"


def table_name(path: Path) -> str:
    """activities_w1_score.csv.gz -> activities_w1_score (also .csv / .json / .jsonl / .parquet)."""
    name = Path(path).name
    for suffix in (SUFFIX, *config.INPUT_FORMATS):
        if name.lower().endswith(suffix):
            return name[: -len(suffix)]
    return Path(path).stem


def file_format(path: Path) -> str:
    """".csv.gz", ".csv", ".json", ".jsonl" or ".parquet" ("" for anything else)."""
    name = Path(path).name.lower()
    return next((f for f in (SUFFIX, *config.INPUT_FORMATS) if name.endswith(f)), "")


def format_value(v) -> str:
    """One cell -> its text in the FSDR CSV standard."""
    if v is None or v is pd.NA or v is pd.NaT:
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "1" if v else "0"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        f = float(v)
        if math.isnan(f) or math.isinf(f):
            return ""
        return np.format_float_positional(f, trim="0")              # 7.0, 12.5, 0.0001 -- never 1e-04
    if isinstance(v, Decimal):
        return format(v, "f")
    if isinstance(v, datetime):                                     # pd.Timestamp included
        return v.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(v, date):
        return v.isoformat()
    return str(v)


def to_text_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Every cell formatted as the standard writes it (also what read_csv gives back)."""
    out = {}
    for c in df.columns:
        out[c] = _text_column(df[c])
    return pd.DataFrame(out, columns=df.columns, index=df.index)


def _float_text(s: pd.Series) -> pd.Series:
    """Vectorised repr of floats ("7.0", "12.5"); the rare exponent forms are rewritten positionally."""
    f = s.astype("float64")
    t = f.astype(str)
    exp = t.str.contains("e", regex=False) & f.notna() & np.isfinite(f)
    if exp.any():
        t[exp] = [np.format_float_positional(v, trim="0") for v in f[exp]]
    return t.where(f.notna() & np.isfinite(f), "")


def _text_column(s: pd.Series) -> pd.Series:
    """One column -> text, by dtype first (fast paths for 1M-row tables), value by value only for mixed columns."""
    if pd.api.types.is_datetime64_any_dtype(s):
        d = s.dropna()
        fmt = "%Y-%m-%d" if bool((d.dt.normalize() == d).all()) else "%Y-%m-%d %H:%M:%S"
        return s.dt.strftime(fmt).fillna("")
    if pd.api.types.is_bool_dtype(s):
        return s.map(format_value)
    if pd.api.types.is_integer_dtype(s):
        return s.astype("string").fillna("").astype(object)
    if pd.api.types.is_float_dtype(s):
        return _float_text(s)
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind in ("string", "empty"):
        return s.where(s.notna(), "").astype(str)
    if kind == "integer":
        return s.astype("Int64").astype("string").fillna("").astype(object)
    if kind == "floating":
        return _float_text(s)
    return s.map(format_value)


def column_types(df: pd.DataFrame) -> list[dict]:
    """Column names and types for <name>.schema.json (identifier columns are always text)."""
    cols = []
    for c in df.columns:
        s = df[c]
        if str(c).upper() in config.ID_TEXT_COLUMNS:
            t = "string"
        elif pd.api.types.is_bool_dtype(s):
            t = "boolean"
        elif pd.api.types.is_integer_dtype(s):
            t = "bigint"
        elif pd.api.types.is_float_dtype(s):
            t = "double"
        elif pd.api.types.is_datetime64_any_dtype(s):
            t = "timestamp"
        else:
            t = "string"
        cols.append({"name": str(c), "type": t})
    return cols


def write_csv(df: pd.DataFrame, path: Path, schema: bool = True, compress: bool = True) -> Path:
    """Writes df in the FSDR CSV standard. path may end in .csv.gz (default) or .csv (compress=False)."""
    path = Path(path)
    if compress and not path.name.endswith(SUFFIX):
        path = path.with_name(table_name(path) + SUFFIX)
    path.parent.mkdir(parents=True, exist_ok=True)
    buf = io.StringIO()
    w = csv.writer(buf, delimiter=",", quotechar='"', quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    w.writerow([str(c) for c in df.columns])
    w.writerows(to_text_frame(df).itertuples(index=False, name=None))
    data = buf.getvalue().encode("utf-8")
    if compress:
        with path.open("wb") as fh, gzip.GzipFile(fileobj=fh, mode="wb", compresslevel=config.CSV_GZIP_LEVEL, mtime=0,
                                                  filename="") as gz:
            gz.write(data)
    else:
        path.write_bytes(data)
    if schema:
        schema_path(path).write_text(json.dumps({"table": table_name(path), "columns": column_types(df)}, indent=2),
                                     encoding="utf-8")
    return path


def schema_path(path: Path) -> Path:
    return Path(path).with_name(table_name(path) + SCHEMA_SUFFIX)


def read_csv(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """A .csv.gz / .csv of the standard -> text frame ("" stays "", so a missing value is never "nan")."""
    path = Path(path)
    opener = gzip.open if path.name.lower().endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as fh:
        return pd.read_csv(fh, dtype=str, keep_default_na=False, na_filter=False, usecols=columns)


def write_outputs_manifest(folder: Path, files: list[Path], run_id: str = "", versions: dict | None = None) -> Path:
    from nbx.fsdr.steps import sha256_file

    entries = []
    for f in files:
        f = Path(f)
        rows = len(read_csv(f)) if file_format(f) in (SUFFIX, ".csv") else None
        entries.append({"file": f.name, "rows": rows, "sha256": sha256_file(f),
                        "schema": schema_path(f).name if schema_path(f).exists() else None})
    p = Path(folder) / OUTPUTS_MANIFEST
    p.write_text(json.dumps({"run_id": run_id, "versions": versions or {}, "files": entries}, indent=2), encoding="utf-8")
    return p


In [ ]:
%%engine nbx.fsdr.offer_rules
"""Offer business rules -- port of production Codes 3 (constraint arm) and 4 (free arm).

  - catalogue, conflicts, eligibility                       BR-63 .. BR-65, conflicting_offers_h
  - ARPU anchor = mean + std of periods, data 90 % / voice 10 %     BR-68 .. BR-77
  - action -> offer families and generosity (normal | all_L2)      BR-66, free arm BR-67
  - 3-pass price-band selection, at most 2 offers per validity      BR-78 .. BR-82
  - NEW: two conflicting offers are never sent together

Fixes vs the original: CROSS SELL VOICE gets its own family filter (the
original repeated "CROSS SELL DATA"), and conflicts are enforced.
"""
from __future__ import annotations

import hashlib
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS, WEEKS
from nbx.fsdr.datasource import DataSource
from nbx.fsdr.features import norm_val, upper

ANCHOR_BY_SERVICE = {"COMBO": "TOTAL", "DATA": "DATA", "VOICE": "VOICE"}   # BR-77


# ---------------------------------------------------------------------------
# Catalogue and conflicts
# ---------------------------------------------------------------------------
def load_catalog(src: DataSource) -> pd.DataFrame:
    """btl_catalog with numeric price / validity / gb / min (production stores them as strings)."""
    df = src.read("btl_catalog").copy()
    df.columns = [str(c).upper().replace(" ", "_") for c in df.columns]
    for c in ("PRICE", "GB", "MIN", "PRICE_DATA", "PRICE_VOICE"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["VALIDITY"] = pd.to_numeric(df["VALIDITY"], errors="coerce").astype("Int64")
    for c in ("SERVICE", "GENEROSITY"):
        df[c] = df[c].astype(str).str.strip().str.upper()
    df["BTL_NAME"] = df["BTL_NAME"].astype(str).str.strip()
    return df.drop_duplicates("BTL_NAME").reset_index(drop=True)


def catalog_version(catalog: pd.DataFrame) -> str:
    key = "|".join(f"{r.BTL_NAME}:{r.PRICE}:{r.VALIDITY}" for r in catalog.sort_values("BTL_NAME").itertuples())
    return hashlib.sha1(key.encode()).hexdigest()[:10]


def load_conflicts(src: DataSource) -> set[frozenset]:
    if not src.has("conflicting_offers_h"):
        return set()
    df = src.read("conflicting_offers_h")
    return {frozenset((str(a).strip(), str(b).strip())) for a, b in zip(df["Offer_A"], df["Offer_B"]) if a and b}


def conflicts_with(name: str, chosen: list[str], conflicts: set[frozenset]) -> bool:
    return any(frozenset((name, c)) in conflicts for c in chosen)


# ---------------------------------------------------------------------------
# ARPU anchors (BR-68 .. BR-77)
# ---------------------------------------------------------------------------
def arpu_anchors(src: DataSource) -> pd.DataFrame:
    """One row per line: revenue per period = subscriptions + total PAYG (BR-68),
    anchor = mean + std (BR-71, pandas ddof=1), data = 0.9 x anchor, voice = 0.1 x anchor."""
    sub = upper(src.read("subscriptions_tg")) if src.has("subscriptions_tg") else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    use = upper(src.read("usage_data_voice_payg_tg")) if src.has("usage_data_voice_payg_tg") else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    m_cols = [f"TOTAL_SUB_AMT_{m}" for m in MONTHS] + [f"TOTAL_SUB_AMT_{w}" for w in WEEKS]
    p_cols = [f"PG_TOTAL_AMOUNT_{p}" for p in MONTHS + WEEKS]
    sub = sub[["ACCESS_METHOD_VAL"] + [c for c in m_cols if c in sub.columns]].copy()
    use = use[["ACCESS_METHOD_VAL"] + [c for c in p_cols if c in use.columns]].copy()
    for df in (sub, use):
        for c in df.columns[1:]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    sub = sub.dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL").sum()
    use = use.dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL").sum()
    out = sub.join(use, how="outer").fillna(0.0)                                       # BR-76 missing = 0
    for c in m_cols + p_cols:
        if c not in out.columns:
            out[c] = 0.0
    for p in MONTHS + WEEKS:
        out[f"REV_{p}"] = out[f"TOTAL_SUB_AMT_{p}"] + out[f"PG_TOTAL_AMOUNT_{p}"]
    for name, periods in (("MONTHLY", MONTHS), ("WEEKLY", WEEKS)):
        cols = [f"REV_{p}" for p in periods]
        out[f"MAX_REV_{name}"] = out[cols].max(axis=1)                                  # BR-72 (not used)
        out[f"STD_REV_{name}"] = out[cols].std(axis=1, ddof=1).fillna(0.0)
        out[f"AVG_REV_{name}"] = out[cols].mean(axis=1) + out[f"STD_REV_{name}"]        # BR-71
        out[f"AVG_REV_DATA_{name}"] = config.ARPU_DATA_SHARE * out[f"AVG_REV_{name}"]   # BR-74
        out[f"AVG_REV_VOICE_{name}"] = config.ARPU_VOICE_SHARE * out[f"AVG_REV_{name}"]  # BR-75
    keep = [f"REV_{p}" for p in MONTHS + WEEKS] + [c for c in out.columns if c.startswith(("AVG_REV", "STD_REV", "MAX_REV"))]
    return out[keep]


def cap_anchor_growth(anchors: pd.DataFrame, previous: dict[str, dict], growth: float) -> pd.DataFrame:
    """M4 price-escalation guard: AVG_REV_* at most (1 + growth) x the anchor logged for the line last time
    (campaign purchases raise next week's spend, so anchors -- and prices -- could climb week after week).
    The data / voice anchors follow (BR-74/75). Lines never logged keep their anchor."""
    out = anchors.copy()
    out["ANCHOR_CAPPED"] = False
    for period, key in (("MONTHLY", "monthly"), ("WEEKLY", "weekly")):
        prev = pd.Series({v: a.get(key) for v, a in previous.items() if a and a.get(key) is not None}, dtype=float)
        prev = prev.reindex(out.index)
        cap = prev * (1.0 + growth)
        over = prev.notna() & (out[f"AVG_REV_{period}"] > cap)
        out.loc[over, f"AVG_REV_{period}"] = cap[over]
        out["ANCHOR_CAPPED"] |= over
        out[f"AVG_REV_DATA_{period}"] = config.ARPU_DATA_SHARE * out[f"AVG_REV_{period}"]
        out[f"AVG_REV_VOICE_{period}"] = config.ARPU_VOICE_SHARE * out[f"AVG_REV_{period}"]
    return out


def anchor_column(service: str, validity: int) -> str:
    """BR-77: COMBO vs total ARPU, DATA vs data ARPU, VOICE vs voice ARPU; weekly offers use weekly anchors.
    Services outside BR-77 (the roaming / international add-on families) use the total anchor, as the add-on
    engine does -- an unknown service must never crash an agent tool."""
    kind = ANCHOR_BY_SERVICE.get(str(service).upper(), "TOTAL")
    period = "WEEKLY" if int(validity) == 7 else "MONTHLY"
    return f"AVG_REV_{period}" if kind == "TOTAL" else f"AVG_REV_{kind}_{period}"


def anchor_value(anchors: pd.DataFrame, val: str, service: str, validity: int) -> float:
    if val not in anchors.index:
        return 0.0
    return float(anchors.at[val, anchor_column(service, validity)])


# ---------------------------------------------------------------------------
# Families (BR-66, BR-67) and price margins (BR-78)
# ---------------------------------------------------------------------------
def families(action: str | None, arm: str, mapping: str | None = None) -> list[tuple[str, str]]:
    """[(service, generosity) for type A, type B]. cstr: from the action; free: L2 COMBO + DATA."""
    if arm == "free":
        return [(s, config.FREE_ARM_GENEROSITY) for s in config.FREE_ARM_SERVICES]
    if action not in config.ACTION_FAMILIES:
        raise ValueError(f"Unknown action {action!r}")
    svc_a, svc_b = config.ACTION_FAMILIES[action]                 # CROSS SELL VOICE has its own entry (typo fixed)
    mapping = mapping or config.GENEROSITY_MAPPING
    if mapping == "all_L2":
        return [(svc_a, "L2"), (svc_b, "L2")]
    gen_a, gen_b = config.ACTION_GENEROSITY_NORMAL[action]
    return [(svc_a, gen_a), (svc_b, gen_b)]


def k_margin(action: str | None, arm: str) -> float:
    return config.PRICE_BAND_K["free" if arm == "free" else action]


def family_items(catalog: pd.DataFrame, fams: list[tuple[str, str]], validity: int) -> pd.DataFrame:
    mask = np.zeros(len(catalog), bool)
    for svc, gen in fams:
        mask |= (catalog["SERVICE"] == svc) & (catalog["GENEROSITY"] == gen)
    return catalog[mask & (catalog["VALIDITY"] == validity)]


# ---------------------------------------------------------------------------
# 3-pass selection (BR-78 .. BR-82)
# ---------------------------------------------------------------------------
@dataclass
class Pick:
    btl_name: str
    service: str
    generosity: str
    validity: int
    price: float
    score: float
    rank: int
    pass_no: int
    anchor: float
    anchor_type: str
    p_accept: float | None = None        # V1.0 value policy: acceptance model's P(accept); None for the price bands

    def as_dict(self) -> dict:
        return asdict(self)


def select_offers(cands: pd.DataFrame, anchors: pd.DataFrame, val: str, fams: list[tuple[str, str]], k: float,
                  validity: int, conflicts: set[frozenset], already: list[str] | None = None) -> list[Pick]:
    """At most one offer per family (so <= 2) for one customer and one validity.

    cands: ranker candidates for this customer (BTL_NAME, SERVICE, GENEROSITY, VALIDITY, PRICE, SCORE, RANK).
    pass 1: anchor < price < anchor + k, sorted score desc, price desc        (BR-78, BR-81 since 9 Oct)
    pass 2: the missing type only, price > anchor, no cap, score desc, price asc (BR-79)
    pass 3: no offer yet: both types, price > anchor, no cap, score desc, price asc (BR-80)
    A customer whose anchor is above every price gets nothing (BR-82)."""
    chosen: list[Pick] = []
    taken = list(already or [])
    c = cands[cands["VALIDITY"] == validity]
    # numpy arrays once per call: pick() runs up to 4 times per customer (review #15: pandas filtering per pick was
    # ~4 ms, i.e. hours for a 1.7 M base); same filters, same order (score desc, then price; ties keep input order)
    names, svcs, gens = c["BTL_NAME"].to_numpy(), c["SERVICE"].to_numpy(), c["GENEROSITY"].to_numpy()
    price, score, rank_ = c["PRICE"].to_numpy(float), c["SCORE"].to_numpy(float), c["RANK"].to_numpy()

    def pick(svc: str, gen: str, lo: float, hi: float | None, asc_price: bool, pass_no: int) -> Pick | None:
        m = (svcs == svc) & (gens == gen) & (price > lo)
        if hi is not None:
            m &= price < hi
        idx = np.flatnonzero(m)
        for j in idx[np.lexsort((price[idx] if asc_price else -price[idx], -score[idx]))]:
            if not conflicts_with(names[j], taken, conflicts):
                return Pick(names[j], svc, gen, int(validity), float(price[j]), float(score[j]), int(rank_[j]), pass_no,
                            round(lo, 6), anchor_column(svc, validity))   # 6 dp: 12.9995 must not print as 13.0
        return None

    anchors_f = [anchor_value(anchors, val, svc, validity) for svc, _ in fams]
    first = []
    for (svc, gen), a in zip(fams, anchors_f):          # sequential: the 2nd pick must see the 1st (conflicts)
        p = pick(svc, gen, a, a + k, False, 1)
        first.append(p)
        if p:
            chosen.append(p)
            taken.append(p.btl_name)
    found = [p is not None for p in first]
    if any(found) and not all(found):                       # pass 2: missing type only
        i = found.index(False)
        svc, gen = fams[i]
        p = pick(svc, gen, anchors_f[i], None, True, 2)
        if p:
            chosen.append(p)
            taken.append(p.btl_name)
    elif not any(found):                                    # pass 3: nothing yet
        for (svc, gen), a in zip(fams, anchors_f):
            p = pick(svc, gen, a, None, True, 3)
            if p:
                chosen.append(p)
                taken.append(p.btl_name)
    return chosen[: config.MAX_OFFERS_PER_VALIDITY]


# ---------------------------------------------------------------------------
# Hard rules (code-enforced; the LLM cannot override them)
# ---------------------------------------------------------------------------
def check_offers(names: list[str], catalog: pd.DataFrame, conflicts: set[frozenset]) -> list[str]:
    """Violations for a proposed list of offers (empty list = all hard rules pass)."""
    v: list[str] = []
    cat = catalog.set_index("BTL_NAME")
    if len(names) != len(set(names)):
        v.append("duplicate offer in the list")
    known = [n for n in names if n in cat.index]
    for n in names:
        if n not in cat.index:
            v.append(f"{n}: not in the catalogue (BR-63)")
        elif not cat.at[n, "PRICE"] > 0:
            v.append(f"{n}: price must be > 0")
    if len(names) > config.MAX_OFFERS_TOTAL:
        v.append(f"{len(names)} offers: at most {config.MAX_OFFERS_TOTAL} in total (BR-82)")
    for validity in config.VALIDITIES:
        k = sum(1 for n in known if int(cat.at[n, "VALIDITY"]) == validity)
        if k > config.MAX_OFFERS_PER_VALIDITY:
            v.append(f"{k} offers with validity {validity}: at most {config.MAX_OFFERS_PER_VALIDITY} (BR-82)")
    for i, a in enumerate(known):
        for b in known[i + 1:]:
            if frozenset((a, b)) in conflicts:
                v.append(f"{a} and {b} conflict (conflicting_offers_h)")
    return v


def customer_eligibility(base_row: dict | None, segment: str | None = None) -> list[str]:
    """Reasons a line may not be contacted (BR-02, BR-10, BR-36); empty = eligible."""
    if base_row is None:
        return ["not in the eligible prepaid base"]
    reasons = []
    val = str(base_row.get("ACCESS_METHOD_VAL") or "")
    if not (len(val) == 8 and val.isdigit() and val > config.MSISDN_MIN_EXCLUSIVE):
        reasons.append("MSISDN outside the prepaid range (BR-02)")
    if base_row.get("CURRENT_PLAN") in config.EXCLUDED_PLANS:
        reasons.append("internal / employee plan (BR-10)")
    if segment and segment.strip().lower() in config.EXCLUDED_SEGMENTS:
        reasons.append("Nursering segment, tenure < 45 days (BR-36)")
    return reasons


In [ ]:
%%engine nbx.fsdr.pipelines package
"""Deterministic FSDR pipelines (train, score, recommend). Each step returns a JSON report,
so the agents can call the steps as tools."""
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

OUTPUT_DIR = "outputs"


def outputs_dir(data_dir: str | Path) -> Path:
    p = Path(data_dir) / OUTPUT_DIR
    p.mkdir(parents=True, exist_ok=True)
    return p


def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA or v is pd.NaT:
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return round(float(v), 6)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, date):
        return v.isoformat()
    return v


def write_table(df: pd.DataFrame, data_dir: str | Path, name: str) -> Path:
    """Writes an output table as a JSON array of records (lower-case columns, as the production outputs)."""
    path = outputs_dir(data_dir) / f"{name}.json"
    cols = [str(c).lower() for c in df.columns]
    values = [_column_values(df.iloc[:, i]) for i in range(df.shape[1])]
    records = [dict(zip(cols, row)) for row in zip(*values)] if cols else [{} for _ in range(len(df))]
    path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
    return path


def _column_values(s: pd.Series) -> list:
    """The JSON values of one column, as the per-value _clean of df.to_dict("records") gave them: NaN -> None,
    numpy numbers -> Python numbers (unrounded: to_dict already boxes them), dates -> ISO strings. Number and
    string columns are converted in one go (review #15: _clean ran once per value, 5 M calls at 100 k lines)."""
    k = s.dtype.kind
    if k in "iub" and not pd.api.types.is_extension_array_dtype(s.dtype):
        return s.tolist()
    if k == "f" and not pd.api.types.is_extension_array_dtype(s.dtype):
        return [None if v != v else v for v in s.tolist()]
    if s.dtype == object and pd.api.types.infer_dtype(s, skipna=True) in ("string", "empty"):
        return [v if isinstance(v, str) else None for v in s.tolist()]
    return [_clean(v) for v in s.to_frame().to_dict("list")[s.name if s.name is not None else 0]]


def read_table(data_dir: str | Path, name: str) -> pd.DataFrame:
    path = Path(data_dir) / OUTPUT_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found")
    df = pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))
    df.columns = [c.upper() for c in df.columns]
    return df


def write_report(report: dict, data_dir: str | Path, name: str) -> Path:
    path = outputs_dir(data_dir) / f"{name}.json"
    path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=_clean), encoding="utf-8")
    return path


In [ ]:
%%engine nbx.fsdr.segmentation package
"""② Segmentation (master prompt §7, §8) and ②b segment actions (§7.1): S-rules, bands, micro-segments,
action_original (code 1's dictionary) and action_agent (the agent dictionary). All vectorised, no LLM here."""


In [ ]:
%%engine nbx.fsdr.segmentation.actions
"""②b The two actions of every line (master prompt §7.1), plus the week-0 history of code 1 (B-02, B-03, B-09, B-10).

action_original   code 1's dict_segm_actions (case-sensitive, last value wins) applied to the line's segment:
                  the segment-engine segment of targeting_dataiku_auto when the line has one this week, otherwise
                  the S-rule segment translated by the crosswalk (config.SEGMENT_CROSSWALK). Deterministic, no LLM.
action_agent      the agent dictionary dict_segm_actions_agent_vN: per S-segment x nationality group, ranked actions.
                  Week 0 it is a copy of the original dictionary (v0); the LLM changes cells from Phase 4 on, and
                  every version is validated here before it is applied (invalid -> the previous cell is kept).
"""
from __future__ import annotations

import json
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.features import norm_id, norm_val

ENGINE, CROSSWALK = "segment_engine", "crosswalk"
AT_RISK_SEGMENTS = {s for _, s, _ in config.S_RULES if s.endswith("_At_Risk") and "Not_At_Risk" not in s}
# H-06 for the dictionary: an at-risk segment may not be moved TO an upsell action. Its original action is kept
# allowed (Onboarding_Early_Churn_At_Risk is UPSELL in code 1). CHECK item (plan §11).
UPSELL_ACTIONS = {"UPSELL", "UPSELL DATA", "UPSELL VOICE"}


def is_nursering(segment: pd.Series) -> pd.Series:
    return segment.astype("string").str.lower().str.startswith("nursering").fillna(False)


# ---------------------------------------------------------------------------
# Crosswalk and action_original
# ---------------------------------------------------------------------------
def crosswalk_table() -> pd.DataFrame:
    """One row per S-rule: the original segment name it maps to, both actions, and the CHECK note."""
    rows = []
    for rid, seg, rule_action in config.S_RULES:
        orig, note = config.SEGMENT_CROSSWALK[seg]
        orig_action = config.DICT_SEGM_ACTIONS.get(orig)
        rows.append({"rule_id": rid, "s_segment": seg, "original_segment": orig, "rule_action": rule_action or "excluded",
                     "original_action": orig_action or "excluded", "same_action": (rule_action == orig_action),
                     "check": "CHECK", "note": note})
    return pd.DataFrame(rows)


def week0_targeting(tg: pd.DataFrame | None, fulfil: pd.DataFrame | None, catalog: pd.DataFrame | None,
                    run_date: date, lines: pd.DataFrame) -> pd.DataFrame:
    """Per line in targeting_dataiku_auto: the latest segment (W0_SEGMENT), its original action (B-03; rows without
    a segment or in Nursering get none, B-02) and the B-09 reward: a catalogue BTL fulfilled in D-7 … D-1.
    lines: ACCESS_METHOD_ID -> ACCESS_METHOD_VAL of the base (targeting carries the id, fulfilments the MSISDN)."""
    cols = ["ACCESS_METHOD_ID", "W0_SEGMENT", "W0_DATE", "W0_ACTION", "W0_REWARD", "W0_BTL_REVENUE"]
    if tg is None or not len(tg):
        return pd.DataFrame(columns=cols)
    t = pd.DataFrame({"ACCESS_METHOD_ID": norm_id(tg["ACCESS_METHOD_ID"]),
                      "W0_SEGMENT": tg["SEGMENT"].astype("string").str.strip().astype(object),
                      "W0_DATE": pd.to_datetime(tg["DATE1"], errors="coerce")})
    t = t.dropna(subset=["ACCESS_METHOD_ID"]).sort_values("W0_DATE", kind="stable").drop_duplicates("ACCESS_METHOD_ID", keep="last")
    t["W0_SEGMENT"] = t["W0_SEGMENT"].where(t["W0_SEGMENT"].notna(), None)
    t["W0_ACTION"] = t["W0_SEGMENT"].map(lambda s: config.DICT_SEGM_ACTIONS.get(s) if isinstance(s, str) else None)
    t = t.merge(lines[["ACCESS_METHOD_ID", "ACCESS_METHOD_VAL"]].drop_duplicates("ACCESS_METHOD_ID"),
                on="ACCESS_METHOD_ID", how="left")
    t["W0_REWARD"] = 0
    t["W0_BTL_REVENUE"] = 0.0
    return t[cols].reset_index(drop=True) if fulfil is None else _rewards(t, fulfil, catalog, run_date)[cols]


def _rewards(t: pd.DataFrame, fulfil: pd.DataFrame, catalog: pd.DataFrame | None, run_date: date) -> pd.DataFrame:
    from nbx.fsdr import calendar as cal

    w = cal.campaign_windows(run_date)["reward"]
    f = pd.DataFrame({"ACCESS_METHOD_VAL": norm_val(fulfil["ACCESS_METHOD_VAL"]),
                      "PRODUCTNAME": fulfil["PRODUCTNAME"].astype(str).str.strip(),
                      "DAY": pd.to_datetime(fulfil["FULFILMENTSUCCESSDATE_TIME"], errors="coerce").dt.date})
    f = f.dropna(subset=["ACCESS_METHOD_VAL", "DAY"])
    f = f[(f["DAY"] >= w.start) & (f["DAY"] <= w.end)]
    if catalog is not None:
        price = catalog.set_index("BTL_NAME")["PRICE"]
        f = f[f["PRODUCTNAME"].isin(price.index)]
        f = f.assign(PRICE=f["PRODUCTNAME"].map(price).astype("float64").fillna(0.0))
    else:
        f = f.assign(PRICE=0.0)
    by_line = f.groupby("ACCESS_METHOD_VAL")["PRICE"].agg(["size", "sum"])
    t = t.reset_index(drop=True)
    vals = t["ACCESS_METHOD_VAL"]
    t["W0_REWARD"] = vals.map(by_line["size"]).fillna(0).gt(0).astype("int8")
    t["W0_BTL_REVENUE"] = vals.map(by_line["sum"]).fillna(0.0).round(3)
    return t


def apply_original(seg: pd.DataFrame) -> pd.DataFrame:
    """ORIGINAL_SEGMENT, ACTION_ORIGINAL, ACTION_ORIGINAL_SOURCE from W0_SEGMENT (engine) or the crosswalk.
    An engine segment that the dictionary does not know (spelling) falls back to the crosswalk; an engine Nursering
    segment is kept (the line gets no original action, as in code 1)."""
    w0 = seg["W0_SEGMENT"] if "W0_SEGMENT" in seg.columns else pd.Series(None, index=seg.index, dtype=object)
    known = w0.map(lambda s: isinstance(s, str) and s in config.DICT_SEGM_ACTIONS) | is_nursering(w0)
    cross = seg["SEGMENT"].map(lambda s: config.SEGMENT_CROSSWALK[s][0])
    seg["ORIGINAL_SEGMENT"] = np.where(known, w0, cross).astype(object)
    seg["ACTION_ORIGINAL"] = seg["ORIGINAL_SEGMENT"].map(config.DICT_SEGM_ACTIONS).astype(object)
    seg["ACTION_ORIGINAL"] = seg["ACTION_ORIGINAL"].where(seg["ACTION_ORIGINAL"].notna(), None)
    seg["ACTION_ORIGINAL_SOURCE"] = np.where(known, ENGINE, CROSSWALK).astype(object)
    return seg


# ---------------------------------------------------------------------------
# Agent dictionary (dict_segm_actions_agent_vN)
# ---------------------------------------------------------------------------
def initial_agent_dictionary(run_date: date) -> dict:
    """v0 = the original dictionary through the crosswalk, for every S-segment x nationality group (70 cells)."""
    cells = []
    for _, seg, _ in config.S_RULES:
        orig = config.DICT_SEGM_ACTIONS.get(config.SEGMENT_CROSSWALK[seg][0])
        for nat in config.NATIONALITY_GROUPS:
            cells.append({"segment": seg, "nationality_group": nat, "excluded": orig is None,
                          "actions": [] if orig is None else
                          [{"action": orig, "rank": 1, "score": 1.0, "evidence_ids": ["B-03"],
                            "reason": "week 0: copy of the original dictionary"}]})
    return {"version": 0, "name": f"{config.AGENT_DICTIONARY_PREFIX}0", "created": run_date.isoformat(),
            "parent": None, "made_by": "code (copy of dict_segm_actions)", "cells": cells}


def validate_agent_dictionary(d: dict, previous: dict | None = None) -> tuple[dict, list[dict]]:
    """⑥ check of every cell: known segment and nationality, allowed actions only, excluded segments stay excluded,
    no aggressive upsell in an at-risk segment (H-06). An invalid cell is replaced by the previous version's cell
    (or v0's). Returns the valid dictionary and the problems found."""
    base = {(c["segment"], c["nationality_group"]): c
            for c in (previous or initial_agent_dictionary(date.fromisoformat(d.get("created", "2000-01-01"))))["cells"]}
    problems, cells, seen = [], [], set()
    for c in d.get("cells", []):
        key = (c.get("segment"), c.get("nationality_group"))
        why = []
        if key[0] not in config.S_SEGMENTS or key[1] not in config.NATIONALITY_GROUPS:
            why.append("unknown segment or nationality group")
        acts = [a.get("action") for a in c.get("actions", [])]
        if any(a not in config.ACTIONS for a in acts):
            why.append(f"action not allowed: {[a for a in acts if a not in config.ACTIONS]}")
        original = config.DICT_SEGM_ACTIONS.get(config.SEGMENT_CROSSWALK.get(key[0], (None,))[0])
        excluded = original is None
        if excluded and acts:
            why.append("excluded segment got an action (B-02)")
        if not excluded and not acts:
            why.append("no action for a targetable segment")
        if key[0] in AT_RISK_SEGMENTS and acts[:1] and acts[0] in UPSELL_ACTIONS and acts[0] != original:
            why.append("upsell as top action of an at-risk segment (H-06)")
        if len(acts) != len(set(acts)) or len(acts) > 3:
            why.append("ranked actions must be 1-3 distinct ids")
        if why:
            problems.append({"segment": key[0], "nationality_group": key[1], "problems": why})
            if key in base and key not in seen:
                cells.append(base[key])
                seen.add(key)
        elif key not in seen:
            cells.append(c)
            seen.add(key)
    for key, c in base.items():                                   # a missing cell keeps its previous value
        if key not in seen:
            cells.append(c)
            problems.append({"segment": key[0], "nationality_group": key[1], "problems": ["missing cell (kept previous)"]})
    return {**d, "cells": cells}, problems


def agent_lookup(d: dict) -> pd.DataFrame:
    rows = [{"SEGMENT": c["segment"], "NATIONALITY_GROUP": c["nationality_group"],
             "ACTION_AGENT": (c["actions"][0]["action"] if c["actions"] else None),
             "ACTION_AGENT_RANK2": (c["actions"][1]["action"] if len(c["actions"]) > 1 else None)} for c in d["cells"]]
    return pd.DataFrame(rows)


def apply_agent_dictionary(seg: pd.DataFrame, d: dict) -> pd.DataFrame:
    """ACTION_AGENT, ACTION_AGENT_RANK2, ACTION_AGENT_VERSION of every line, by (S-segment, nationality group)."""
    lk = agent_lookup(d)
    out = seg[["SEGMENT", "NATIONALITY_GROUP"]].merge(lk, on=["SEGMENT", "NATIONALITY_GROUP"], how="left")
    seg["ACTION_AGENT"] = out["ACTION_AGENT"].where(out["ACTION_AGENT"].notna(), None).to_numpy()
    seg["ACTION_AGENT_RANK2"] = out["ACTION_AGENT_RANK2"].where(out["ACTION_AGENT_RANK2"].notna(), None).to_numpy()
    seg["ACTION_AGENT_VERSION"] = d["name"]
    return seg


def dictionary_files(step_dir: Path) -> list[Path]:
    rx = re.compile(re.escape(config.AGENT_DICTIONARY_PREFIX) + r"(\d+)\.json$")
    files = [p for p in Path(step_dir).glob(f"{config.AGENT_DICTIONARY_PREFIX}*.json") if rx.search(p.name)]
    return sorted(files, key=lambda p: int(rx.search(p.name).group(1)))


def previous_agent_dictionary(runs_root: Path, run_date: date) -> tuple[dict | None, Path | None]:
    """The latest agent dictionary of an EARLIER run date (data/runs/<D'>/train/T3_segments/), if any."""
    root = Path(runs_root)
    dates = sorted(p.name for p in root.glob("20??-??-??") if p.is_dir() and p.name < run_date.isoformat())
    for d in reversed(dates):
        files = dictionary_files(root / d / "train" / "T3_segments")
        if files:
            return json.loads(files[-1].read_text(encoding="utf-8")), files[-1]
    return None, None


def action_agreement(original: pd.Series, agent: pd.Series, sent: pd.Series) -> np.ndarray:
    """§7.1 log field: all_same / sent_original / sent_agent / other."""
    o, a, s = (x.astype("string") for x in (original, agent, sent))
    so, sa = (s == o).fillna(False).to_numpy(bool), (s == a).fillna(False).to_numpy(bool)
    return np.select([so & sa, so, sa], ["all_same", "sent_original", "sent_agent"], "other").astype(object)


In [ ]:
%%engine nbx.fsdr.segmentation.bands
"""Bands and micro-segments (master prompt §8). Micro-segment = nationality group x age band x tenure band
(5 x 4 x 6 = 120 cells); value band and usage type are context in customer_360, not splits."""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd

from nbx.fsdr import config


def _bands(x: pd.Series, bands: list[tuple[str, float, float]], valid: pd.Series) -> np.ndarray:
    v = pd.to_numeric(x, errors="coerce")
    conds = [(valid & (v >= lo) & (v < hi)).fillna(False).to_numpy(dtype=bool) for _, lo, hi in bands]
    return np.select(conds, [b for b, _, _ in bands], "unknown").astype(object)


def age_band(age: pd.Series) -> np.ndarray:
    """young < 40 (incl. < 20), middle 40-59, senior 60+, unknown = missing or outside 15-100."""
    a = pd.to_numeric(age, errors="coerce")
    return _bands(a, config.AGE_BANDS, (a >= config.AGE_MIN) & (a <= config.AGE_MAX))


def young_split(age: pd.Series) -> np.ndarray:
    """Open item 5: young under 25 / 25-39 (tested in Phase 4, not a split yet)."""
    a = pd.to_numeric(age, errors="coerce")
    young = (a >= config.AGE_MIN) & (a < config.AGE_BANDS[0][2])
    return np.where(young & (a < config.YOUNG_SPLIT_AGE), "lt25", np.where(young, "25_39", None)).astype(object)


def tenure_band(days: pd.Series) -> np.ndarray:
    """t0_lt3m < 90 d, t1_3_6m, t2_6_12m, t3_1_2y, t4_gt2y > 730 d, unknown = missing or negative."""
    d = pd.to_numeric(days, errors="coerce")
    return _bands(d, config.TENURE_BANDS, d >= 0)


def micro_segment(nat: pd.Series, age_b, tenure_b) -> pd.Series:
    return (pd.Series(nat, dtype="string").fillna(config.NATIONALITY_OTHERS) + "|" + pd.Series(age_b, index=nat.index, dtype="string")
            + "|" + pd.Series(tenure_b, index=nat.index, dtype="string")).astype(object)


def all_micro_segments() -> list[tuple[str, str, str]]:
    return list(itertools.product(config.NATIONALITY_GROUPS, config.AGE_BAND_NAMES, config.TENURE_BAND_NAMES))


def value_band(arpu: pd.Series, cutoffs: list[float]) -> np.ndarray:
    """zero (ARPU_M1 = 0), then low / mid / high = tertiles of the paying lines over the whole base."""
    a = pd.to_numeric(arpu, errors="coerce").fillna(0.0)
    lo, hi = cutoffs
    return np.select([a <= 0, a <= lo, a <= hi], ["zero", "low", "mid"], "high").astype(object)


def usage_type(voice_user: pd.Series, data_user: pd.Series) -> np.ndarray:
    v, d = voice_user.astype(bool), data_user.astype(bool)
    return np.select([v & d, d, v], ["dual", "data_only", "voice_only"], "none").astype(object)


In [ ]:
%%engine nbx.fsdr.segmentation.rules
"""S-rules (master prompt §7): the current segment of every line, first match wins, thresholds in config.

    seg = apply_s_rules(features)      # adds RULE_ID, SEGMENT, RULE_ACTION, EXCLUDED and the trigger values

B-05: a missing score never triggers "at risk" (nor reconnect, fragility or appetency): the triggers read a score
only where its *_MISSING flag is 0, so the imputed median cannot move a line into a risk segment.
An unknown tenure is not "new": it never matches S-1 … S-4.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from nbx.fsdr import config

TRIGGERS = ["S_REV_M1", "S_RGS", "S_VOICE_USER", "S_DATA_USER", "S_AT_RISK", "S_RECONNECT_P", "S_FRAGILITY",
            "S_REV_M1_M2_PCT"]


def _observed(df: pd.DataFrame, feature: str) -> pd.Series:
    """The score where it was observed, 0 where it was missing (or absent)."""
    if feature not in df.columns:
        return pd.Series(0.0, index=df.index)
    s = pd.to_numeric(df[feature], errors="coerce").fillna(0.0)
    flag = f"{feature}{config.MISSING_FLAG_SUFFIX}"
    return s.where(df[flag] == 0, 0.0) if flag in df.columns else s


def _col(df: pd.DataFrame, name: str) -> pd.Series:
    return pd.to_numeric(df[name], errors="coerce").fillna(0.0) if name in df.columns else pd.Series(0.0, index=df.index)


def apply_s_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a frame aligned on df with the segment columns and the values that triggered them."""
    c = config
    tenure = pd.to_numeric(df["TENURE_DAYS"], errors="coerce")
    churn, churn_1d = _observed(df, "PROBA_CHURN"), _observed(df, "PROBA_CHURN_1D")
    rev_m1 = _col(df, "ARPU_M1")
    rgs = (rev_m1 > 0) & (_col(df, "ACTIVE_DAYS_COUNT_M1") > 0)
    voice = _col(df, "LOCAL_VOICE_MIN_M1") >= c.S_VOICE_USER_MIN
    data = _col(df, "DATA_VOLUME_MB_M1") >= c.S_DATA_USER_MB
    reconnect = np.maximum(_observed(df, "PROBA_REACT_30"), _observed(df, "PROBA_REACT_15"))
    fragility = np.maximum.reduce([_observed(df, "PROBA_FRAG_DATA"), _observed(df, "PROBA_FRAG_VOICE"),
                                   _observed(df, "PROBA_REV_DECR")])
    rev_change = _col(df, "VAR_REV_M1_M2")
    at_risk = churn >= c.S_CHURN_AT_RISK
    new_45 = (tenure < c.S_TENURE_NURSERING).fillna(False)
    new_90 = (tenure < c.S_TENURE_ONBOARDING).fillna(False)
    conds = [
        new_45 & at_risk, new_45,                                                       # S-1, S-2
        new_90 & at_risk, new_90,                                                       # S-3, S-4
        ~rgs & (reconnect >= c.S_RECONNECT_HIGH), ~rgs & (reconnect >= c.S_RECONNECT_MED), ~rgs,   # S-5 … S-7
        at_risk | (churn_1d >= c.S_CHURN_1D_AT_RISK),                                   # S-8 (RGS from here)
        (fragility >= c.S_FRAGILITY) | (rev_change <= c.S_REV_DROP_PCT),                # S-9
        voice & data,                                                                   # S-10
        data & (_observed(df, "PROBA_APP_VOICE") >= c.S_APPETENCY), data,               # S-11, S-12
        _observed(df, "PROBA_APP_DATA") >= c.S_APPETENCY,                               # S-13 (voice only or none)
    ]
    idx = np.select([np.asarray(x, dtype=bool) for x in conds], list(range(len(conds))), default=len(conds))  # S-14
    rules = c.S_RULES
    out = pd.DataFrame(index=df.index)
    out["RULE_ID"] = np.array([r[0] for r in rules], dtype=object)[idx]
    out["SEGMENT"] = np.array([r[1] for r in rules], dtype=object)[idx]
    out["RULE_ACTION"] = np.array([r[2] for r in rules], dtype=object)[idx]
    out["EXCLUDED"] = out["RULE_ACTION"].isna().astype("int8")
    out["SEGMENT_SOURCE"] = "s_rules"
    out["S_REV_M1"] = rev_m1.round(4)
    out["S_RGS"] = rgs.astype("int8")
    out["S_VOICE_USER"] = voice.astype("int8")
    out["S_DATA_USER"] = data.astype("int8")
    out["S_AT_RISK"] = at_risk.astype("int8")
    out["S_RECONNECT_P"] = np.round(reconnect, 6)
    out["S_FRAGILITY"] = np.round(fragility, 6)
    out["S_REV_M1_M2_PCT"] = rev_change.round(4)
    return out


def segment_family(segment: pd.Series) -> pd.Series:
    """Nursering / Onboarding / Retention / RGS (for the tenure-agreement report)."""
    s = segment.astype("string")
    return pd.Series(np.select([s.str.startswith("Nursering"), s.str.startswith("Onboarding"),
                                s.str.startswith("Retention")], ["Nursering", "Onboarding", "Retention"], "RGS"),
                     index=segment.index)


In [ ]:
%%engine nbx.fsdr.pipelines.training
"""V2.0 training pipeline, stages T2 and T3 (plan §14, §15; master prompt §6 - §8, §18).

    run_t2(D)        T2_features   base_features/part-NN.parquet (B-01 … B-12) for the full base
    run_t3(D)        T3_segments   segments/part-NN.parquet: S-rule segment, bands, micro-segment, week-0 history,
                                   action_original + crosswalk, action_agent (agent dictionary), and the reports
    run_features(D)  T2 then T3 (fsdr-features)

Both stages read T0's Parquet cache and run partition by partition, `workers` partitions in parallel
(FSDR_FEATURE_WORKERS: 2 on the dev laptop, 8 on the P16). Each worker writes its own partition file and returns
small count tables, which the parent sums: nothing of size 1M is ever gathered in one process except the few
columns T2 needs for the B-05 medians and the ARPU tertiles.

Resume: a stage whose _step.json is done with the same inputs (the cache manifest, the previous stage, the rule
constants) is skipped; force=True re-runs it.
"""
from __future__ import annotations

import hashlib
import json
from concurrent.futures import ProcessPoolExecutor
from datetime import date
from pathlib import Path

import pandas as pd

from nbx.fsdr import base_features as bf
from nbx.fsdr import config
from nbx.fsdr.contract import data_agent
from nbx.fsdr.contract.cache import CACHE_MANIFEST, CacheSource, cache_dir
from nbx.fsdr.offer_rules import load_catalog
from nbx.fsdr.segmentation import actions, bands, rules
from nbx.fsdr.steps import Step, fingerprint, read_step, runs_dir, sha256_file

T2, T3 = "T2_features", "T3_segments"
SEGMENT_COLUMNS = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "RULE_ID", "SEGMENT", "RULE_ACTION", "EXCLUDED",
                   "SEGMENT_SOURCE", *rules.TRIGGERS, "NATIONALITY_GROUP", "AGE", "AGE_BAND", "YOUNG_SPLIT",
                   "TENURE_DAYS", "TENURE_BAND", "MICRO_SEGMENT", "ARPU_M1", "VALUE_BAND", "USAGE_TYPE", "REGION",
                   "DV_SEGMENT", "W0_SEGMENT", "W0_DATE", "W0_TARGETED", "W0_ACTION", "W0_REWARD", "W0_BTL_REVENUE",
                   "ORIGINAL_SEGMENT", "ACTION_ORIGINAL", "ACTION_ORIGINAL_SOURCE", "ACTION_AGENT",
                   "ACTION_AGENT_RANK2", "ACTION_AGENT_VERSION"]


class StageBlocked(RuntimeError):
    """A stage cannot start (its previous stage is missing or blocked)."""


def _iso(d: date | str) -> str:
    return d.isoformat() if isinstance(d, date) else str(d)


def _run(fn, jobs: list, workers: int) -> list:
    if workers <= 1 or len(jobs) <= 1:
        return [fn(j) for j in jobs]
    with ProcessPoolExecutor(max_workers=min(workers, len(jobs))) as ex:
        return list(ex.map(fn, jobs))


def rules_version() -> str:
    """Hash of every constant the S-rules, bands and dictionaries depend on (a change re-runs T3)."""
    keys = [k for k in dir(config) if k.startswith(("S_", "AGE_", "TENURE_", "YOUNG_", "VALUE_", "USAGE_", "NATIONALITY_"))]
    keys += ["DICT_SEGM_ACTIONS_SOURCE", "SEGMENT_CROSSWALK", "ACTIONS"]
    blob = json.dumps({k: getattr(config, k) for k in sorted(keys)}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]


def _t0(run_date: date, runs_root: Path | None) -> dict:
    t0 = data_agent.load_summary(run_date, runs_root)
    if not t0:
        raise StageBlocked(f"no T0_load for {run_date}: run fsdr-load first")
    if t0.get("status") != "done":
        raise StageBlocked(f"T0_load of {run_date} is {t0.get('status')}: fix the data, re-run fsdr-load")
    return t0


def _sum_counts(frames: list[pd.DataFrame], by: list[str], value: list[str] | None = None) -> pd.DataFrame:
    frames = [f for f in frames if f is not None and len(f)]
    if not frames:
        return pd.DataFrame(columns=by + (value or ["lines"]))
    return pd.concat(frames, ignore_index=True).groupby(by, dropna=False, as_index=False)[value or ["lines"]].sum()


# ---------------------------------------------------------------------------
# T2 features
# ---------------------------------------------------------------------------
def _t2_pass1(job: tuple) -> dict:
    cache_root, run_date, part, anchor, out = job
    cache = CacheSource(run_date, Path(cache_root))
    df, stats = bf.partition_features(cache, part, date.fromisoformat(run_date), anchor)
    df.to_parquet(out, index=False)
    return stats


def _t2_pass2(job: tuple) -> pd.DataFrame:
    path, medians = job
    df = pd.read_parquet(path)
    bf.impute(df, medians)
    df.to_parquet(path, index=False)
    return bf.column_stats(df)


def run_t2(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None, workers: int | None = None,
           force: bool = False) -> dict:
    t0 = _t0(run_date, runs_root)
    anchor = bf.anchor_of(t0["pipeline"])
    cache = CacheSource(run_date, cache_root)
    workers = int(workers or config.FEATURE_WORKERS)
    inputs = fingerprint({"cache": sha256_file(cache_dir(run_date, cache_root) / CACHE_MANIFEST), "anchor": anchor,
                          "registry": [m.table for m in config.MODEL_SCORE_REGISTRY], "trend": config.TREND_PAIRS})
    with Step(runs_dir(run_date, runs_root) / "train", T2, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        parts = list(range(cache.partitions))
        outs = [st.output(f"base_features/part-{k:02d}.parquet") for k in parts]
        stats = _run(_t2_pass1, [(str(cache_dir(run_date, cache_root).parent), _iso(run_date), k, anchor, str(outs[k]))
                                 for k in parts], workers)
        medians = bf.score_medians(outs)                                       # B-05, whole base
        cutoffs = bf.arpu_cutoffs(outs)
        col_stats = _run(_t2_pass2, [(str(o), medians) for o in outs], workers)

        cs = pd.concat(col_stats).groupby(level=0).agg({"count": "sum", "nulls": "sum", "sum": "sum", "min": "min", "max": "max"})
        cs["mean"] = (cs["sum"] / cs["count"].where(cs["count"] > 0)).round(6)
        st.write_csv(cs.drop(columns=["sum"]).reset_index(names="feature"), "feature_summary.csv")
        st.write_csv(pd.read_parquet(outs[0]).head(config.STEP_PREVIEW_ROWS), "base_features_preview.csv")
        vals = pd.concat([pd.read_parquet(o, columns=["ACCESS_METHOD_VAL"]) for o in outs])["ACCESS_METHOD_VAL"]
        lines = int(sum(s["lines"] for s in stats))
        missing = {f: int(sum(s["missing_scores"][f] for s in stats)) for f in config.proba_features()}
        src_counts = {k: _merge_counts([s[k] for s in stats]) for k in ("age_source", "tenure_source")}
        stats_doc = {"medians_by_nationality": medians, "arpu_tertiles": cutoffs, "anchor": anchor,
                     "party_table": bf.party_table(anchor, cache)}
        st.write_json("stats.json", stats_doc)
        summary = {"run_date": _iso(run_date), "stage": T2, "pipeline": t0["pipeline"], "anchor": anchor,
                   "tables": "*_auto (week 0)" if anchor == "T" else "*_score (week n)",
                   "base_rows": int(sum(s["base_rows"] for s in stats)), "lines": lines,
                   "eligible_filters": {k: int(sum(s[k] for s in stats)) for k in
                                        ("null_keys", "excluded_plan", "bad_msisdn", "duplicates_removed")},
                   "msisdn_in_two_partitions": int(vals.duplicated().sum()),
                   "features": int(len(cs)), "missing_scores": missing,
                   "missing_share": {f: round(n / lines, 4) if lines else 0.0 for f, n in missing.items()},
                   **src_counts, "no_nationality": int(sum(s["no_nationality"] for s in stats)),
                   "arpu_tertiles": cutoffs, "partitions": len(parts), "workers": workers, "step_dir": str(st.dir)}
        st.rows(rows_in=summary["base_rows"], rows_out=lines)
        st.note(anchor=anchor, workers=workers)
        st.write_json("summary.json", summary)
        return summary


def _merge_counts(dicts: list[dict]) -> dict:
    out: dict = {}
    for d in dicts:
        for k, v in d.items():
            out[k] = out.get(k, 0) + int(v)
    return out


# ---------------------------------------------------------------------------
# T3 segments and the two actions
# ---------------------------------------------------------------------------
def segment_partition(feat: pd.DataFrame, cache, part: int, run_date: date, cutoffs: list[float],
                      agent_dict: dict, catalog: pd.DataFrame | None) -> pd.DataFrame:
    """Every T3 column of one partition (pure function of the features, the cache partition and the dictionary)."""
    seg = pd.concat([feat[["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "NATIONALITY_GROUP", "AGE", "TENURE_DAYS",
                           "ARPU_M1", "REGION", "DV_SEGMENT"]], rules.apply_s_rules(feat)], axis=1)
    seg["AGE_BAND"] = bands.age_band(seg["AGE"])
    seg["YOUNG_SPLIT"] = bands.young_split(seg["AGE"])
    seg["TENURE_BAND"] = bands.tenure_band(seg["TENURE_DAYS"])
    seg["MICRO_SEGMENT"] = bands.micro_segment(seg["NATIONALITY_GROUP"], seg["AGE_BAND"], seg["TENURE_BAND"])
    seg["VALUE_BAND"] = bands.value_band(seg["ARPU_M1"], cutoffs)
    seg["USAGE_TYPE"] = bands.usage_type(seg["S_VOICE_USER"], seg["S_DATA_USER"])

    tg = bf._part(cache, "targeting_dataiku_auto", part)
    ful = bf._part(cache, "subs_btl_auto", part)
    w0 = actions.week0_targeting(tg, ful, catalog, run_date, seg)
    seg = seg.merge(w0.drop(columns=["ACCESS_METHOD_VAL"], errors="ignore"), on="ACCESS_METHOD_ID", how="left")
    seg["W0_TARGETED"] = seg["W0_DATE"].notna().astype("int8")
    # numeric first: without targeting_dataiku_auto (a weekly run) the merged columns are empty object columns
    seg["W0_REWARD"] = pd.to_numeric(seg["W0_REWARD"], errors="coerce").fillna(0).astype("int8")
    seg["W0_BTL_REVENUE"] = pd.to_numeric(seg["W0_BTL_REVENUE"], errors="coerce").fillna(0.0)
    seg = actions.apply_original(seg)
    seg = actions.apply_agent_dictionary(seg, agent_dict)
    return seg[SEGMENT_COLUMNS]


def _counts(seg: pd.DataFrame) -> dict[str, pd.DataFrame]:
    def g(by, **agg):
        d = seg.assign(lines=1, **{k: seg[v] for k, v in agg.items()})
        return d.groupby(by, dropna=False, as_index=False)[["lines", *agg]].sum()
    s = seg.assign(W0_SEGMENT=seg["W0_SEGMENT"].fillna("(not targeted)"),
                   ACTION_ORIGINAL=seg["ACTION_ORIGINAL"].fillna("(none)"), ACTION_AGENT=seg["ACTION_AGENT"].fillna("(none)"),
                   RULE_ACTION=seg["RULE_ACTION"].fillna("excluded"), W0_ACTION=seg["W0_ACTION"].fillna("(none)"),
                   YOUNG_SPLIT=seg["YOUNG_SPLIT"].fillna("-"), FAMILY=rules.segment_family(seg["SEGMENT"]))
    seg = s
    return {
        "segments": g(["RULE_ID", "SEGMENT", "RULE_ACTION"], arpu_sum="ARPU_M1", w0_targeted="W0_TARGETED", w0_takers="W0_REWARD"),
        "micro": g(["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"]),
        "agreement": g(["SEGMENT", "W0_SEGMENT"]),
        "tenure": g(["FAMILY", "TENURE_BAND"]),
        "actions": g(["ACTION_ORIGINAL_SOURCE", "ACTION_ORIGINAL", "ACTION_AGENT"]),
        "context": g(["VALUE_BAND", "USAGE_TYPE"]),
        "young": g(["NATIONALITY_GROUP", "YOUNG_SPLIT"], takers="W0_REWARD", targeted="W0_TARGETED"),
        "w0": g(["W0_ACTION"], targeted="W0_TARGETED", takers="W0_REWARD", btl_revenue="W0_BTL_REVENUE"),
    }


def _t3_part(job: tuple) -> dict:
    feat_path, cache_root, run_date, part, cutoffs, agent_dict, out = job
    cache = CacheSource(run_date, Path(cache_root))
    catalog = load_catalog(cache) if cache.has("btl_catalog") else None
    seg = segment_partition(pd.read_parquet(feat_path), cache, part, date.fromisoformat(run_date), cutoffs, agent_dict, catalog)
    seg.to_parquet(out, index=False)
    return _counts(seg)


def run_t3(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None, workers: int | None = None,
           force: bool = False) -> dict:
    _t0(run_date, runs_root)
    train = runs_dir(run_date, runs_root) / "train"
    t2 = read_step(train / T2)
    if t2.get("status") != "done":
        raise StageBlocked(f"T2_features of {run_date} is {t2.get('status', 'missing')}: run fsdr-features")
    workers = int(workers or config.FEATURE_WORKERS)
    root = Path(runs_root or config.FSDR_RUNS_DIR)
    prev_dict, prev_path = actions.previous_agent_dictionary(root, run_date)
    inputs = fingerprint({"t2": t2.get("inputs"), "t2_started": t2.get("started"), "rules": rules_version(),
                          "previous_dictionary": str(prev_path or "")})
    with Step(train, T3, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        cache = CacheSource(run_date, cache_root)
        cutoffs = json.loads((train / T2 / "stats.json").read_text(encoding="utf-8"))["arpu_tertiles"]
        agent_dict, problems = actions.validate_agent_dictionary(prev_dict or actions.initial_agent_dictionary(run_date), prev_dict)
        for f in actions.dictionary_files(st.dir):          # an agent-2b version of an earlier T3 run of this date:
            if f.name != f"{agent_dict['name']}.json":     # the segments are rewritten with agent_dict, so drop it
                f.unlink()
        st.write_json(f"{agent_dict['name']}.json", agent_dict)
        st.write_json("agent_dictionary_check.json", {"version": agent_dict["name"], "source": str(prev_path or "v0 (week 0)"),
                                                      "problems": problems})
        st.write_json("dict_segm_actions_original.json", {"source_order": config.DICT_SEGM_ACTIONS_SOURCE,
                                                          "effective (last value wins)": config.DICT_SEGM_ACTIONS})
        crosswalk = actions.crosswalk_table()
        st.write_csv(crosswalk, "crosswalk.csv")

        parts = list(range(cache.partitions))
        outs = [st.output(f"segments/part-{k:02d}.parquet") for k in parts]
        jobs = [(str(train / T2 / "base_features" / f"part-{k:02d}.parquet"), str(cache_dir(run_date, cache_root).parent),
                 _iso(run_date), k, cutoffs, agent_dict, str(outs[k])) for k in parts]
        counts = _run(_t3_part, jobs, workers)
        summary = _t3_reports(st, counts, run_date, crosswalk, agent_dict)
        st.write_csv(pd.read_parquet(outs[0]).head(config.STEP_PREVIEW_ROWS), "segments_preview.csv")
        summary.update({"run_date": _iso(run_date), "stage": T3, "rules_version": rules_version(),
                        "agent_dictionary": agent_dict["name"], "agent_dictionary_problems": len(problems),
                        "partitions": len(parts), "workers": workers, "step_dir": str(st.dir)})
        st.rows(rows_in=summary["lines"], rows_out=summary["lines"])
        st.note(rules_version=rules_version(), agent_dictionary=agent_dict["name"])
        st.write_json("summary.json", summary)
        return summary


def current_agent_dictionary(run_date: date, runs_root: Path | None = None) -> dict:
    """The agent dictionary T3 of this run date applied last (the highest version in its folder)."""
    t3 = runs_dir(run_date, runs_root) / "train" / T3
    files = actions.dictionary_files(t3)
    if not files:
        raise StageBlocked(f"T3_segments of {run_date} has no agent dictionary: run fsdr-features first")
    return json.loads(files[-1].read_text(encoding="utf-8"))


def update_agent_dictionary(run_date: date, proposal: dict, runs_root: Path | None = None,
                            made_by: str = "agent 2b") -> dict:
    """Agent 2b (V4.0): a new version of the agent dictionary for this run date. The proposal (cells as in v0) is
    checked by validate_agent_dictionary against the current version (an invalid cell keeps its current value),
    saved as dict_segm_actions_agent_v<N+1>.json in T3_segments/, and applied to every segment partition
    (ACTION_AGENT, ACTION_AGENT_RANK2, ACTION_AGENT_VERSION), so the panel and Target 2 use it. Next week's T3
    starts from this version. Returns the version, the changed cells and the problems."""
    t3 = runs_dir(run_date, runs_root) / "train" / T3
    if read_step(t3).get("status") != "done":
        raise StageBlocked(f"T3_segments of {run_date} is not done: run fsdr-features first")
    current = current_agent_dictionary(run_date, runs_root)
    version = int(current["version"]) + 1
    draft = {**proposal, "version": version, "name": f"{config.AGENT_DICTIONARY_PREFIX}{version}",
             "created": _iso(run_date), "parent": current["name"], "made_by": made_by}
    new, problems = actions.validate_agent_dictionary(draft, current)
    old = {(c["segment"], c["nationality_group"]): [a["action"] for a in c["actions"]] for c in current["cells"]}
    changed = [{"segment": c["segment"], "nationality_group": c["nationality_group"],
                "before": old.get((c["segment"], c["nationality_group"])), "after": [a["action"] for a in c["actions"]],
                "reason": (c["actions"][0].get("reason", "") if c["actions"] else "")}
               for c in new["cells"] if old.get((c["segment"], c["nationality_group"])) != [a["action"] for a in c["actions"]]]
    (t3 / f"{new['name']}.json").write_text(json.dumps(new, indent=2, default=str), encoding="utf-8")
    (t3 / f"agent_dictionary_check_v{version}.json").write_text(
        json.dumps({"version": new["name"], "parent": current["name"], "made_by": made_by, "changed": changed,
                    "problems": problems}, indent=2, default=str), encoding="utf-8")
    lines = 0
    for part in sorted((t3 / "segments").glob("part-*.parquet")):
        seg = actions.apply_agent_dictionary(pd.read_parquet(part), new)
        seg.to_parquet(part, index=False)
        lines += len(seg)
    return {"version": new["name"], "parent": current["name"], "changed_cells": changed, "problems": problems,
            "lines": lines}


def _t3_reports(st: Step, counts: list[dict], run_date: date, crosswalk: pd.DataFrame, agent_dict: dict) -> dict:
    get = lambda k: [c[k] for c in counts]                                        # noqa: E731
    seg = _sum_counts(get("segments"), ["RULE_ID", "SEGMENT", "RULE_ACTION"], ["lines", "arpu_sum", "w0_targeted", "w0_takers"])
    order = {r: i for i, (r, _, _) in enumerate(config.S_RULES)}
    seg = seg.sort_values("RULE_ID", key=lambda s: s.map(order)).reset_index(drop=True)
    lines = int(seg["lines"].sum())
    seg["share"] = (seg["lines"] / max(lines, 1)).round(4)
    seg["avg_arpu_m1"] = (seg["arpu_sum"] / seg["lines"].where(seg["lines"] > 0)).round(3)
    st.write_csv(seg.drop(columns=["arpu_sum"]), "segment_counts.csv")

    micro = _sum_counts(get("micro"), ["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"])
    grid = pd.DataFrame(bands.all_micro_segments(), columns=["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"])
    micro = grid.merge(micro, how="left").fillna({"lines": 0})
    micro["lines"] = micro["lines"].astype(int)
    micro["MICRO_SEGMENT"] = micro["NATIONALITY_GROUP"] + "|" + micro["AGE_BAND"] + "|" + micro["TENURE_BAND"]
    st.write_csv(micro[["MICRO_SEGMENT", "NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND", "lines"]], "microsegment_counts.csv")

    agree = _sum_counts(get("agreement"), ["SEGMENT", "W0_SEGMENT"])
    cw = {s: config.SEGMENT_CROSSWALK[s][0].lower() for s in config.S_SEGMENTS}
    targeted = agree[agree["W0_SEGMENT"] != "(not targeted)"]
    match = targeted["SEGMENT"].map(cw) == targeted["W0_SEGMENT"].astype(str).str.lower()
    agree["match"] = (agree["SEGMENT"].map(cw) == agree["W0_SEGMENT"].astype(str).str.lower())
    st.write_csv(agree.sort_values(["SEGMENT", "lines"], ascending=[True, False]), "segment_agreement.csv")

    ten = _sum_counts(get("tenure"), ["FAMILY", "TENURE_BAND"])
    st.write_csv(ten.pivot_table(index="FAMILY", columns="TENURE_BAND", values="lines", aggfunc="sum", fill_value=0)
                   .astype(int).reset_index(), "tenure_agreement.csv")
    acts = _sum_counts(get("actions"), ["ACTION_ORIGINAL_SOURCE", "ACTION_ORIGINAL", "ACTION_AGENT"])
    st.write_csv(acts, "action_counts.csv")
    st.write_csv(_sum_counts(get("context"), ["VALUE_BAND", "USAGE_TYPE"]), "value_usage.csv")
    young = _sum_counts(get("young"), ["NATIONALITY_GROUP", "YOUNG_SPLIT"], ["lines", "takers", "targeted"])
    st.write_csv(young[young["YOUNG_SPLIT"] != "-"], "young_split.csv")

    w0 = _sum_counts(get("w0"), ["W0_ACTION"], ["lines", "targeted", "takers", "btl_revenue"])
    acted = w0[w0["W0_ACTION"].isin(config.ACTIONS)]
    n = int(acted["targeted"].sum())
    recomputed = {a: round(float(acted.loc[acted["W0_ACTION"] == a, "targeted"].sum()) / n, 7) if n else None
                  for a in config.ACTIONS}
    take = {a: (round(float(r["takers"]) / r["targeted"], 4) if r["targeted"] else None)
            for a, r in acted.set_index("W0_ACTION").iterrows()}
    priors = {"rule": "B-10", "configured": config.ACTION_PRIOR, "recomputed_week0": recomputed,
              "targeted_with_action": n, "take_rate_week0": take, "order": config.ACTIONS,
              "used": "recomputed_week0" if n else "configured"}
    st.write_json("priors.json", priors)

    excluded = int(seg.loc[seg["RULE_ACTION"] == "excluded", "lines"].sum())
    src = acts.groupby("ACTION_ORIGINAL_SOURCE")["lines"].sum().to_dict()
    same = acts[acts["ACTION_ORIGINAL"] == acts["ACTION_AGENT"]]["lines"].sum()
    non_empty = int((micro["lines"] > 0).sum())
    return {"lines": lines, "excluded_nursering": excluded, "targetable": lines - excluded,
            "segments_non_empty": int((seg["lines"] > 0).sum()),
            "segment_lines": dict(zip(seg["SEGMENT"], seg["lines"].astype(int))),
            "micro_segments_non_empty": non_empty, "micro_segments_total": len(micro),
            "micro_segments_ge_300": int((micro["lines"] >= 300).sum()),
            "engine_targeted_lines": int(targeted["lines"].sum()),
            "engine_vs_s_rules_agreement": round(float(targeted.loc[match, "lines"].sum() / max(targeted["lines"].sum(), 1)), 4),
            "action_original_source": {k: int(v) for k, v in src.items()},
            "action_original_eq_agent": round(float(same / max(lines, 1)), 4),
            "crosswalk_same_action": int(crosswalk["same_action"].sum()), "crosswalk_lines": len(crosswalk),
            "priors_used": priors["used"], "week0_targeted_with_action": n}


def run_features(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                 workers: int | None = None, force: bool = False) -> dict:
    """fsdr-features: T2 then T3 (force re-runs both)."""
    t2 = run_t2(run_date, runs_root, cache_root, workers, force)
    t3 = run_t3(run_date, runs_root, cache_root, workers, force)
    return {"T2_features": t2, "T3_segments": t3}


In [ ]:
%%engine nbx.fsdr.synth package
"""Dummy-data world, scenarios and generator (same JSON files as the real data)."""


In [ ]:
%%engine nbx.fsdr.synth.scenarios
"""Use-case scenarios for the dummy data (combinable, e.g. "churn_wave,data_growth").

The behavioural scenarios change the hidden world (synth/world.py reads
World.scenarios): drifts ramp up over the last 4 weeks before the world's
as_of date, so a single generated date already shows them in the W1..W4 trends,
and the weekly simulation keeps them going. `data_quality_issues` instead
damages the generated tables after they are built, so validation has real
problems to catch.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from nbx.fsdr import config


@dataclass(frozen=True)
class Scenario:
    name: str
    description: str
    expected_signal: str


SCENARIOS: dict[str, Scenario] = {s.name: s for s in [
    Scenario("baseline", "Normal weekly behaviour.", "-"),
    Scenario("churn_wave", "Churn risk rises over 4 weeks for lines in North Batinah (generate --weeks 4).",
             "PROBA_CHURN and REV_DECR up and usage down in those regions; RETAIN pays off there."),
    Scenario("data_growth", "Voice-only customers with data appetite gradually start using data.",
             "DATA_VOLUME_MB W1 > W4 for them, PROBA_APP_DATA up; UPSELL / CROSS SELL DATA grow."),
    Scenario("price_sensitive", "Price sensitivity x2.5: acceptance drops fast above the ARPU anchor.",
             "Take rate falls steeply with (price - anchor)."),
    Scenario("new_offers", "3 new BTL offers (name ends with _NEW) in the catalogue, with no purchase history (cold start).",
             "Offers in btl_catalog absent from sub_btl_3months; ranker must fall back."),
    Scenario("campaign_fatigue", "Acceptance falls with the number of contacts in the last 4 weeks.",
             "Take rate decreases with NB_CONTACTS_LAST_4W."),
    Scenario("data_quality_issues", "Duplicate MSISDNs, null keys, stacked churn duplicates, keys as strings.",
             "Validation reports each issue."),
]}


def parse(value: str | list[str] | None) -> list[str]:
    """'churn_wave,data_growth' -> ['churn_wave', 'data_growth'] (validated, order kept, duplicates dropped)."""
    if value is None:
        return ["baseline"]
    items = value.split(",") if isinstance(value, str) else list(value)
    out = []
    for s in (x.strip() for x in items):
        if not s:
            continue
        if s not in config.SCENARIOS:
            raise ValueError(f"Unknown scenario '{s}'. Valid: {', '.join(config.SCENARIOS)}")
        if s not in out:
            out.append(s)
    return out or ["baseline"]


# ---------------------------------------------------------------------------
# data_quality_issues: applied to the raw (object) frames just before writing
# ---------------------------------------------------------------------------
QUALITY_SHARE = 0.03          # share of rows damaged per issue (above the 1 % blocking threshold)


def _rows(rng: np.random.Generator, df: pd.DataFrame, share: float = QUALITY_SHARE) -> np.ndarray:
    k = max(1, int(len(df) * share))
    return rng.choice(len(df), size=min(k, len(df)), replace=False)


def inject_quality_issues(tables: dict[str, pd.DataFrame], seed: int) -> list[dict]:
    """Damages the tables in place; returns what was injected (written to the manifest)."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 99]))
    log: list[dict] = []

    # 1. duplicate MSISDNs in base_prepaid (same line, another ACCESS_METHOD_ID)
    base = tables["base_prepaid"]
    dup = base.iloc[_rows(rng, base)].copy()
    dup["ACCESS_METHOD_ID"] = [int(v) + 900_000_000 for v in dup["ACCESS_METHOD_ID"]]
    tables["base_prepaid"] = pd.concat([base, dup], ignore_index=True)
    log.append({"issue": "duplicate_msisdn", "table": "base_prepaid", "rows": len(dup)})

    # 2. null keys
    for table, col in [("base_prepaid", "PARTY_ID"), ("activities_w1_auto", "ACCESS_METHOD_ID"),
                       ("base_prepaid", "ACCESS_METHOD_VAL")]:
        df = tables[table]
        idx = _rows(rng, df)
        df.loc[df.index[idx], col] = None
        log.append({"issue": "null_key", "table": table, "column": col, "rows": len(idx)})

    # 3. stacked duplicates in the global churn model (same line scored twice, different probabilities)
    churn_t = config.MODEL_SCORE_REGISTRY[0].table
    ch = tables[churn_t]
    stacked = ch.iloc[_rows(rng, ch, 0.05)].copy()
    stacked["proba_1"] = [round(float(rng.uniform(0, 1)), 6) for _ in range(len(stacked))]
    stacked["proba_0"] = [round(1 - p, 6) for p in stacked["proba_1"]]
    tables[churn_t] = pd.concat([ch, stacked], ignore_index=True)
    log.append({"issue": "stacked_duplicates", "table": churn_t, "rows": len(stacked)})

    # 4. integer keys delivered as strings
    for table, col in [("subs_amt_auto", "ACCESS_METHOD_ID"), (churn_t, "access_method_val")]:
        df = tables[table]
        idx = _rows(rng, df)
        df.loc[df.index[idx], col] = [None if v is None else str(v) for v in df.loc[df.index[idx], col]]
        log.append({"issue": "key_as_string", "table": table, "column": col, "rows": len(idx)})

    # 5. customer_profile disagreeing with party_* / the activation date, out of range, negative (V2.0)
    prof = tables.get(config.PROFILE_TABLE)
    if prof is not None and len(prof) >= 10:
        idx = _rows(rng, prof, 0.04)
        q = len(idx) // 4
        ages, tenures = prof["age"].copy(), prof["tenure"].copy()
        for i in idx[:q]:
            ages.iat[i] = int(ages.iat[i]) + 7                              # > 2 years from party_*.AGE
        for i in idx[q:2 * q]:
            tenures.iat[i] = int(tenures.iat[i]) + 90                       # > 30 days from the activation date
        for i in idx[2 * q:3 * q]:
            ages.iat[i] = 7                                                 # outside 15-100
        for i in idx[3 * q:]:
            tenures.iat[i] = -5                                             # negative
        prof["age"], prof["tenure"] = ages, tenures
        log.append({"issue": "profile_disagreement", "table": config.PROFILE_TABLE, "rows": len(idx),
                    "age_off": q, "tenure_off": q, "age_out_of_range": q, "negative_tenure": len(idx) - 3 * q})
    return log


In [ ]:
%%engine nbx.fsdr.synth.world
"""Hidden customer world for the dummy data and the closed-loop simulation.

Every customer has a hidden state (churn risk, data / voice appetite, price
sensitivity, preferred validity, spend level, usage type). From it the world
produces:
  - a daily behaviour history (calls, minutes, MB, PAYG, subscriptions) --
    the generator aggregates it with the business-rule windows;
  - the hidden signals that drive the 11 model scores (scores are noisy
    readings of the truth, not pure noise);
  - a hidden response function P(accept | customer, action, offer, price).

Daily values come from a random stream keyed on (seed, calendar day, chunk),
so two generation dates of the same world agree on every shared day: W2 of
date D equals W1 of date D-7 by construction.

Nothing here is visible to the pipelines; they only see the JSON tables.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date, timedelta

import numpy as np
import pandas as pd

from nbx.fsdr import config

USAGE_TYPES = np.array(["voice_only", "data_only", "dual", "inactive"])
NATIONALITIES = ["Oman", "India", "Bangladesh", "Pakistan", "Egypt", "Philippines", "Sri Lanka", "Nepal",
                 "Jordan", "Sudan", "Yemen", "Syria", "Indonesia", "United Kingdom", "Tanzania"]
NATIONALITY_P = [0.45, 0.22, 0.12, 0.09, 0.025, 0.02, 0.015, 0.012, 0.01, 0.008, 0.008, 0.006, 0.005, 0.004, 0.007]
REGULAR_PLANS = ["FSDR Prepaid Basic", "FSDR Prepaid Plus", "FSDR Prepaid Smart", "FSDR Hala Flex", "FSDR Visitor SIM",
                 "FSDR Shababiah", "FSDR Prepaid Data"]
REGULAR_PLAN_P = [0.30, 0.22, 0.16, 0.12, 0.05, 0.08, 0.07]
EXCLUDED_PLAN_SHARE = 0.02
DV_SEGMENTS = ["DV_SMARTPHONE_HIGH", "DV_SMARTPHONE_MID", "DV_SMARTPHONE_LOW", "DV_FEATURE_PHONE", "DV_ROUTER"]
REGIONS = ["Muscat", "Dhofar", "Musandam", "Al Buraimi", "Ad Dakhiliyah", "North Batinah", "South Batinah",
           "South Sharqiyah", "North Sharqiyah", "Ad Dhahirah", "Al Wusta"]          # the 11 governorates
WAVE_REGIONS = {"North Batinah"}                    # churn_wave hits this region
GENEROSITY_BONUS = {"L0": 0.0, "L1": 0.35, "L2": 0.7}
ORGANIC_WEEKLY_BTL_RATE = 0.03                      # BTL bought without contact (control arm baseline)
RESPONSE_BASE_LOGIT = -5.6                          # tuned: ~10 % take rate per targeted customer (4 offers)


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def _unique_ids(rng: np.random.Generator, n: int, start: int, max_step: int) -> np.ndarray:
    """n unique integers >= start in random order (cheap, no huge permutation)."""
    ids = start + np.cumsum(rng.integers(1, max_step, n))
    rng.shuffle(ids)
    return ids


def msisdns(rng: np.random.Generator, n: int) -> np.ndarray:
    """n unique 8-digit MSISDNs > 69999999 (BR-02) at any size: the mean step shrinks above ~590k lines, so the
    1M / 2M modes stay below 100000000 (smaller runs draw exactly as before)."""
    return _unique_ids(rng, n, 70_000_000, max(2, min(100, 59_000_000 // max(n, 1)))).astype(str)


# ---------------------------------------------------------------------------
# Offer catalogue
# ---------------------------------------------------------------------------
_PRICE_LADDER = {  # (service, validity) -> {generosity: [prices]}; generosity monotonic with price
    ("DATA", 30): {"L0": [2.0, 3.0, 4.0], "L1": [5.0, 6.0, 7.0], "L2": [8.0, 10.0, 12.0]},
    ("VOICE", 30): {"L0": [1.5, 2.5, 3.5], "L1": [4.0, 5.0, 6.0], "L2": [7.0, 8.5, 10.0]},
    ("COMBO", 30): {"L0": [3.0, 4.5, 6.0], "L1": [7.0, 8.5, 10.0], "L2": [11.0, 13.0, 15.0]},
    ("DATA", 7): {"L0": [0.5, 0.8, 1.0], "L1": [1.2, 1.5, 1.8], "L2": [2.0, 2.5, 3.0]},
    ("VOICE", 7): {"L0": [0.5, 0.7, 0.9], "L1": [1.0, 1.3, 1.6], "L2": [1.8, 2.2, 2.6]},
    ("COMBO", 7): {"L0": [0.8, 1.2, 1.5], "L1": [1.8, 2.2, 2.6], "L2": [3.0, 3.5, 4.0]},
    # add-on families for the batch flow (roaming data, international minutes)
    ("ROAMING", 30): {"L0": [5.0], "L1": [9.0], "L2": [14.0]},
    ("ROAMING", 7): {"L0": [2.0], "L1": [3.5], "L2": [5.0]},
    ("INTERNATIONAL", 30): {"L0": [2.0], "L1": [4.0], "L2": [7.0]},
    ("INTERNATIONAL", 7): {"L0": [0.7], "L1": [1.3], "L2": [2.2]},
}
_GB_PER_OMR = {"L0": 0.8, "L1": 1.2, "L2": 1.8}
_MIN_PER_OMR = {"L0": 25, "L1": 35, "L2": 50}
_GB_SHARE = {"DATA": 1.0, "COMBO": 0.6, "ROAMING": 0.25}           # services that carry data
_MIN_SHARE = {"VOICE": 1.0, "COMBO": 0.5, "INTERNATIONAL": 0.4}     # services that carry minutes


def build_catalog(new_offers: bool = False) -> pd.DataFrame:
    """~54 BTL offers (+3 'new' ones, never bought yet, for the new_offers scenario). Typed frame;
    the generator writes it with the production string types."""
    rows = []
    for (service, validity), ladder in _PRICE_LADDER.items():
        for gen, prices in ladder.items():
            for price in prices:
                rows.append(_offer_row(service, validity, gen, price, is_new=False))
    if new_offers:
        for service, validity, gen, price in [("DATA", 30, "L2", 9.0), ("COMBO", 30, "L2", 12.0), ("COMBO", 7, "L1", 2.4)]:
            rows.append(_offer_row(service, validity, gen, price, is_new=True))
    df = pd.DataFrame(rows).drop_duplicates("btl_name").reset_index(drop=True)
    return df


def _offer_row(service: str, validity: int, gen: str, price: float, is_new: bool) -> dict:
    gb = round(price * _GB_PER_OMR[gen] * _GB_SHARE[service], 1) if service in _GB_SHARE else 0.0
    mins = int(price * _MIN_PER_OMR[gen] * _MIN_SHARE[service]) if service in _MIN_SHARE else 0
    parts = [f"BTL_{service}_{gen}_{validity}D"]
    if gb:
        parts.append(f"{gb:g}GB")
    if mins:
        parts.append(f"{mins}MIN")
    if is_new:
        parts.append("NEW")
    price_data = price if service == "DATA" else (round(price * 0.7, 3) if service == "COMBO" else 0.0)
    return {"btl_name": "_".join(parts), "type": f"BTL_{service}", "generosity": gen, "gb": gb, "min": mins,
            "price": price, "price_data": price_data, "price_voice": round(price - price_data, 3),
            "service": service, "validity": validity, "is_new": is_new}


# ---------------------------------------------------------------------------
# World
# ---------------------------------------------------------------------------
@dataclass
class World:
    seed: int
    scenarios: list[str]
    as_of: date                       # scenario drifts end here (ramp over the previous 4 weeks)
    c: pd.DataFrame                   # one row per customer: ids, demographics, hidden state
    catalog: pd.DataFrame
    params: dict = field(default_factory=dict)

    @property
    def n(self) -> int:
        return len(self.c)

    def has(self, scenario: str) -> bool:
        return scenario in self.scenarios

    # -- creation ------------------------------------------------------------
    @classmethod
    def create(cls, n: int, seed: int = config.GENERATOR_SEED, scenarios: list[str] | None = None,
               as_of: date | None = None, arms: list[str] | None = None) -> "World":
        scenarios = [s for s in (scenarios or ["baseline"]) if s]
        unknown = set(scenarios) - set(config.SCENARIOS)
        if unknown:
            raise ValueError(f"Unknown scenario(s) {sorted(unknown)}; valid: {config.SCENARIOS}")
        if not 1 <= n <= config.GENERATOR_MAX_N:
            raise ValueError(f"n must be between 1 and {config.GENERATOR_MAX_N}")
        as_of = as_of or date.today()
        rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))

        nat = rng.choice(NATIONALITIES, n, p=np.array(NATIONALITY_P) / sum(NATIONALITY_P))
        omani = nat == "Oman"
        u = rng.random(n)
        tenure = np.where(u < 0.06, rng.integers(1, 45, n),
                          np.where(u < 0.11, rng.integers(45, 91, n),
                                   np.clip(90 + rng.lognormal(np.log(700), 0.9, n), 91, 3650))).astype(int)
        usage_p = np.where(omani[:, None], [[0.22, 0.20, 0.53, 0.05]], [[0.38, 0.10, 0.47, 0.05]])
        usage = USAGE_TYPES[(usage_p.cumsum(axis=1) < rng.random(n)[:, None]).sum(axis=1).clip(0, 3)]
        plan = rng.choice(REGULAR_PLANS, n, p=REGULAR_PLAN_P)
        excl = rng.random(n) < EXCLUDED_PLAN_SHARE
        plan = np.where(excl, rng.choice(config.EXCLUDED_PLANS, n), plan)
        spend = np.clip(rng.lognormal(np.log(4.5), 0.75, n), 0.3, 60.0)          # OMR / month, heavy-tailed
        churn = rng.beta(1.5, 6.0, n)
        churn = np.clip(churn + np.where(tenure < 60, 0.18, 0.0) + np.where(usage == "inactive", 0.35, 0.0), 0, 0.97)
        data_app = rng.beta(2.0, 4.0, n)
        data_app = np.where((usage == "voice_only") & (rng.random(n) < 0.45), np.clip(data_app + 0.4, 0, 1), data_app)
        voice_app = rng.beta(2.0, 4.0, n)
        voice_app = np.where((usage == "data_only") & (rng.random(n) < 0.45), np.clip(voice_app + 0.4, 0, 1), voice_app)
        price_sens = rng.beta(2.0, 3.0, n) * 2.0
        if "price_sensitive" in scenarios:
            price_sens = price_sens * 2.5
        churned = rng.random(n) < churn * 0.35                                   # some lines stop being active
        churn_day = np.where(churned, rng.integers(-40, 25, n), 10_000)          # offset (days) from as_of

        region = rng.choice(REGIONS, n)
        c = pd.DataFrame({
            "access_method_id": _unique_ids(rng, n, 1_000_000, 60),
            "msisdn": msisdns(rng, n),
            "account_id": _unique_ids(rng, n, 30_000_000, 50),
            "party_id": _unique_ids(rng, n, 5_000_000, 50),
            "subscription_id": (4_000_000_000 + _unique_ids(rng, n, 0, 40)).astype(str),
            "tenure": tenure,
            "nationality": nat,
            "gender": np.where(omani, rng.choice(["M", "F"], n), rng.choice(["M", "F"], n, p=[0.8, 0.2])),
            "age": np.clip(rng.normal(34, 10, n), 18, 75).astype(int),
            "plan": plan,
            "usage_type": usage,
            "region": region,
            "wilayat": [f"WILAYAT_{REGIONS.index(r) + 1:02d}{k}" for r, k in zip(region, rng.integers(1, 6, n))],
            "area": [f"AREA_{x:04d}" for x in rng.integers(1, 400, n)],
            "dv_segment": rng.choice(DV_SEGMENTS, n, p=[0.25, 0.35, 0.22, 0.12, 0.06]),
            "spend": spend,
            "churn_risk": churn,
            "data_appetite": data_app,
            "voice_appetite": voice_app,
            "price_sensitivity": price_sens,
            "preferred_validity": np.where(rng.random(n) < 0.55, 7, 30),
            "onnet_share": rng.uniform(0.4, 0.8, n),
            "intl_share": np.where(omani, rng.uniform(0.0, 0.04, n), rng.uniform(0.05, 0.3, n)),
            "payg_share": rng.uniform(0.1, 0.6, n),
            "churn_day": churn_day,
            "contacts_last_4w": rng.poisson(0.8, n),
            # travels abroad (drives roaming purchases): more often expats and high spenders
            "traveller": rng.random(n) < np.clip(np.where(omani, 0.05, 0.12) + 0.02 * np.clip(spend / 10, 0, 2), 0, 0.5),
            "wave": np.isin(region, list(WAVE_REGIONS)) if "churn_wave" in scenarios else np.zeros(n, bool),
            "growth": ((usage == "voice_only") & (data_app > 0.5)) if "data_growth" in scenarios else np.zeros(n, bool),
        })
        arms = [a for a in config.ARMS if a in (arms or config.ARMS)]
        if not arms:
            raise ValueError(f"arms must be a subset of {config.ARMS}")
        params = {"fatigue": 0.35 if "campaign_fatigue" in scenarios else 0.05, "arms": arms}
        return cls(seed, scenarios, as_of, c, build_catalog("new_offers" in scenarios), params)

    # -- time-varying state ----------------------------------------------------
    def ramp(self, day: date) -> float:
        """0 before the scenario drift starts, 1 at as_of (linear over SCENARIO_RAMP_DAYS)."""
        start = self.as_of - timedelta(days=config.SCENARIO_RAMP_DAYS)
        return float(np.clip((day - start).days / config.SCENARIO_RAMP_DAYS, 0.0, 1.0))

    def churn_risk_at(self, day: date, idx=slice(None)) -> np.ndarray:
        base = self.c["churn_risk"].to_numpy()[idx]
        wave = self.c["wave"].to_numpy()[idx]
        return np.clip(base + wave * 0.55 * self.ramp(day) * (1 - base), 0, 0.99)

    def data_appetite_at(self, day: date, idx=slice(None)) -> np.ndarray:
        base = self.c["data_appetite"].to_numpy()[idx]
        return np.clip(base + self.c["growth"].to_numpy()[idx] * 0.3 * self.ramp(day), 0, 1)

    # -- daily behaviour -------------------------------------------------------
    def daily(self, day: date, idx: slice, chunk_id: int) -> dict[str, np.ndarray]:
        """One day of behaviour for customers idx. Deterministic in (seed, day, chunk_id)."""
        rng = np.random.default_rng(np.random.SeedSequence([self.seed, 2, day.toordinal(), chunk_id]))
        c = self.c.iloc[idx]
        m = len(c)
        usage = c["usage_type"].to_numpy()
        spend = c["spend"].to_numpy()
        churn_eff = self.churn_risk_at(day, idx)
        off = (day - self.as_of).days
        gone = off >= c["churn_day"].to_numpy()
        base_act = np.select([usage == "dual", usage == "voice_only", usage == "data_only"], [0.88, 0.82, 0.82], 0.08)
        p_act = np.where(gone, 0.03, base_act * (1.0 - 0.45 * churn_eff))
        active = rng.random(m) < p_act

        voice_user = usage != "data_only"
        growth_on = c["growth"].to_numpy() & (rng.random(m) < 0.9 * self.ramp(day))
        data_user = (usage == "data_only") | (usage == "dual") | growth_on | ((usage == "inactive") & (rng.random(m) < 0.3))

        level = np.sqrt(spend / 4.5)
        calls = rng.poisson(4.0 * level) * active * voice_user
        mins_total = np.where(calls > 0, rng.gamma(np.maximum(1.5 * calls, 1e-9), 1.6), 0.0)
        intl = mins_total * c["intl_share"].to_numpy()
        local_min = mins_total - intl
        local_calls = np.rint(calls * (1 - c["intl_share"].to_numpy())).astype(int)
        onnet = c["onnet_share"].to_numpy()
        onnet_calls = np.rint(local_calls * onnet).astype(int)

        mb_per_session = 35.0 * level * (1.0 + 0.8 * self.data_appetite_at(day, idx))
        sessions = rng.poisson(np.where(growth_on & (usage == "voice_only"), 2.0 + 4.0 * self.ramp(day), 6.0) * level) * active * data_user
        mb = np.where(sessions > 0, rng.gamma(np.maximum(0.8 * sessions, 1e-9), mb_per_session), 0.0)

        payg = c["payg_share"].to_numpy()
        pg_voice = local_min * 0.012 * payg + intl * 0.08
        pg_data = mb * 0.0015 * payg
        pg_other = np.where(active & (rng.random(m) < 0.1), 0.05, 0.0)
        pg_neg = np.where(active & (rng.random(m) < 0.005), -rng.uniform(0.05, 0.5, m), 0.0)

        p_sub = np.clip(spend * (1 - 0.5 * payg) / 3.2 / 30.0, 0, 0.9) * np.where(gone, 0.05, 1.0)
        buys = rng.random(m) < p_sub
        amt = np.where(buys, np.maximum(0.3, np.round(spend * rng.uniform(0.15, 0.5, m) * 2) / 2), 0.0)
        cls_u = rng.random(m)
        p_data_cls = np.where(data_user, 0.6, 0.1)
        is_data = buys & (cls_u < p_data_cls)
        is_voice = buys & ~is_data & (cls_u < p_data_cls + 0.3)

        return {
            "voice_calls": calls.astype(float),
            "voice_min_rounded": mins_total * 1.07,
            "local_calls": local_calls.astype(float),
            "local_min": local_min,
            "onnet_calls": onnet_calls.astype(float),
            "onnet_min": local_min * onnet,
            "offnet_calls": (local_calls - onnet_calls).astype(float),
            "offnet_min": local_min * (1 - onnet),
            "data_sessions": sessions.astype(float),
            "data_mb_rounded": mb * 1.03,
            "data_mb": mb,
            "pg_voice": pg_voice,
            "pg_data": pg_data,
            "pg_pos": pg_voice + pg_data + pg_other,                 # positive charges only (BR-33 _tg)
            "pg_all": pg_voice + pg_data + pg_other + pg_neg,        # every charge (activities)
            "sub_amt": amt,                                          # SUBSCRIPTION_AMT (BR-19 _tg)
            "sub_tot_amt": amt * 1.05,                               # SUBSCRIPTION_TOT_AMT incl. tax (BR-19 NBA)
            "sub_amt_data": np.where(is_data, amt, 0.0),
            "sub_amt_voice": np.where(is_voice, amt, 0.0),
            "sub_count": buys.astype(float),
            "voice_day": (calls > 0).astype(float),
            "data_day": (sessions > 0).astype(float),
            "any_day": ((calls > 0) | (sessions > 0) | (np.abs(pg_other + pg_neg) > 0)).astype(float),
        }

    # -- hidden signals behind the model scores --------------------------------
    def signal(self, name: str, idx=slice(None), day: date | None = None) -> np.ndarray:
        """True propensity in [0, 1] that model `name` tries to estimate on `day` (default as_of): the scores of
        week D read the world at D, so a scenario drift shows up week after week."""
        day = day or self.as_of
        c = self.c.iloc[idx]
        churn = self.churn_risk_at(day, idx)
        gone = (day - self.as_of).days >= c["churn_day"].to_numpy()
        tenure = c["tenure"].to_numpy()
        usage = c["usage_type"].to_numpy()
        if name == "churn":
            return np.clip(churn + 0.25 * gone + np.where(tenure < 60, 0.08, 0.0), 0, 0.99)
        if name == "churn_1d":
            return np.clip(0.8 * churn * ~gone + 0.05, 0, 0.99)
        if name in ("reactivation_30", "reactivation_15"):
            k = 1.0 if name == "reactivation_30" else 0.75
            low = gone | (usage == "inactive")
            return np.clip(k * np.where(low, 0.75 * (1 - c["churn_risk"].to_numpy()), 0.08), 0, 0.99)
        if name == "data_appetite":
            return self.data_appetite_at(day, idx)
        if name == "voice_appetite":
            return c["voice_appetite"].to_numpy()
        if name == "revenue_decrease":
            return np.clip(0.6 * churn + 0.3 * c["wave"].to_numpy() * self.ramp(day) + 0.05, 0, 0.99)
        if name == "fragility_data":
            return np.clip(np.where(usage == "voice_only", 0.05, 0.5 * churn + 0.3 * (1 - c["data_appetite"].to_numpy())), 0, 0.99)
        if name == "fragility_voice":
            return np.clip(np.where(usage == "data_only", 0.05, 0.5 * churn + 0.3 * (1 - c["voice_appetite"].to_numpy())), 0, 0.99)
        # unknown signal (a model added through config only): a noisy mix of the main signals
        return np.clip(0.5 * churn + 0.5 * self.data_appetite_at(day, idx), 0, 0.99)

    def score(self, name: str, rng: np.random.Generator, idx=slice(None), day: date | None = None) -> np.ndarray:
        """Noisy model score = sigmoid(logit(signal) + noise)."""
        s = self.signal(name, idx, day)
        return _sigmoid(_logit(s) + rng.normal(0.0, 0.6, len(s)))

    # -- hidden response function ---------------------------------------------
    def accept_probability(self, idx: np.ndarray, action: np.ndarray, service: np.ndarray, generosity: np.ndarray,
                           price: np.ndarray, validity: np.ndarray, day: date | None = None) -> np.ndarray:
        """P(accept offer | customer, action, offer, price vs spend) for aligned arrays."""
        day = day or self.as_of
        c = self.c.iloc[idx]
        churn = self.churn_risk_at(day, idx)
        da = self.data_appetite_at(day, idx)
        va = c["voice_appetite"].to_numpy()
        usage = c["usage_type"].to_numpy()
        voice_only, data_only, dual = usage == "voice_only", usage == "data_only", usage == "dual"
        spend = c["spend"].to_numpy()

        fit = np.select(
            [action == "RETAIN", action == "UPSELL DATA", action == "CROSS SELL DATA", action == "UPSELL VOICE",
             action == "CROSS SELL VOICE", action == "UPSELL", action == "MAINTAIN"],
            [2.5 * churn, 2.0 * da * voice_only, 1.5 * da * ~data_only * (da > 0.3), 2.0 * va * data_only,
             1.5 * va * ~voice_only * (va > 0.3), 1.2 * np.clip(spend / 10, 0, 1), 0.8 * (1 - churn) * dual],
            0.0)
        service_fit = np.select([service == "DATA", service == "VOICE"], [da, va], 0.5 * (da + va))
        gen = np.vectorize(GENEROSITY_BONUS.get)(generosity) if len(generosity) else np.zeros(0)
        anchor = np.where(validity == 7, spend / 4.3, spend)
        gap = np.maximum(0.0, price - anchor) / np.maximum(anchor, 0.5)
        logit = (RESPONSE_BASE_LOGIT + 1.6 * fit + 1.2 * service_fit + gen
                 - 1.2 * c["price_sensitivity"].to_numpy() * gap
                 + 0.3 * (validity == c["preferred_validity"].to_numpy())
                 - self.params["fatigue"] * c["contacts_last_4w"].to_numpy())
        return _sigmoid(logit)

    def organic_probability(self, idx: np.ndarray) -> np.ndarray:
        """Weekly chance of buying a BTL with no contact (control arm baseline)."""
        spend = self.c["spend"].to_numpy()[idx]
        return np.clip(ORGANIC_WEEKLY_BTL_RATE + 0.02 * np.clip(spend / 10, 0, 1), 0, 0.2)


In [ ]:
%%engine nbx.fsdr.synth.generator
"""Dummy-data generator: writes exactly the files of the real data contract.

Every usage / revenue table is aggregated from the world's daily history with
the windows of nbx/fsdr/calendar.py (BR §4), so the values are consistent
with each other and with the run date: activities M1 = W1 + W2 + W3 + W4,
Sub_Amt_m1 >= the four weeks inside it, and W2 of D = W1 of D-7.

Last week's campaign is simulated too: targeting (D-8..D-2) by the segment
policy, offers chosen like production's constraint arm (all-L2), acceptances
drawn from the hidden response function, fulfilments in D-7..D-1 plus organic
purchases and non-BTL noise, and 270 days of purchase history for the ranker.

    run = generate_run(World.create(5000, seed=42, scenarios=["baseline"], as_of=d), d)
    write_run(run, Path("data/input") / d.isoformat())                    # 5 folders, JSON
    write_run(run, Path("data/input") / d.isoformat(), fmt="parquet")     # 1M mode: ~10x smaller on disk

V2.0 (plan §13): the files go to the 5 input folders (usage / models /
subscription / catalog / others) unless layout="flat" (V1.0), in JSON, JSON
Lines, CSV or Parquet. customer_profile (age, tenure in days) agrees with
party_*.AGE and the activation date, except where the data_quality_issues
scenario breaks it on purpose.
"""
from __future__ import annotations

import json
import time
from collections import defaultdict
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import calendar as cal
from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import MONTHS, WEEKS
from nbx.fsdr.synth import scenarios as scen
from nbx.fsdr.synth.world import GENEROSITY_BONUS, World

NON_BTL_PRODUCTS = ["DAILY_DATA_PASS_1GB", "WEEKLY_SOCIAL_PASS", "INTL_CALLING_PACK_5", "HALA_FLEX_RENEWAL"]
NON_BTL_SHARE = 0.06


@dataclass
class GeneratedRun:
    run_date: date
    anchor_date: date
    tables: dict[str, pd.DataFrame]       # raw frames: python values, schema column order, None = null
    manifest: dict
    world: World


# ---------------------------------------------------------------------------
# Raw conversion (typed frame -> JSON-ready python values)
# ---------------------------------------------------------------------------
def to_raw(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Typed frame -> object frame holding exactly what the JSON file will hold."""
    schema = schemas.TABLES[table]
    n = len(df)
    cols = {}
    for f in schema.fields:
        if f.name not in df.columns:
            cols[f.name] = np.full(n, None, dtype=object)
            continue
        s = df[f.name]
        mask = s.isna().to_numpy()
        if f.raw_type in schemas.INT_TYPES:
            vals = pd.to_numeric(s, errors="coerce").fillna(0).round().astype("int64").astype(object)
        elif f.raw_type == "double":
            vals = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float).round(3).astype(object)
        else:
            vals = s.astype(str).astype(object)
        arr = vals.to_numpy(dtype=object)
        arr[mask] = None
        cols[f.name] = arr
    return pd.DataFrame(cols, columns=schema.field_names)


def _fmt_num(x: float) -> str:
    return f"{x:g}"


# ---------------------------------------------------------------------------
# Usage / revenue tables from the daily history
# ---------------------------------------------------------------------------
def _usage_tables(world: World, run_date: date) -> dict[str, pd.DataFrame]:
    t_anchor = cal.anchor_date(run_date)
    start = cal.usage_history_start(run_date)
    days = [start + timedelta(days=i) for i in range(config.HISTORY_DAYS_USAGE)]
    parts: dict[str, list[pd.DataFrame]] = defaultdict(list)

    def ix(d: date) -> int:
        i = (d - start).days
        if i < 0 or i >= len(days):
            raise ValueError(f"window day {d} outside the generated history {start}..{days[-1]}")
        return i

    for chunk_id, lo in enumerate(range(0, world.n, config.GENERATOR_CHUNK)):
        sl = slice(lo, min(lo + config.GENERATOR_CHUNK, world.n))
        daily = [world.daily(d, sl, chunk_id) for d in days]
        m = sl.stop - sl.start
        cum = {k: np.vstack([np.zeros((1, m)), np.cumsum(np.vstack([x[k] for x in daily]), axis=0)]) for k in daily[0]}

        def win(k: str, w: cal.Window) -> np.ndarray:
            return cum[k][ix(w.end) + 1] - cum[k][ix(w.start)]

        def span(k: str, a: date, b: date) -> np.ndarray:
            return cum[k][ix(min(b, days[-1])) + 1] - cum[k][ix(max(a, start))]

        c = world.c.iloc[sl]
        val, amid, acc = c["msisdn"].to_numpy(), c["access_method_id"].to_numpy(), c["account_id"].to_numpy()

        # activities_* (BR 4.2): one table per period and anchor
        for anchor_name, anchor in (("T", t_anchor), ("D", run_date)):
            for p, w in cal.activity_windows(anchor).items():
                cols = {
                    "ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc,
                    f"Call_Count_{p}": win("voice_calls", w), f"Data_Session_Count_{p}": win("data_sessions", w),
                    f"Voice_Traffic_min_{p}": win("voice_min_rounded", w), f"Data_Volume_MB_{p}": win("data_mb_rounded", w),
                    f"PG_Total_Amount_{p}": win("pg_all", w), f"PG_Voice_Amount_{p}": win("pg_voice", w),
                    f"PG_Data_Amount_{p}": win("pg_data", w), f"Onnet_Call_Count_{p}": win("onnet_calls", w),
                    f"Onnet_Voice_min_{p}": win("onnet_min", w), f"Offnet_Call_Count_{p}": win("offnet_calls", w),
                    f"Offnet_Voice_min_{p}": win("offnet_min", w), f"Local_Call_Count_{p}": win("local_calls", w),
                    f"Local_Voice_min_{p}": win("local_min", w), f"Active_Days_Count_{p}": win("any_day", w),
                    f"Active_Days_Voice_Count_{p}": win("voice_day", w), f"Active_Days_Data_Count_{p}": win("data_day", w),
                }
                present = (cols[f"Active_Days_Count_{p}"] > 0) | (np.abs(cols[f"PG_Total_Amount_{p}"]) > 0)
                parts[schemas.activity_table_name(p, anchor_name)].append(pd.DataFrame(cols)[present])

        # subs_amt_* (BR 4.1, SUBSCRIPTION_TOT_AMT): rows for lines with any subscription
        has_sub = span("sub_tot_amt", start, run_date) > 0
        for table, anchor in (("subs_amt_auto", t_anchor), ("subs_amt_score", run_date)):
            cols = {"ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc.astype(float)}
            cols.update({f"Sub_Amt_{p.lower()}": win("sub_tot_amt", w) for p, w in cal.subs_windows(anchor).items()})
            parts[table].append(pd.DataFrame(cols)[has_sub])

        # paygo_activity_auto (T, positive charges only)
        cols = {"ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc, "ACCESS_METHOD_VAL": val}
        cols.update({f"PG_Total_Amount_{p.lower()}": win("pg_pos", w) for p, w in cal.payg_windows(t_anchor).items()})
        present = span("pg_pos", start, t_anchor) > 0
        parts["paygo_activity_auto"].append(pd.DataFrame(cols)[present])

        # subscriptions_tg (D, SUBSCRIPTION_AMT, BR-17/18/19)
        cols = {"ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc.astype(float)}
        for p, w in cal.subscriptions_tg_windows(run_date).items():
            sfx = p.lower() if p in WEEKS else p
            if p in MONTHS:
                cols[f"Sub_Count_{p}"] = win("sub_count", w)
            cols[f"Total_Sub_Amt_{sfx}"] = win("sub_amt", w)
            cols[f"Data_Sub_Amt_{sfx}"] = win("sub_amt_data", w)
            cols[f"Voice_Sub_Amt_{sfx}"] = win("sub_amt_voice", w)
        present = span("sub_amt", run_date - timedelta(days=95), run_date - timedelta(days=1)) > 0
        parts["subscriptions_tg"].append(pd.DataFrame(cols)[present])

        # usage_data_voice_payg_tg (D, actual durations, positive PAYG, BR-33/34)
        cols = {"ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc, "ACCESS_METHOD_VAL": val}
        for p, w in cal.usage_tg_volume_windows(run_date).items():
            cols[f"local_Voice_min_{p}"] = win("local_min", w)
            cols[f"data_volume_mb_{p}"] = win("data_mb", w)
        cols.update({f"PG_Total_Amount_{p.lower()}": win("pg_pos", w) for p, w in cal.payg_windows(run_date).items()})
        present = span("any_day", run_date - timedelta(days=95), run_date - timedelta(days=2)) > 0
        parts["usage_data_voice_payg_tg"].append(pd.DataFrame(cols)[present])

    return {t: pd.concat(v, ignore_index=True) for t, v in parts.items()}


# ---------------------------------------------------------------------------
# Base and reference tables
# ---------------------------------------------------------------------------
def _catalog_strings(cat: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "btl_name": cat["btl_name"], "type": cat["type"], "generosity": cat["generosity"],
        "gb": cat["gb"].map(_fmt_num), "min": cat["min"].map(_fmt_num), "price": cat["price"].map(_fmt_num),
        "Price data": cat["price_data"].map(_fmt_num), "Price voice": cat["price_voice"].map(_fmt_num),
        "service": cat["service"], "validity": cat["validity"].map(str),
    })


def _offer_text(o) -> tuple[str, str]:
    bits_en, bits_ar = [], []
    if o["gb"]:
        bits_en.append(f"{o['gb']:g}GB data")
        bits_ar.append(f"{o['gb']:g} جيجابايت")
    if o["min"]:
        bits_en.append(f"{int(o['min'])} local minutes")
        bits_ar.append(f"{int(o['min'])} دقيقة محلية")
    en = f"{config.OPERATOR_NAME}: enjoy {' + '.join(bits_en)} for {o['price']:g} {config.CURRENCY}, valid {o['validity']} days. Reply YES to activate."
    ar = f"{config.OPERATOR_NAME}: استمتع بـ {' + '.join(bits_ar)} مقابل {o['price']:g} ر.ع لمدة {o['validity']} يوم. أرسل نعم للتفعيل."
    return en, ar


def _conflicts(cat: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (_, _), g in cat.sort_values("price").groupby(["service", "validity"]):
        g = g.reset_index(drop=True)
        for i in range(len(g) - 1):
            a, b = g.iloc[i], g.iloc[i + 1]
            if b["price"] - a["price"] > 1.5:
                continue
            rows.append({
                "Offer_A": a["btl_name"], "Gen_Level_A": a["generosity"], "Price_A": _fmt_num(a["price"]),
                "Data_A": _fmt_num(a["gb"]), "Min_A": _fmt_num(a["min"]), "Validity_A": str(a["validity"]),
                "Offer_Type_A": a["service"],
                "Offer_B": b["btl_name"], "Gen_Level_B": b["generosity"], "Price_B": _fmt_num(b["price"]),
                "Data_B": _fmt_num(b["gb"]), "Min_B": _fmt_num(b["min"]), "Validity_B": str(b["validity"]),
                "Offer_Type_B": b["service"],
                "Price_Difference": _fmt_num(round(b["price"] - a["price"], 3)),
                "Data_Difference": _fmt_num(round(b["gb"] - a["gb"], 3)),
                "SAME_VALIDITY": "Y", "SAME_OFFER_TYPE": "Y",
            })
    return pd.DataFrame(rows)


def _base_tables(world: World, run_date: date, rng: np.random.Generator) -> dict[str, pd.DataFrame]:
    c = world.c
    act = [(run_date - timedelta(days=int(t))).isoformat() for t in c["tenure"]]
    month, year = cal.location_month(run_date)
    party = pd.DataFrame({"party_id": c["party_id"], "AGE": c["age"], "GENDER": c["gender"], "NATIONALITY": c["nationality"],
                          "nationality_identifier": np.where(c["nationality"] == "Oman", "OMANI", "EXPAT")})
    loc = pd.DataFrame({"Access_Method_id": c["access_method_id"], "Calling_Month": month, "Calling_Year": year,
                        "REGION": c["region"], "WILAYAT": c["wilayat"], "AREA": c["area"]})
    extra = loc.sample(frac=0.03, random_state=int(rng.integers(1 << 30))).copy()     # BR-14: several locations
    extra["AREA"] = [f"AREA_{x:04d}" for x in rng.integers(400, 800, len(extra))]
    cat = world.catalog
    scripts = [_offer_text(o) for _, o in cat.iterrows()]
    dv = c.sample(frac=0.92, random_state=int(rng.integers(1 << 30)))
    return {
        "base_prepaid": pd.DataFrame({
            "UBSCRIPTION_ID": c["subscription_id"], "ACCESS_METHOD_VAL": c["msisdn"], "ACCESS_METHOD_ID": c["access_method_id"],
            "ACCOUNT_ID": c["account_id"], "PARTY_ID": c["party_id"], "access_method_subs_start_dt": act,
            "CURRENT_PLAN": c["plan"]}),
        "consumer_aact_account": pd.DataFrame({"ACCOUNT_ID": c["account_id"], "PARTY_ID": c["party_id"]}),
        "access_method_id_auto": pd.DataFrame({"ACCESS_METHOD_ID": c["access_method_id"], "MSISDN": c["msisdn"],
                                               "ACTIVATION_DATE": act}),
        "party_auto": party,
        "party_scoring": party.copy(),
        "location_auto": pd.concat([loc, extra], ignore_index=True),
        "btl_catalog": _catalog_strings(cat),
        "BTL_script_english_arabic_ok": pd.DataFrame({
            "Sl No.": [str(i + 1) for i in range(len(cat))], "destination_offer_name": cat["btl_name"],
            "English Script": [s[0] for s in scripts], "Arabic Script": [s[1] for s in scripts]}),
        "conflicting_offers_h": _conflicts(cat),
        "DEVICEBEHAVIORSEGMENTATION.dv_segment": pd.DataFrame({"ACCESS_METHOD_ID": dv["access_method_id"],
                                                                "DV_SEGMENT": dv["dv_segment"]}),
    }


# ---------------------------------------------------------------------------
# Model scores (one shared structure)
# ---------------------------------------------------------------------------
def _life_range(t: np.ndarray) -> np.ndarray:
    return np.select([t <= 30, t <= 90, t <= 180, t <= 365, t <= 730], ["0-30", "31-90", "91-180", "181-365", "1Y-2Y"], "2Y+")


def generate_model_score(world: World, m: config.ModelScore, rng: np.random.Generator, day: date | None = None) -> pd.DataFrame:
    """Any model file in MODEL_SCORE_SCHEMA: driven by the matching hidden signal + noise, partial coverage."""
    t = world.c["tenure"].to_numpy()
    mask = np.ones(world.n, bool) if m.tenure_range is None else (t >= m.tenure_range[0]) & (t <= m.tenure_range[1])
    cand = np.flatnonzero(mask)
    k = int(len(cand) * rng.uniform(*m.coverage))
    idx = np.sort(rng.choice(cand, size=k, replace=False)) if k else np.array([], dtype=int)
    p1 = np.round(world.score(m.signal or m.feature.lower(), rng, idx, day), 6) if k else np.array([])
    c = world.c.iloc[idx]
    return pd.DataFrame({
        "subscription_id": c["subscription_id"].to_numpy(), "access_method_val": c["msisdn"].astype("int64").to_numpy(),
        "access_method_id": c["access_method_id"].to_numpy(), "tenure": c["tenure"].to_numpy(),
        "life_in_nw_range": _life_range(c["tenure"].to_numpy()), "proba_0": np.round(1 - p1, 6), "proba_1": p1,
        "prediction": np.where(p1 >= 0.5, "1", "0"),
    })


# ---------------------------------------------------------------------------
# Segments, last week's campaign, purchase history
# ---------------------------------------------------------------------------
def _segments(world: World, rng: np.random.Generator) -> np.ndarray:
    c = world.c
    churn = world.score("churn", rng)
    react = world.score("reactivation_30", rng)
    da, va = world.score("data_appetite", rng), world.score("voice_appetite", rng)
    frag = np.maximum(world.score("fragility_data", rng), world.score("fragility_voice", rng))
    tenure, usage = c["tenure"].to_numpy(), c["usage_type"].to_numpy()
    low = (c["churn_day"].to_numpy() <= 0) | (usage == "inactive")
    seg = np.select(
        [(tenure < config.NURSERING_MAX_TENURE_DAYS) & (churn >= 0.5), tenure < config.NURSERING_MAX_TENURE_DAYS,
         (tenure < config.ONBOARDING_MAX_TENURE_DAYS) & (churn >= 0.5), tenure < config.ONBOARDING_MAX_TENURE_DAYS,
         low & (react < 0.3), low & (react < 0.6), low,
         churn >= 0.6, frag >= 0.55, usage == "dual",
         (usage == "data_only") & (va >= 0.4), usage == "data_only", da >= 0.4],
        ["Nursering_Early_Churn_At_Risk", "Nursering_Early_Churn_Not_At_Risk",
         "Onboarding_Early_Churn_At_Risk", "Onboarding_Early_Churn_Not_At_Risk",
         "Retention_Low_Probability_To_Reconnect", "Retention_Med_Probability_To_Reconnect",
         "Retention_High_Probability_To_Reconnect",
         "Retention_RGS_At_Risk", "Retention_RGS_Fragile", "RGS_Not_At_Risk_DualPlay_Upgrade",
         "RGS_Not_At_Risk_Data_Only_Voice_Appetency", "RGS_Not_At_Risk_Data_Only_No_Voice_Appetency",
         "RGS_Not_At_Risk_Voice_Only_Data_Appetency"],
        "RGS_Not_At_Risk_Voice_Only_No_Data_Appetency").astype(object)
    # production emits both spellings ("RGS_..." and "Rgs_..."): mix them
    mixed = (rng.random(len(seg)) < 0.5) & np.array(["RGS" in s for s in seg])
    seg[mixed] = [s.replace("RGS", "Rgs").replace("DualPlay", "Dualplay") for s in seg[mixed]]
    return seg


def _preference_weights(world: World, idx: np.ndarray, cat: pd.DataFrame) -> np.ndarray:
    c = world.c.iloc[idx]
    da, va = c["data_appetite"].to_numpy()[:, None], c["voice_appetite"].to_numpy()[:, None]
    svc = cat["service"].to_numpy()[None, :]
    fit = np.where(svc == "DATA", da, np.where(svc == "VOICE", va, 0.5 * (da + va)))
    gen = np.vectorize(GENEROSITY_BONUS.get)(cat["generosity"].to_numpy())[None, :]
    validity = cat["validity"].to_numpy()[None, :]
    spend = c["spend"].to_numpy()[:, None]
    anchor = np.where(validity == 7, spend / 4.3, spend)
    gap = np.maximum(0, cat["price"].to_numpy()[None, :] - anchor) / np.maximum(anchor, 0.5)
    w = np.exp(1.4 * fit + 0.5 * gen - 0.8 * c["price_sensitivity"].to_numpy()[:, None] * gap)
    # add-on families: roaming is bought by travellers, international packs by people who call abroad
    travel = np.where(c["traveller"].to_numpy(), 1.0, 0.01)[:, None]
    intl = np.clip(c["intl_share"].to_numpy() * 6, 0.01, 2.0)[:, None]
    w = w * np.where(svc == "ROAMING", travel, 1.0) * np.where(svc == "INTERNATIONAL", intl, 1.0)
    w = w * (~cat["is_new"].to_numpy())[None, :]                              # new offers: no history (cold start)
    return w.astype(np.float32)


def _sample_offers(rng: np.random.Generator, weights: np.ndarray) -> np.ndarray:
    cdf = np.cumsum(weights / weights.sum(axis=1, keepdims=True), axis=1)
    return np.minimum((cdf < rng.random(len(weights))[:, None]).sum(axis=1), weights.shape[1] - 1)


def _legacy_offers(world: World, idx: np.ndarray, actions: np.ndarray) -> list[np.ndarray]:
    """Offers last week's campaign sent: production constraint arm with all-L2 (BR-66 active), the
    cheapest family offer above the customer's spend, per validity -> up to 4 destination offers."""
    cat = world.catalog[~world.catalog["is_new"]]
    spend = world.c["spend"].to_numpy()[idx]
    slots = [np.full(len(idx), None, dtype=object) for _ in range(4)]
    for action, (svc_a, svc_b) in config.ACTION_FAMILIES.items():
        sel = np.flatnonzero(actions == action)
        if not len(sel):
            continue
        for v_i, validity in enumerate((30, 7)):
            anchor = spend[sel] / (4.3 if validity == 7 else 1.0)
            for s_i, svc in enumerate((svc_a, svc_b)):
                fam = cat[(cat["service"] == svc) & (cat["validity"] == validity) & (cat["generosity"] == "L2")].sort_values("price")
                if fam.empty:
                    continue
                prices = fam["price"].to_numpy()
                pos = np.minimum(np.searchsorted(prices, anchor, side="right"), len(prices) - 1)
                slots[v_i * 2 + s_i][sel] = fam["btl_name"].to_numpy()[pos]
    return slots


def _ts(rng: np.random.Generator, days: list[date]) -> list[str]:
    secs = rng.integers(0, 86_400, len(days))
    return [(datetime.combine(d, datetime.min.time()) + timedelta(seconds=int(s))).isoformat() for d, s in zip(days, secs)]


def _our_campaign(world: World, campaign: pd.DataFrame, segments: np.ndarray):
    """Last week's campaign as logged by the app (campaign_log rows with offers): who, which action, which
    offers, which day -> the same arrays as the simulated legacy campaign."""
    pos = {m: i for i, m in enumerate(world.c["msisdn"].astype(str))}
    sent = campaign[campaign["offers"].map(lambda o: isinstance(o, list) and len(o) > 0)]
    sent = sent[sent["msisdn"].astype(str).isin(pos)].drop_duplicates("msisdn")
    targeted = np.array([pos[str(m)] for m in sent["msisdn"]], dtype=int)
    actions = sent["action"].astype(str).to_numpy(dtype=object)
    date1 = [date.fromisoformat(str(d)) for d in sent["run_date"]]
    slots = [np.full(len(sent), None, dtype=object) for _ in range(4)]
    for r, offers in enumerate(sent["offers"]):
        for i, o in enumerate(offers[:4]):
            slots[i][r] = o["code"]
    return targeted, segments[targeted].copy(), actions, date1, slots


def _campaign_tables(world: World, run_date: date, segments: np.ndarray, rng: np.random.Generator,
                     campaign: pd.DataFrame | None = None) -> dict[str, pd.DataFrame]:
    """Last week's campaign (targeting, responses, organic purchases, history). campaign = the app's own
    log of last week (closed-loop simulation); None = a simulated production campaign."""
    c = world.c
    n = world.n
    cat = world.catalog.set_index("btl_name")
    scripts = {name: _offer_text(row) for name, row in world.catalog.set_index("btl_name", drop=False).iterrows()}
    if campaign is not None:
        targeted, seg, actions, date1, slots = _our_campaign(world, campaign, segments)
    else:
        targeted = np.flatnonzero(rng.random(n) < config.TARGETED_SHARE)
        seg = segments[targeted].copy()
        seg[rng.random(len(seg)) < 0.01] = None                              # BR-37 rows without segment
        lower = np.array([s.lower() if s else "" for s in seg])
        actions = np.array([config.SEGMENT_TO_ACTION.get(s, "UPSELL" if s else "") for s in lower], dtype=object)
        d1_off = rng.integers(config.TARGETING_WINDOW[0], config.TARGETING_WINDOW[1] + 1, len(targeted))
        date1 = [run_date + timedelta(days=int(o)) for o in d1_off]
        slots = _legacy_offers(world, targeted, actions)

    targeting = pd.DataFrame({"date1": [d.isoformat() for d in date1], "access_method_id": c["access_method_id"].to_numpy()[targeted],
                              "account_id": c["account_id"].to_numpy()[targeted], "segment": seg})
    for i in range(6):
        targeting[f"destination_offer_name_{i + 1}"] = slots[i] if i < 4 else None

    # acceptances among the targeted (hidden response function)
    probs = np.zeros((len(targeted), 4))
    for i, s in enumerate(slots):
        ok = np.array([x is not None for x in s])
        if ok.any():
            o = cat.loc[s[ok]]
            probs[ok, i] = world.accept_probability(targeted[ok], actions[ok].astype(str), o["service"].to_numpy(),
                                                    o["generosity"].to_numpy(), o["price"].to_numpy(),
                                                    o["validity"].to_numpy(), day=run_date)
    p_any = 1 - np.prod(1 - probs, axis=1)
    takers = np.flatnonzero(rng.random(len(targeted)) < p_any)
    fulfil_rows = []
    btl_names = world.catalog.loc[~world.catalog["is_new"], "btl_name"].to_numpy()
    for t in takers:
        p = probs[t] / probs[t].sum()
        product = slots[int(rng.choice(4, p=p))][t]
        if rng.random() < 0.05:                                              # took another BTL than the one offered
            product = rng.choice(btl_names)
        first = max(date1[t], run_date + timedelta(days=config.REWARD_WINDOW[0]))
        span = (run_date + timedelta(days=config.REWARD_WINDOW[1]) - first).days
        fulfil_rows.append((targeted[t], product, first + timedelta(days=int(rng.integers(0, span + 1)))))

    # organic BTL purchases and non-BTL noise, D-7..D-1: every line that did not take a campaign offer -- being
    # contacted must not remove a customer's own buying (else contacted arms fall below the control)
    not_takers = np.setdiff1d(np.arange(n), targeted[takers])
    organic = not_takers[rng.random(len(not_takers)) < world.organic_probability(not_takers)]
    if len(organic):
        picks = _sample_offers(rng, _preference_weights(world, organic, world.catalog))
        for i, j in zip(organic, picks):
            fulfil_rows.append((i, world.catalog["btl_name"].iat[j], run_date - timedelta(days=int(rng.integers(1, 8)))))
    for i in np.flatnonzero(rng.random(n) < NON_BTL_SHARE):
        fulfil_rows.append((i, rng.choice(NON_BTL_PRODUCTS), run_date - timedelta(days=int(rng.integers(1, 8)))))

    f_idx = np.array([r[0] for r in fulfil_rows], dtype=int)
    fulfil = pd.DataFrame({"access_method_val": c["msisdn"].to_numpy()[f_idx], "PRODUCTNAME": [r[1] for r in fulfil_rows],
                           "FULFILMENTSUCCESSDATE_TIME": _ts(rng, [r[2] for r in fulfil_rows]), "sub_flag": 1, "nb_sub": 1})
    fulfil = fulfil.groupby(["access_method_val", "PRODUCTNAME", "FULFILMENTSUCCESSDATE_TIME"], as_index=False).agg(
        sub_flag=("sub_flag", "max"), nb_sub=("nb_sub", "sum"))

    # arbitration table (BR-22): a subset of the targeted, start_date in D-7..D
    arb = np.flatnonzero(rng.random(len(targeted)) < 0.35)
    start = [run_date + timedelta(days=int(o)) for o in rng.integers(config.ARBITRATION_WINDOW[0], config.ARBITRATION_WINDOW[1] + 1, len(arb))]
    arb_df = pd.DataFrame({"DATE1": [d.isoformat() for d in start], "START_DATE": [d.isoformat() for d in start],
                           "END_DATE": [(d + timedelta(days=6)).isoformat() for d in start],
                           "MSISDN": c["msisdn"].to_numpy()[targeted[arb]]})
    for i in range(1, 7):
        names = slots[i - 1][arb] if i <= 4 else np.full(len(arb), None, dtype=object)
        arb_df[f"CHANNEL_{i}"] = np.where([x is not None for x in names], "SMS", None)
        arb_df[f"OPPORTUNITY_NAME_{i}"] = [f"OPP_{a.replace(' ', '_')}" if x is not None else None for a, x in zip(actions[arb], names)]
        arb_df[f"DESTINATION_OFFER_NAME_{i}"] = names
        arb_df[f"OFFER_SCRIPT_EN_{i}"] = [scripts[x][0] if x is not None else None for x in names]
        arb_df[f"OFFER_SCRIPT_AR_{i}"] = [scripts[x][1] if x is not None else None for x in names]
    arb_df["FLAG_SPLIO"] = "N"
    arb_df["SEGMENT"] = seg[arb]
    arb_df["FILENAME"] = [f"arbitration_{d.isoformat()}.csv" for d in start]
    arb_df["ETL_CREATE_DT"] = [d.isoformat() for d in start]
    arb_df["JOB_ID"] = 10_000 + (run_date.toordinal() % 10_000)
    arb_df["SEGMENT_BTL_TAKER"] = rng.choice(["Y", "N"], len(arb))
    arb_df["SEGMENT_BTL_TAKER_UPSELL"] = rng.choice(["Y", "N"], len(arb))

    # 270-day purchase history (BR-24) for the ranker: preference-driven BTL purchases + last week's
    usage = c["usage_type"].to_numpy()
    lam = 3.0 * (0.4 + np.clip(c["spend"].to_numpy() / 8, 0, 1.2)) * np.where(usage == "inactive", 0.3, 1.0)
    counts = rng.poisson(lam)
    hist_rows = []
    for lo in range(0, n, config.GENERATOR_CHUNK):
        sel = np.arange(lo, min(lo + config.GENERATOR_CHUNK, n))
        ev = np.repeat(sel, counts[sel])
        if len(ev):
            w = _preference_weights(world, sel, world.catalog)
            picks = _sample_offers(rng, w[ev - lo])
            hist_rows.append(pd.DataFrame({"i": ev, "PRODUCTNAME": world.catalog["btl_name"].to_numpy()[picks]}))
    hist = pd.concat(hist_rows + [pd.DataFrame({"i": f_idx, "PRODUCTNAME": [r[1] for r in fulfil_rows]})], ignore_index=True)
    hist["access_method_val"] = c["msisdn"].to_numpy()[hist["i"].to_numpy()]
    history = hist.groupby(["access_method_val", "PRODUCTNAME"], as_index=False).size().rename(columns={"size": "nb_sub"})
    history["sub_flag"] = 1
    return {"targeting_dataiku_auto": targeting, "targeting_auto": arb_df, "subs_btl_auto": fulfil,
            "sub_btl_3months": history[["access_method_val", "PRODUCTNAME", "sub_flag", "nb_sub"]]}


# ---------------------------------------------------------------------------
# Test sample (BR-59..62): control + 3 arms, stratified
# ---------------------------------------------------------------------------
def _bucket(x: np.ndarray, edges: list[float], labels: list[str]) -> np.ndarray:
    return np.array(labels, dtype=object)[np.searchsorted(edges, x, side="left")]


def _arm_pattern(length: int, offset: int = 0, arms: list[str] | None = None) -> list[str]:
    """Interleaved arm sequence respecting ARM_SHARES over the chosen arms (stride scheduling). A random
    offset per stratum keeps small strata from all starting with the same arm."""
    shares = {a: s for a, s in config.ARM_SHARES.items() if a in (arms or config.ARMS)}
    total = sum(shares.values())
    marks = []
    for arm, share in ((a, s / total) for a, s in shares.items()):
        k = max(1, round(share * 100))
        marks += [((i + 0.5) / k, arm) for i in range(k)]
    seq = [a for _, a in sorted(marks)]
    return [seq[(offset + i) % len(seq)] for i in range(length)]


def _sample_table(world: World, usage: dict[str, pd.DataFrame], rng: np.random.Generator) -> pd.DataFrame:
    c = world.c
    eligible = c[~c["plan"].isin(config.EXCLUDED_PLANS)]
    k = int(len(eligible) * config.TARGET_SAMPLE_SHARE)
    s = eligible.sample(n=k, random_state=int(rng.integers(1 << 30))).copy()
    sub = usage["subscriptions_tg"].set_index("ACCESS_METHOD_VAL")
    use = usage["usage_data_voice_payg_tg"].set_index("ACCESS_METHOD_VAL")

    def col(frame, name):
        return frame[name].reindex(s["msisdn"]).fillna(0.0).to_numpy()

    sub_m1, payg_m1 = col(sub, "Total_Sub_Amt_M1"), col(use, "PG_Total_Amount_m1")
    total = sub_m1 + payg_m1
    total_3m = sum(col(sub, f"Total_Sub_Amt_M{i}") + col(use, f"PG_Total_Amount_m{i}") for i in (1, 2, 3))
    voice, data = col(use, "local_Voice_min_p1"), col(use, "data_volume_mb_p1")
    nat_group = np.where(s["nationality"].isin(config.NATIONALITY_KEEP), s["nationality"], config.NATIONALITY_OTHERS)
    rev_band = _bucket(total_3m, [0, 5, 15, 40], ["0", "0-5", "5-15", "15-40", "40+"])
    voice_g = _bucket(voice, [0, 100, 500], ["0", "1-100", "100-500", "500+"])
    data_g = _bucket(data, [0, 1000, 10000], ["0", "1-1000", "1000-10000", "10000+"])
    strata = np.array(["|".join(map(str, t)) for t in zip(nat_group, rev_band, voice_g, data_g, s["plan"])], dtype=object)

    group = np.empty(len(s), dtype=object)
    counts = np.empty(len(s), dtype=object)
    for st in pd.unique(strata):
        pos = np.flatnonzero(strata == st)
        rng.shuffle(pos)
        group[pos] = _arm_pattern(len(pos), int(rng.integers(0, 100)), world.params.get("arms"))
        counts[pos] = str(len(pos))

    def f3(x):
        return [f"{v:.3f}" for v in x]

    return pd.DataFrame({
        "access_method_val": s["msisdn"].to_numpy(), "access_method_id": s["access_method_id"].astype(str).to_numpy(),
        "account_id": s["account_id"].astype(str).to_numpy(), "tenure": s["tenure"].astype(str).to_numpy(),
        "activation_date": [(world.as_of - timedelta(days=int(t))).isoformat() for t in s["tenure"]],
        "plan_name": s["plan"].to_numpy(), "payg_revenue": f3(payg_m1), "subscription_revenue": f3(sub_m1),
        "total_revenue": f3(total), "total_revenue_3m": f3(total_3m), "voice_traffic": f3(voice), "data_volume": f3(data),
        "nationality": s["nationality"].to_numpy(), "region": s["region"].to_numpy(), "ARPU": f3(total),
        "ARPU_3m": f3(total_3m / 3), "voice_trf_group": voice_g, "data_trf_group": data_g,
        "Final": np.where(group == "cg", "CG", "TG"), "channel": "SMS", "group": group, "strata": strata, "counts": counts,
    })


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------
def _profile_table(world: World) -> pd.DataFrame:
    """customer_profile: msisdn, age, tenure. Same customers, ages and tenures as the other tables."""
    c = world.c
    tenure = c["tenure"].to_numpy()
    if config.PROFILE_TENURE_UNIT == "months":
        tenure = tenure // 30
    return pd.DataFrame({"msisdn": c["msisdn"].to_numpy(), "age": c["age"].to_numpy(), "tenure": tenure})


def _consent_table(world: World, run_date: date) -> pd.DataFrame:
    """V1.0: ~3 % do-not-contact, ~2 % marketing opt-out, fixed per customer. Own random stream, so every other
    table is identical to the one generated without consent data (same world for V0 / V1.0 comparisons)."""
    rng = np.random.default_rng(np.random.SeedSequence([world.seed, 9]))
    n = world.n
    return pd.DataFrame({"access_method_val": world.c["msisdn"].to_numpy(), "dnd": (rng.random(n) < 0.03).astype(int),
                         "marketing_opt_in": (rng.random(n) >= 0.02).astype(int),
                         "updated_at": [(run_date - timedelta(days=int(d))).isoformat() for d in rng.integers(1, 400, n)]})


def generate_run(world: World, run_date: date, campaign: pd.DataFrame | None = None) -> GeneratedRun:
    """Builds every table of the contract for run date D from the world. campaign: the app's logged campaign
    of D-7 (closed-loop simulation); its responses become this run's fulfilments."""
    t0 = time.time()
    rng = np.random.default_rng(np.random.SeedSequence([world.seed, 5, run_date.toordinal()]))
    typed: dict[str, pd.DataFrame] = {}
    typed.update(_base_tables(world, run_date, rng))
    usage = _usage_tables(world, run_date)
    typed.update(usage)
    for k, m in enumerate(config.MODEL_SCORE_REGISTRY):
        typed[m.table] = generate_model_score(world, m, np.random.default_rng(np.random.SeedSequence([world.seed, 3, k])), run_date)
    typed.update(_campaign_tables(world, run_date, _segments(world, rng), rng, campaign))
    typed[config.TARGET_SAMPLE_TABLE] = _sample_table(world, usage, rng)
    typed[config.CONSENT_TABLE] = _consent_table(world, run_date)
    typed[config.PROFILE_TABLE] = _profile_table(world)

    missing = set(schemas.TABLES) - set(typed)
    if missing:
        raise RuntimeError(f"generator did not build: {sorted(missing)}")
    tables = {name: to_raw(typed[name], name) for name in schemas.TABLES}
    issues = scen.inject_quality_issues(tables, world.seed) if world.has("data_quality_issues") else []

    manifest = {
        "run_date": run_date.isoformat(),
        "anchor_date": cal.anchor_date(run_date).isoformat(),
        "source": "generator",
        "operator": config.OPERATOR_NAME,
        "schema_version": config.SCHEMA_VERSION,
        "generator": {"n": world.n, "seed": world.seed, "scenarios": world.scenarios, "world_as_of": world.as_of.isoformat(),
                      "catalog_size": len(world.catalog), "new_offers": int(world.catalog["is_new"].sum())},
        "row_counts": {name: len(df) for name, df in tables.items()},
        "quality_issues_injected": issues,
        "windows": cal.all_windows(run_date),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "generation_seconds": round(time.time() - t0, 2),
    }
    return GeneratedRun(run_date, cal.anchor_date(run_date), tables, manifest, world)


WRITE_FORMATS = ("csv.gz", "json", "jsonl", "csv", "parquet")   # csv.gz = production (V3.0)
LAYOUTS = ("split", "folders", "flat")         # split = V4.0 data/training/<D> + data/scoring/<D>


def _arrow_frame(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Raw frame -> typed columns for Parquet: Int64 / float64 / string by raw type. A column holding a value of
    the wrong type (data_quality_issues) is written as text, so the reader sees the same defect as in JSON."""
    types = schemas.TABLES[table].types()
    out = {}
    for c in df.columns:
        rt = types.get(c, "string")
        try:
            if rt in schemas.INT_TYPES:
                out[c] = pd.array(df[c].tolist(), dtype="Int64")
            elif rt == "double":
                out[c] = pd.array(df[c].tolist(), dtype="Float64")
            else:
                raise TypeError
        except (TypeError, ValueError):
            out[c] = pd.array([None if v is None else str(v) for v in df[c].tolist()], dtype="string")
    return pd.DataFrame(out, columns=df.columns)


def write_table(df: pd.DataFrame, table: str, path: Path, fmt: str) -> Path:
    """One raw frame -> one file in the given format."""
    if fmt == "csv.gz":
        from nbx.fsdr import csvstd

        csvstd.write_csv(df, path)                      # FSDR CSV standard + <table>.schema.json
    elif fmt == "parquet":
        _arrow_frame(df, table).to_parquet(path, index=False)
    elif fmt == "csv":
        df.to_csv(path, index=False, encoding="utf-8")
    else:
        records = df.to_dict("records")
        with path.open("w", encoding="utf-8") as fh:
            if fmt == "json":
                json.dump(records, fh, ensure_ascii=False, allow_nan=False)
            else:
                for r in records:
                    fh.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    return path


def write_run(run: GeneratedRun, out_dir: Path, fmt: str = "json", layout: str = "folders") -> Path:
    """One file per table + _manifest.json. layout "folders": <out_dir>/<input folder>/<table>.<fmt> (plan §13);
    "flat": every file in out_dir (V1.0)."""
    if fmt not in WRITE_FORMATS:
        raise ValueError(f"fmt must be one of {WRITE_FORMATS}")
    if layout not in LAYOUTS:
        raise ValueError(f"layout must be one of {LAYOUTS}")
    out_dir = Path(out_dir)
    if layout == "split":
        return _write_split(run, out_dir, fmt)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df in run.tables.items():
        folder = out_dir / schemas.TABLES[name].folder if layout == "folders" else out_dir
        folder.mkdir(exist_ok=True)
        write_table(df, name, folder / f"{name}.{fmt}", fmt)
    manifest = {**run.manifest, "format": fmt, "layout": layout,
                "folders": ({t: schemas.TABLES[t].folder for t in run.tables} if layout == "folders" else {})}
    (out_dir / config.MANIFEST_FILE).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return out_dir


def split_twins(training_dir: Path) -> tuple[Path, Path]:
    """data/training/<D> -> (data/training/<D>, data/scoring/<D>). Any other folder name: <folder>/training/<D> is
    not guessed, the caller passes the training folder of the V4.0 layout."""
    training_dir = Path(training_dir)
    if training_dir.parent.name != config.TRAINING_FOLDER:
        raise ValueError(f"split layout: write to <data>/{config.TRAINING_FOLDER}/<D>, got {training_dir}")
    return training_dir, training_dir.parent.parent / config.SCORING_FOLDER / training_dir.name


def _write_split(run: GeneratedRun, training_dir: Path, fmt: str) -> Path:
    """V4.0: the training tables under data/training/<D>/<folder>/, the scoring tables under data/scoring/<D>/<folder>/
    (config.SCORING_TABLES; btl_catalog in both), the same 5 folders on both sides, one _manifest.json in each."""
    twins = dict(zip((config.TRAINING_FOLDER, config.SCORING_FOLDER), split_twins(training_dir)))
    for root in twins.values():
        for f in config.INPUT_FOLDERS:
            (root / f).mkdir(parents=True, exist_ok=True)
    placed: dict[str, list[str]] = {k: [] for k in twins}
    for name, df in run.tables.items():
        sides = list(twins) if name in config.BOTH_TABLES else [config.table_side(name)]
        for side in sides:
            folder = twins[side] / (schemas.TABLES[name].folder if name in schemas.TABLES else "others")
            write_table(df, name, folder / f"{name}.{fmt}", fmt)
            placed[side].append(name)
    for side, root in twins.items():
        manifest = {**run.manifest, "format": fmt, "layout": "split", "side": side,
                    "twin": str(twins[config.SCORING_FOLDER if side == config.TRAINING_FOLDER else config.TRAINING_FOLDER]),
                    "folders": {t: schemas.TABLES[t].folder if t in schemas.TABLES else "others" for t in placed[side]}}
        (root / config.MANIFEST_FILE).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return training_dir


def generate(n: int = config.GENERATOR_DEFAULT_N, run_date: date | None = None, scenarios=None,
             seed: int = config.GENERATOR_SEED, out_dir: Path | None = None, fmt: str = "json",
             layout: str = "folders") -> GeneratedRun:
    """Convenience entry point (CLI / API): new world as of D, generate, optionally write."""
    run_date = run_date or date.today()
    world = World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=run_date)
    run = generate_run(world, run_date)
    if out_dir is not None:
        write_run(run, out_dir, fmt, layout)
    return run


def generate_weeks(n: int, first_date: date, weeks: int, scenarios=None, seed: int = config.GENERATOR_SEED,
                   out_root: Path | None = None, fmt: str = "json", arms: list[str] | None = None,
                   layout: str = "folders") -> list[GeneratedRun]:
    """Consecutive weekly runs D, D+7, ... from ONE world anchored on the last run date, so the shared days
    agree (W2 of D+7 = W1 of D) and scenario drifts grow week after week. Writes <out_root>/<date>/."""
    if weeks < 1:
        raise ValueError("weeks must be >= 1")
    dates = [first_date + timedelta(days=7 * k) for k in range(weeks)]
    world = World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=dates[-1], arms=arms)
    runs = [generate_run(world, d) for d in dates]
    if out_root is not None:
        for run in runs:
            write_run(run, Path(out_root) / run.run_date.isoformat(), fmt, layout)
    return runs


## Part B · The steps

The engine is loaded. Each step below calls it and shows what it wrote.

In [ ]:
# --- paths and helpers (do not edit) ---------------------------------------------
import json, os, sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:      # works from notebooks/ or the repo root
    ROOT = ROOT.parent
os.chdir(ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

config, csvstd, steps = use("nbx.fsdr", "config", "csvstd", "steps")

pd.set_option("display.max_columns", 80, "display.width", 220, "display.max_colwidth", 60)
D = date.fromisoformat(RUN_DATE)
TRAINING, SCORING = config.training_folder(RUN_DATE), config.scoring_folder(RUN_DATE)
RUN, CACHE, OUTPUT = config.FSDR_RUNS_DIR / RUN_DATE, config.FSDR_CACHE_DIR / RUN_DATE, config.output_folder(RUN_DATE)


def show(obj, n=PREVIEW_ROWS, title=None):
    """A table (DataFrame, list of dicts, dict or CSV / Parquet / csv.gz path), first n rows."""
    if isinstance(obj, (str, Path)):
        p = Path(obj)
        title = title or (str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p))
        if not p.exists() or p.stat().st_size == 0:
            print(f"{title}: empty or not written")
            return
        try:
            obj = (csvstd.read_csv(p) if p.name.endswith(".csv.gz") else pd.read_parquet(p) if p.suffix == ".parquet"
                   else pd.read_csv(p) if p.suffix == ".csv" else json.loads(p.read_text(encoding="utf-8")))
        except pd.errors.EmptyDataError:
            print(f"{title}: no rows")
            return
    if isinstance(obj, dict):
        obj = pd.DataFrame([{"key": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v, default=str)[:150]}
                            for k, v in obj.items()])
    if isinstance(obj, list):
        obj = pd.DataFrame(obj)
    if title:
        display(Markdown(f"**{title}** · {len(obj):,} rows x {obj.shape[1]} columns"))
    display(obj.head(n))


def tree(folder, depth=3, skip_schemas=True):
    """The files of a folder, as a tree with sizes."""
    folder = Path(folder)
    if not folder.exists():
        print(f"{folder}: does not exist yet")
        return
    print(folder)
    for p in sorted(folder.rglob("*")):
        rel = p.relative_to(folder)
        if len(rel.parts) > depth or (skip_schemas and p.name.endswith(".schema.json")):
            continue
        pad = "    " * (len(rel.parts) - 1)
        print(f"{pad}{p.name}{'/' if p.is_dir() else f'  ({p.stat().st_size / 1e3:,.1f} kB)'}")


def step_status(phase_dir, name):
    meta = steps.read_step(Path(phase_dir) / name)
    print(f"{name}: {meta.get('status', 'not run')}  rows {meta.get('rows', {})}  {meta.get('seconds', '-')} s"
          + (f"  reason: {meta['reason']}" if meta.get("reason") else ""))
    return meta


print(f"run date : {RUN_DATE}")
for name, p in (("training", TRAINING), ("scoring", SCORING), ("runs", RUN), ("cache", CACHE), ("output", OUTPUT)):
    print(f"{name:9s}: {p}  ({'found' if p.exists() else 'not yet'})")
print(f"offers per line: {config.RECO_OFFERS}  (FSDR_RECO_OFFERS)")

## Step 0 · Dummy data (optional)

The generator writes the 47 tables of run date *D* as `.csv.gz`, the FSDR CSV standard: gzip, UTF-8, IDs kept as text.

- The training tables (base, usage history, model scores) go to `data/training/<D>/`.
- The scoring tables of the week go to `data/scoring/<D>/`: `subscriptions_tg`, `usage_data_voice_payg_tg`, `sub_btl_3months`, `targeting_auto`, the catalogue, the scripts, the conflicts, consent, and the `*_score` tables.
- `btl_catalog` is written to both sides.

The manifest says `source: generator`, so real exports are never overwritten: with real files in the folder and `GENERATE = True`, this cell stops. With real data, set `GENERATE = False` and copy your files into the folders (see **How to add new data** above).

In [ ]:
if GENERATE:
    generator = use("nbx.fsdr.synth", "generator")

    man = TRAINING / config.MANIFEST_FILE
    if TRAINING.exists() and any(TRAINING.iterdir()) and (not man.exists() or json.loads(man.read_text())["source"] != "generator"):
        raise SystemExit(f"{TRAINING} holds real data: set GENERATE = False")
    run = generator.generate(N_LINES, D, ["baseline"], config.GENERATOR_SEED, TRAINING, "csv.gz", "split")
    print(f"{len(run.tables)} tables generated for {N_LINES:,} lines")
else:
    if not any(p.is_file() for p in TRAINING.rglob("*.csv*")) if TRAINING.exists() else True:
        raise SystemExit(f"no exports in {TRAINING}: create the folders with `uv run python run.py fsdr-folders "
                         f"--date {RUN_DATE} --layout split`, copy the files, then run again (or GENERATE = True for a test)")
    print("GENERATE = False: using the files already in", TRAINING, "and", SCORING)

## Step 1 · The two input folders

Every table lives in one of the 5 sub-folders. Its side (training or scoring) is set by `config.SCORING_TABLES`.
The table below lists, for each table of the data contract:
- the side and folder it was found in;
- its format and size;
- its row count from the manifest.

**Check:** `misfiled` should be empty, and so should `duplicates` (two files for one table block T0).

In [ ]:
schemas = use("nbx.fsdr", "schemas")
JsonSource = use("nbx.fsdr.datasource", "JsonSource")

src = JsonSource(TRAINING)
man = src.manifest()
rows = []
for t, s in schemas.TABLES.items():
    p = src.path(t)
    rows.append({"side": src.side_of.get(t, "-"), "folder": s.folder, "table": t, "present": p is not None,
                 "expected side": config.table_side(t), "MB": round(p.stat().st_size / 1e6, 3) if p else 0,
                 "rows (manifest)": (man.get("row_counts") or {}).get(t)})
inputs = pd.DataFrame(rows).sort_values(["side", "folder", "table"])
print(f"twins: {', '.join(f'{k} = {v}' for k, v in src.twins.items())}")
print(f"tables found: {inputs.present.sum()} / {len(inputs)} | misfiled: {src.misfiled() or 'none'} | "
      f"duplicates: {src.duplicate_files or 'none'}")
display(inputs.groupby(["side", "folder"]).agg(tables=("table", "count"), MB=("MB", "sum")).round(2))
show(inputs, n=60)

## Step 2 · T0: validation, completeness, Parquet cache · Agent 1 (①) data agent

T0 runs before anything else and does three things.

1. **Validation** against the data contract: columns, types, keys, value ranges, date windows, the `.csv.gz` format and the folders.
   - A *blocking* issue stops the run.
   - A *warning* is reported and the run continues.
2. **Completeness**: are the tables the pipeline needs all present?
   - T0 chooses the mode from the files. **Week 0** (`train_week0`): `activities_w1_auto`, `subs_amt_auto` and `targeting_dataiku_auto` are present, and the features come from the `*_auto` history.
   - **Every next week** (`train_weekly`): no `*_auto` history; the features come from this week's `*_score` tables. The missing history tables are then warnings ("not needed by the train_weekly pipeline"), not blocking issues.
   - A missing optional table (sample, consent) falls back to a documented default.
3. **Cache**: every table is written to Parquet in 16 partitions, by a hash of `ACCESS_METHOD_ID`.
   - Every later step reads this cache, never the `.csv.gz` again, so 1M lines fit in memory partition by partition.

Outputs: `validation_report.md / .json`, `completeness.csv`, `cache_manifest.json`, `summary.json`.

In [ ]:
data_agent = use("nbx.fsdr.contract", "data_agent")

t0 = data_agent.run_t0(TRAINING, run_date=D, force=FORCE)
print(f"T0_load {t0['status'].upper()}{' (skipped: same inputs)' if t0.get('skipped') else ''} -- pipeline {t0['pipeline']}, "
      f"{t0['tables']} tables, {t0['rows_in']:,} rows, validation {t0['validation']} {t0['issue_counts']}")
print("read as one run:", t0.get("training_source"), "+", t0.get("scoring_source"))
print()
print(t0["explanation"])
if t0["pipeline"] == "train_weekly":
    print()
    print("weekly run: features from this week's *_score tables; the agent dictionary comes from the previous run")

In [ ]:
T0 = RUN / "train" / "T0_load"
rep = json.loads((T0 / "validation_report.json").read_text(encoding="utf-8"))
issues = pd.DataFrame(rep["issues"])
if len(issues):
    display(issues.groupby(["level", "check"]).size().rename("issues").reset_index())
    show(issues[["level", "check", "table", "column", "message"]], n=20, title="validation issues")
else:
    print("validation: no issue")
show(T0 / "completeness.csv", n=50)

## Step 3 · The Parquet cache

Each table is split into 16 partitions. A line is always in the same partition across tables, so T2 / T3 can
join the tables of one partition without reading the whole base. **Check:** `verify_cache` returns no problem.

In [ ]:
CacheSource, verify_cache = use("nbx.fsdr.contract.cache", "CacheSource", "verify_cache")

cache = CacheSource(D)
cm = cache.manifest()
tables = pd.DataFrame([{"table": t, **{k: v for k, v in m.items() if k in ("rows", "partitioned", "bytes")}}
                       for t, m in cm["tables"].items()])
print(f"cache {CACHE}: {cm['partitions']} partitions, {cm['bytes'] / 1e6:,.1f} MB, built in {cm['seconds']} s")
print("verify_cache:", verify_cache(D) or "OK")
show(tables.sort_values("rows", ascending=False), n=50)

## Step 4 · T2: features of the whole base (B-01 … B-12)

T2 works partition by partition, with `WORKERS` partitions in parallel. For every eligible line it builds:

- **Eligibility** (B-01 … B-03):
  - active prepaid lines;
  - valid 8-digit MSISDN;
  - not an internal / employee plan;
  - duplicates removed.
- **Profile**: age, tenure, nationality group (Oman, India, Bangladesh, Pakistan, others), region, device segment.
- **Usage and revenue**: weekly (w1 … w4) and monthly (m1 … m3) activity, BTL and pay-as-you-go spend, the trends between periods.
- **Model scores** (B-05): the 11 model tables (churn, appetency, …). A missing score gets the median of the line's nationality group.

The anchor decides which tables are read: week 0 reads the `*_auto` tables (`anchor = T`), a weekly run reads the `*_score` tables.

Outputs:
- `base_features/part-NN.parquet` (1 file per partition);
- `feature_summary.csv` (count, nulls, mean, min, max per feature);
- `stats.json` (the medians and the ARPU tertiles that T3 uses);
- `base_features_preview.csv`.

In [ ]:
training = use("nbx.fsdr.pipelines", "training")

t2 = training.run_t2(D, workers=WORKERS, force=FORCE)
print(f"T2_features DONE{' (skipped)' if t2.get('skipped') else ''}: {t2['lines']:,} eligible lines of {t2['base_rows']:,}, "
      f"{t2['features']} features, anchor {t2['anchor']} ({t2['tables']})")
show({"eligibility filters": t2["eligible_filters"], "missing model scores": t2["missing_share"],
      "age source": t2["age_source"], "tenure source": t2["tenure_source"], "ARPU tertiles": t2["arpu_tertiles"],
      "msisdn in two partitions (must be 0)": t2["msisdn_in_two_partitions"]}, title="T2 summary")

In [ ]:
T2 = RUN / "train" / "T2_features"
fs = pd.read_csv(T2 / "feature_summary.csv")
fs["null_share"] = (fs["nulls"] / fs["count"].where(fs["count"] > 0)).round(4)
show(fs.sort_values("null_share", ascending=False), n=15, title="features with the most missing values")
stats = json.loads((T2 / "stats.json").read_text(encoding="utf-8"))
show(pd.DataFrame(stats["medians_by_nationality"]).T.round(4), n=10, title="B-05 medians by nationality (missing scores)")
show(T2 / "base_features_preview.csv", n=5)

## Step 5 · T3: segments, micro-segments and the two actions · Agents 2 (②) segmentation and 2b (②b) agent dictionary

For every line of T2:

- **Segment**: the S-rules (in order, the first that matches wins) give one of the 14 segments (Retention, RGS, Onboarding, Nursering).
  - The Nursering lines are **excluded** (B-02): they are never contacted.
- **Bands**:
  - age (young / middle / senior);
  - tenure;
  - value band (ARPU tertiles from T2);
  - usage type (voice / data / both).
- **Micro-segment** = nationality | age band | tenure band. It is used for the balance of the panel and the median anchor.
- **Week-0 history**: what the production engine targeted last week (`W0_*`), and who took an offer.
- **Two actions per line** (one of the 7 actions):
  - `ACTION_ORIGINAL`: the production dictionary (segment -> action), kept for reference (Target 1 is not scored in V4.0).
  - `ACTION_AGENT`: the agent dictionary, versioned and checked against the previous version. Target 2 uses it.

Outputs:
- `segments/part-NN.parquet`;
- `segment_counts.csv`, `microsegment_counts.csv`, `action_counts.csv`, `crosswalk.csv`;
- `priors.json` (B-10 action priors) and the agent dictionary `.json`.

In [ ]:
t3 = training.run_t3(D, workers=WORKERS, force=FORCE)
print(f"T3_segments DONE{' (skipped)' if t3.get('skipped') else ''}: {t3['lines']:,} lines, {t3['targetable']:,} targetable, "
      f"{t3['excluded_nursering']:,} Nursering excluded; {t3['segments_non_empty']} segments, "
      f"{t3['micro_segments_non_empty']} / {t3['micro_segments_total']} micro-segments; agent dictionary {t3['agent_dictionary']}")
T3 = RUN / "train" / "T3_segments"
seg = pd.read_csv(T3 / "segment_counts.csv")
show(seg, n=20)
ax = seg.set_index("SEGMENT")["lines"].sort_values().plot.barh(figsize=(8, 5), title="lines per segment")
plt.tight_layout(); plt.show()

In [ ]:
acts = pd.read_csv(T3 / "action_counts.csv")
pivot = acts.pivot_table(index="ACTION_ORIGINAL", columns="ACTION_AGENT", values="lines", aggfunc="sum", fill_value=0)
display(Markdown(f"**action_original (rows) x action_agent (columns)** · same action on "
                 f"{t3['action_original_eq_agent']:.1%} of the lines"))
display(pivot)
show(T3 / "microsegment_counts.csv", n=10)
show(json.loads((T3 / "priors.json").read_text(encoding="utf-8")), title="B-10 action priors")
show(T3 / "segments_preview.csv", n=5)

## Step 6 · Training summary

Every step folder holds a `_step.json` with its status (done / blocked / failed), its rows, its duration and its
outputs. When all three steps are **done**, open `02_scoring_step_by_step.ipynb`.

In [ ]:
rows = [{"step": s["step"], "status": s["status"], "seconds": s["seconds"], "rows": s["rows"], "files": len(s["files"])}
        for s in steps.list_steps(RUN) if s["phase"] == "train"]
show(rows, n=20, title=f"training steps of {RUN_DATE}")
tree(RUN / "train", depth=1)

## Proof: nothing came from the project folder

Every engine module was defined by a cell of this notebook, and vowpalwabbit was never loaded.

In [ ]:
# --- proof: everything ran from this notebook ------------------------------------------------------------
mods = sorted(m for m in sys.modules if m == "nbx" or m.startswith("nbx."))
from_disk = [m for m in mods if not getattr(sys.modules[m], "__fsdr_notebook__", False)]
print(f"{len(mods)} engine modules, all defined in this notebook: {not from_disk}")
print("vowpalwabbit loaded:", "vowpalwabbit" in sys.modules)
assert not from_disk, f"loaded from the project folder: {from_disk}"
assert "vowpalwabbit" not in sys.modules